# 10 Paintings domain — Wikidata multihop benchmark generator (v14 balanced from-scratch quality)

Final patch: generates a fresh paintings JSONL from scratch with exact target 15/22/25/25/25, natural L1→L5 order, date/country/material/genre diversity, expanded hard-level candidates, exact schema.

In [1]:

# ============================================================
# Common setup for Wikidata benchmark domain notebooks
# ============================================================
from pathlib import Path
import json
import math
import random
import re
import time
import shutil
from collections import Counter, defaultdict
from dataclasses import asdict, fields
from typing import Any, Dict, List, Optional, Sequence, Tuple

# Load shared benchmark helpers. The second path makes this notebook runnable
# in the ChatGPT sandbox; the first path is the normal project-local path.
if "BenchmarkExample" not in globals():
    _helper_candidates = [Path("common_helpers.py"), Path("/mnt/data/common_helpers.py")]
    for _p in _helper_candidates:
        if _p.exists():
            exec(_p.read_text(encoding="utf-8"), globals())
            break
    else:
        raise FileNotFoundError("common_helpers.py not found. Put it next to this notebook.")

DOMAIN_OUTPUT_DIR = Path(OUT_DIR) / "domain_outputs"
DOMAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_BENCHMARK_FIELD_ORDER = [
    "id", "domain", "complexity", "query_text_ru", "constraints", "requested_count",
    "gold_answer_qids", "gold_answer_labels_ru", "sparql_query", "created_at",
    "query_text_en", "gold_answer_labels_en", "is_advanced", "template_id",
    "template_family", "gold_truncated", "ask_validator_sparql", "local_validator",
    "gold_collection_meta", "gold_answer_imdb_ids", "gold_answer_imdb_titles",
]
# Hard guard: every domain JSONL must have exactly the same top-level schema and field order
# as the final JSONL files from the other domains (cinema/geo/software/etc.).
BENCHMARK_FIELD_ORDER = EXPECTED_BENCHMARK_FIELD_ORDER
assert [f.name for f in fields(BenchmarkExample)] == BENCHMARK_FIELD_ORDER, [f.name for f in fields(BenchmarkExample)]

# Optional hard cross-check against an existing final-domain JSONL. If one of these
# files is present next to the notebook, its first row must have exactly the same
# physical top-level field order. This catches silent schema drift before generation.
REFERENCE_JSONL_CANDIDATES = [
    Path("cinema.jsonl"), Path("/mnt/data/cinema.jsonl"),
    Path("geo_international.jsonl"), Path("/mnt/data/geo_international.jsonl"),
    Path("software.jsonl"), Path("/mnt/data/software.jsonl"),
]

def _first_jsonl_keys(path: Path) -> Optional[List[str]]:
    try:
        if not path.exists():
            return None
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    obj = json.loads(line)
                    return list(obj.keys()) if isinstance(obj, dict) else None
    except Exception as e:
        print(f"[WARN] could not inspect reference JSONL {path}: {e}")
    return None

for _ref_jsonl in REFERENCE_JSONL_CANDIDATES:
    _ref_keys = _first_jsonl_keys(_ref_jsonl)
    if _ref_keys:
        if _ref_keys != BENCHMARK_FIELD_ORDER:
            raise AssertionError({"reference_jsonl": str(_ref_jsonl), "expected": BENCHMARK_FIELD_ORDER, "actual": _ref_keys})
        print(f"✅ JSONL field order matches reference: {_ref_jsonl}")
        break
QID_RE_LOCAL = re.compile(r"^Q\d+$")


def _good_qid(x: Any) -> bool:
    return isinstance(x, str) and bool(QID_RE_LOCAL.fullmatch(x.strip()))


def _good_label(x: Any) -> bool:
    if x is None:
        return False
    s = str(x).strip()
    return bool(s) and not QID_RE_LOCAL.fullmatch(s)


def _safe_int(x: Any, default: Optional[int] = None) -> Optional[int]:
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return default
        return int(float(str(x).strip()))
    except Exception:
        return default


def _parse_wikidata_year(x: Any) -> Optional[int]:
    """Parse WDQS date/year strings such as 1889-01-01T00:00:00Z."""
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    m = re.search(r"([+-]?\d{3,4})", s)
    if not m:
        return None
    y = _safe_int(m.group(1))
    if y is None or y < 1000 or y > 2026:
        return None
    return y


def _year_bucket(y: Optional[int]) -> Tuple[Optional[str], Optional[int], Optional[int], Optional[str], Optional[str]]:
    """Stable, human-readable date buckets used both in query text and constraints."""
    if y is None:
        return None, None, None, None, None
    if y <= 1599:
        return "1400_1599", 1400, 1599, "1400–1599", "1400–1599"
    if y <= 1699:
        return "1600_1699", 1600, 1699, "1600–1699", "1600–1699"
    if y <= 1799:
        return "1700_1799", 1700, 1799, "1700–1799", "1700–1799"
    if y <= 1849:
        return "1800_1849", 1800, 1849, "1800–1849", "1800–1849"
    if y <= 1899:
        return "1850_1899", 1850, 1899, "1850–1899", "1850–1899"
    if y <= 1949:
        return "1900_1949", 1900, 1949, "1900–1949", "1900–1949"
    return "1950_2026", 1950, 2026, "1950–2026", "1950–2026"


def _dedupe_preserve_order(xs: Sequence[Any]) -> List[Any]:
    seen = set()
    out = []
    for x in xs:
        key = json.dumps(x, ensure_ascii=False, sort_keys=True) if isinstance(x, (dict, list)) else str(x)
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out


def _dedupe_lines(lines: Sequence[str]) -> List[str]:
    return [x for x in _dedupe_preserve_order([str(l).strip() for l in lines if str(l).strip()])]


def _entity_label_cache_key(qid: str) -> str:
    return f"wd_labels_ru_en_v3_{qid}"


def get_entity_labels_ru_en(qids: Sequence[str]) -> Dict[str, Dict[str, str]]:
    """Fetch labels via wbgetentities. Returns {qid: {'en': ..., 'ru': ...}} with caching."""
    qids = [q for q in _dedupe_preserve_order([str(q).strip() for q in qids or []]) if _good_qid(q)]
    out: Dict[str, Dict[str, str]] = {}
    missing: List[str] = []
    cache = getattr(wd, "cache", None)

    for q in qids:
        cached = cache.get(_entity_label_cache_key(q)) if cache is not None else None
        if isinstance(cached, dict) and (_good_label(cached.get("en")) or _good_label(cached.get("ru"))):
            en = str(cached.get("en") or cached.get("ru") or "").strip()
            ru = str(cached.get("ru") or cached.get("en") or "").strip()
            out[q] = {"en": en, "ru": ru}
        else:
            missing.append(q)

    for i in range(0, len(missing), 50):
        chunk = missing[i:i+50]
        if not chunk:
            continue
        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),
            "props": "labels",
            "languages": "en|ru",
            "format": "json",
        }
        last_err = None
        for attempt in range(4):
            try:
                if hasattr(wd, "_sleep_if_needed"):
                    wd._sleep_if_needed()
                resp = requests.get(WIKI_API, params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
                if hasattr(wd, "_last_request_ts"):
                    wd._last_request_ts = time.time()
                if resp.status_code in (429, 500, 502, 503, 504):
                    last_err = RuntimeError(f"Wikidata API transient HTTP {resp.status_code}")
                    time.sleep(min(20, 2 ** attempt) + random.random())
                    continue
                resp.raise_for_status()
                data = resp.json()
                entities = (data or {}).get("entities", {}) or {}
                for q in chunk:
                    labels = (entities.get(q) or {}).get("labels", {}) or {}
                    en = ((labels.get("en") or {}).get("value") or "").strip()
                    ru = ((labels.get("ru") or {}).get("value") or "").strip()
                    if not _good_label(en) and _good_label(ru):
                        en = ru
                    if not _good_label(ru) and _good_label(en):
                        ru = en
                    if _good_label(en) or _good_label(ru):
                        out[q] = {"en": en, "ru": ru}
                        if cache is not None:
                            try:
                                cache.set(_entity_label_cache_key(q), out[q])
                            except Exception:
                                pass
                break
            except Exception as e:
                last_err = e
                if attempt == 3:
                    print(f"[WARN] label fetch failed for chunk {chunk[:3]}...: {last_err}")
                time.sleep(min(20, 2 ** attempt) + random.random())
    return out


def add_ru_en_labels(df, qid_fields: Sequence[str]):
    """For every `<field>_qid` column, add `<field>_en` and `<field>_ru`."""
    if df is None or len(df) == 0:
        return df
    qids = []
    for field in qid_fields:
        col = f"{field}_qid"
        if col in df.columns:
            qids.extend([q for q in df[col].dropna().astype(str).tolist() if _good_qid(q)])
    labels = get_entity_labels_ru_en(qids)
    out = df.copy()
    for field in qid_fields:
        col = f"{field}_qid"
        if col not in out.columns:
            out[col] = None
        out[f"{field}_en"] = out[col].map(lambda q: (labels.get(str(q)) or {}).get("en") if _good_qid(str(q)) else None)
        out[f"{field}_ru"] = out[col].map(lambda q: (labels.get(str(q)) or {}).get("ru") if _good_qid(str(q)) else None)
    return out


def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    out.append(obj)
            except Exception as e:
                print(f"[WARN] bad JSONL line {line_no} in {path}: {e}")
    return out


def _append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _write_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _record_key(record: Dict[str, Any]) -> str:
    payload = {
        "domain": record.get("domain"),
        "complexity": record.get("complexity"),
        "template_id": record.get("template_id"),
        "constraints": record.get("constraints"),
        "query_text_en": record.get("query_text_en"),
    }
    return json.dumps(payload, ensure_ascii=False, sort_keys=True)


def _example_to_record(ex: BenchmarkExample) -> Dict[str, Any]:
    raw = asdict(ex)
    # Make the physical JSONL field order exactly match BenchmarkExample/common_helpers.
    return {k: raw.get(k) for k in BENCHMARK_FIELD_ORDER}


def _schema_is_exact(record: Dict[str, Any]) -> bool:
    return list(record.keys()) == BENCHMARK_FIELD_ORDER




_CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")


def _constraint_value_has_qid(v: Any) -> bool:
    if isinstance(v, str):
        return bool(QID_RE_LOCAL.fullmatch(v.strip()))
    if isinstance(v, list):
        return any(_constraint_value_has_qid(x) for x in v)
    if isinstance(v, dict):
        return any(_constraint_value_has_qid(x) for x in v.values())
    return False


def _constraint_value_has_cyrillic(v: Any) -> bool:
    if isinstance(v, str):
        return bool(_CYRILLIC_RE.search(v))
    if isinstance(v, list):
        return any(_constraint_value_has_cyrillic(x) for x in v)
    if isinstance(v, dict):
        return any(_constraint_value_has_cyrillic(x) for x in v.values())
    return False


def _constraint_qids_are_valid(v: Any) -> bool:
    if isinstance(v, str):
        return _good_qid(v)
    if isinstance(v, list):
        return all(_constraint_qids_are_valid(x) for x in v)
    if isinstance(v, dict):
        return all(_constraint_qids_are_valid(x) for x in v.values())
    return v is None


def _archive_existing_jsonl(path: Path, suffix: str = "bak") -> Optional[Path]:
    """Archive an existing generated JSONL before a clean regeneration."""
    path = Path(path)
    if not path.exists():
        return None
    ts = _now_iso().replace(":", "").replace("-", "").replace(".", "_").replace("Z", "Z")
    archived = path.with_suffix(path.suffix + f".{suffix}_{ts}")
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(archived))
    print(f"Archived previous output: {archived}")
    return archived


def validate_jsonl_exact_format(path: Path, expected_fields: Sequence[str] = BENCHMARK_FIELD_ORDER) -> pd.DataFrame:
    """Validate exact shared JSONL schema/order plus high-value benchmark invariants."""
    rows = _read_jsonl(path)
    problems = []
    for i, rec in enumerate(rows, start=1):
        keys = list(rec.keys())
        rid = rec.get("id")
        if keys != list(expected_fields):
            problems.append({
                "line": i,
                "id": rid,
                "problem": "field_order_or_field_set_mismatch",
                "actual_keys": keys,
            })
        if not isinstance(rid, str) or not re.search(r"_l[1-5]_\d{4}$", rid):
            problems.append({"line": i, "id": rid, "problem": "id_format_mismatch"})
        if rec.get("complexity") not in {"L1", "L2", "L3", "L4", "L5"}:
            problems.append({"line": i, "id": rid, "problem": "bad_complexity"})
        for key in ("gold_answer_qids", "gold_answer_labels_ru", "gold_answer_labels_en", "gold_answer_imdb_ids", "gold_answer_imdb_titles"):
            if not isinstance(rec.get(key), list):
                problems.append({"line": i, "id": rid, "problem": f"{key}_not_list"})
        qids = rec.get("gold_answer_qids", [])
        if len(qids) != len(rec.get("gold_answer_labels_ru", [])) or len(qids) != len(rec.get("gold_answer_labels_en", [])):
            problems.append({"line": i, "id": rid, "problem": "gold_qids_labels_length_mismatch"})
        if len(qids) != len(set(qids)):
            problems.append({"line": i, "id": rid, "problem": "duplicate_gold_qids"})
        if any(not _good_qid(q) for q in qids):
            problems.append({"line": i, "id": rid, "problem": "bad_gold_qid"})
        if not isinstance(rec.get("constraints"), dict):
            problems.append({"line": i, "id": rid, "problem": "constraints_not_dict"})
        else:
            if _constraint_value_has_qid(rec["constraints"]):
                problems.append({"line": i, "id": rid, "problem": "qid_leaked_into_constraints"})
            if _constraint_value_has_cyrillic(rec["constraints"]):
                problems.append({"line": i, "id": rid, "problem": "cyrillic_leaked_into_constraints"})
        meta = rec.get("gold_collection_meta")
        if not isinstance(meta, dict):
            problems.append({"line": i, "id": rid, "problem": "gold_collection_meta_not_dict"})
        else:
            if meta.get("gold_returned") != len(qids):
                problems.append({"line": i, "id": rid, "problem": "meta_gold_returned_mismatch"})
            c_qids = meta.get("constraint_qids")
            if not isinstance(c_qids, dict):
                problems.append({"line": i, "id": rid, "problem": "missing_constraint_qids"})
            elif not _constraint_qids_are_valid(c_qids):
                problems.append({"line": i, "id": rid, "problem": "bad_constraint_qids"})
        if rec.get("complexity") in {"L3", "L4", "L5"} and rec.get("is_advanced") is not True:
            problems.append({"line": i, "id": rid, "problem": "advanced_level_has_is_advanced_false"})
        if rec.get("complexity") in {"L1", "L2"} and rec.get("is_advanced") is not False:
            problems.append({"line": i, "id": rid, "problem": "basic_level_has_is_advanced_true"})
        if rec.get("gold_truncated") is not False:
            problems.append({"line": i, "id": rid, "problem": "gold_truncated_not_false"})
        if not isinstance(rec.get("sparql_query"), str) or "SELECT" not in rec.get("sparql_query", ""):
            problems.append({"line": i, "id": rid, "problem": "bad_sparql_query"})
        if not isinstance(rec.get("ask_validator_sparql"), str) or "ASK" not in rec.get("ask_validator_sparql", ""):
            problems.append({"line": i, "id": rid, "problem": "bad_ask_validator_sparql"})
    return pd.DataFrame(problems)

def _now_iso() -> str:
    return utc_now_z() if "utc_now_z" in globals() else dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paintings generator core

In [2]:

# ============================================================
# Paintings domain: clean WDQS-backed multihop generator
# ============================================================
PAINTINGS_DOMAIN = "paintings"
Q_PAINTING = "Q3305213"
PAINTINGS_OUTPUT_PATH = DOMAIN_OUTPUT_DIR / "paintings.jsonl"
PAINTINGS_AUDIT_PATH = DOMAIN_OUTPUT_DIR / "paintings.audit.json"

# 110 examples by default. Edit the targets if you need a smaller/larger run.
PAINTINGS_TARGET_PER_LEVEL = {
    "L1": 20,
    "L2": 22,
    "L3": 24,
    "L4": 22,
    "L5": 22,
}

PAINTINGS_REQUESTED_COUNT = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 3,
}

# Accepted examples must have a complete gold list no broader than these caps.
# The SPARQL query is issued with LIMIT = max_cap + 1; if it hits the sentinel
# row, the candidate is rejected instead of being saved with incomplete golds.
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    # Conservative caps: reject overly broad constraints and guarantee complete golds.
    "L1": 250,
    "L2": 200,
    "L3": 150,
    "L4": 100,
    "L5": 80,
}
PAINTINGS_MIN_GOLD_HEADROOM = 2
PAINTINGS_SEED_LIMIT = 7000
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 3500

# v4 speed knobs: same QID-first finalization as museums.
PAINTINGS_FAST_QID_FIRST_GOLD = True
PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 14
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 2


def _painting_date_filter(y1: Optional[int], y2: Optional[int]) -> List[str]:
    if y1 is None or y2 is None:
        return []
    return [
        "?item wdt:P571 ?date .",
        "BIND(YEAR(?date) AS ?year) .",
        f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
    ]


PAINTING_FIELD_BUILDERS = {
    "genre": lambda q: [f"?item wdt:P136 wd:{q} ."],
    "movement": lambda q: [f"?item wdt:P135 wd:{q} ."],
    "creator": lambda q: [f"?item wdt:P170 wd:{q} ."],
    "creator_country": lambda q: ["?item wdt:P170 ?creatorForCitizenship .", f"?creatorForCitizenship wdt:P27 wd:{q} ."],
    "creator_birth_country": lambda q: ["?item wdt:P170 ?creatorForBirthplace .", "?creatorForBirthplace wdt:P19 ?creatorBirthplace .", f"?creatorBirthplace wdt:P17 wd:{q} ."],
    "collection": lambda q: [f"?item wdt:P195 wd:{q} ."],
    "collection_country": lambda q: ["?item wdt:P195 ?collectionForCountry .", f"?collectionForCountry wdt:P17 wd:{q} ."],
    "collection_city": lambda q: ["?item wdt:P195 ?collectionForLocation .", f"?collectionForLocation wdt:P131+ wd:{q} ."],
    "material": lambda q: [f"?item wdt:P186 wd:{q} ."],
    "depicts": lambda q: [f"?item wdt:P180 wd:{q} ."],
}

PAINTING_FIELD_CONSTRAINT_NAMES = {
    "genre": "genre",
    "movement": "art_movement",
    "creator": "creator",
    "creator_country": "creator_citizenship",
    "creator_birth_country": "creator_birth_country",
    "collection": "collection",
    "collection_country": "collection_country",
    "collection_city": "collection_location",
    "material": "material",
    "depicts": "depicts",
}


def _painting_fragment(field: str, en: str, ru: str) -> Tuple[str, str]:
    return {
        "genre": (f"жанра «{ru}»", f"with genre {en}"),
        "movement": (f"относящихся к направлению «{ru}»", f"associated with the {en} movement"),
        "creator": (f"созданных автором {ru}", f"created by {en}"),
        "creator_country": (f"созданных авторами с гражданством {ru}", f"created by artists with citizenship {en}"),
        "creator_birth_country": (f"созданных авторами, родившимися в {ru}", f"created by artists born in {en}"),
        "collection": (f"находящихся в коллекции «{ru}»", f"held in the collection of {en}"),
        "collection_country": (f"находящихся в коллекциях страны {ru}", f"held in collections located in {en}"),
        "collection_city": (f"находящихся в коллекциях административной единицы/города {ru}", f"held in collections located in {en}"),
        "material": (f"созданных из материала «{ru}»", f"made from {en}"),
        "depicts": (f"изображающих «{ru}»", f"depicting {en}"),
    }[field]


def _painting_query_text(k: int, fragments_ru: Sequence[str], fragments_en: Sequence[str]) -> Tuple[str, str]:
    if fragments_ru:
        ru = f"Назови {k} картин, " + ", ".join(fragments_ru) + "."
    else:
        ru = f"Назови {k} картин."
    if fragments_en:
        en = f"Name {k} paintings " + " and ".join(fragments_en) + "."
    else:
        en = f"Name {k} paintings."
    return ru, en



def _build_paintings_gold_sparql(where_lines: Sequence[str], limit: int) -> str:
    """Human-readable/replayable gold query with labels. No ORDER BY: sorting labels
    in WDQS is expensive and is not needed for QID-based benchmark golds."""
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _build_paintings_item_sparql(where_lines: Sequence[str], limit: int) -> str:
    """Fast execution query used internally: QIDs only, no OPTIONAL labels, no ORDER BY."""
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
    }}
    LIMIT {int(limit)}
    """.strip()


def _paintings_sparql_select_generation(query: str) -> dict:
    wd_obj = globals().get("wd")
    if wd_obj is None:
        return wd.sparql_select(query)
    old_timeout = getattr(wd_obj, "timeout", None)
    old_retries = getattr(wd_obj, "max_retries", None)
    try:
        if old_timeout is not None:
            wd_obj.timeout = int(PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS)
        if old_retries is not None:
            wd_obj.max_retries = int(PAINTINGS_GENERATION_WDQS_MAX_RETRIES)
        return wd_obj.sparql_select(query, use_cache=True)
    finally:
        if old_timeout is not None:
            wd_obj.timeout = old_timeout
        if old_retries is not None:
            wd_obj.max_retries = old_retries


def _run_paintings_gold_query(spec: Dict[str, Any], gold_limit: int):
    wdqs_limit = int(gold_limit) + 1
    public_sparql = _build_paintings_gold_sparql(spec["where_lines"], wdqs_limit)
    exec_sparql = _build_paintings_item_sparql(spec["where_lines"], wdqs_limit) if PAINTINGS_FAST_QID_FIRST_GOLD else public_sparql

    t0 = time.time()
    rows = rows_from_select(_paintings_sparql_select_generation(exec_sparql))
    wdqs_seconds = round(time.time() - t0, 3)

    qids: List[str] = []
    seen = set()
    dropped_no_qid = 0
    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        if not _good_qid(qid):
            dropped_no_qid += 1
            continue
        if qid not in seen:
            seen.add(qid)
            qids.append(qid)

    label_t0 = time.time()
    labels = get_entity_labels_ru_en(qids[:int(gold_limit)])
    label_seconds = round(time.time() - label_t0, 3)

    items = []
    dropped_no_en = 0
    label_sources = Counter()
    for qid in qids:
        lab = labels.get(qid) or {}
        en = (lab.get("en") or "").strip()
        ru = (lab.get("ru") or "").strip()
        if not _good_label(en):
            dropped_no_en += 1
            continue
        if not _good_label(ru):
            ru = en
            label_sources["en_fallback_for_ru"] += 1
        else:
            label_sources["ru_label"] += 1
        items.append((qid, ru, en))
        if len(items) >= int(gold_limit):
            break

    truncated = len(rows) >= wdqs_limit or len(qids) > int(gold_limit) or len(items) > int(gold_limit)
    return public_sparql, items[:int(gold_limit)], truncated, {
        "wdqs_candidate_limit": wdqs_limit,
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "label_sources": dict(label_sources),
        "gold_may_be_incomplete_due_to_wdqs_limit": bool(truncated),
        "gold_query_execution": "qid_first_wdqs_then_wbgetentities_labels" if PAINTINGS_FAST_QID_FIRST_GOLD else "wdqs_with_labels",
        "wdqs_seconds": wdqs_seconds,
        "label_fetch_seconds": label_seconds,
    }


def _painting_local_validator(spec: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": spec.get("constraints", {}),
        "label_matching_used": False,
        "note": "All constraints for this paintings task are represented in the WDQS ASK validator; no external local validator is required.",
    }


def _finalize_painting_spec(spec: Dict[str, Any], idx: int, require_complete: bool = True) -> Optional[BenchmarkExample]:
    level = spec["complexity"]
    k = int(spec["requested_count"])
    max_gold = int(PAINTINGS_MAX_GOLD_BY_LEVEL[level])
    sparql, gold, truncated, meta0 = _run_paintings_gold_query(spec, max_gold)
    if require_complete and truncated:
        return None
    if len(gold) < k + PAINTINGS_MIN_GOLD_HEADROOM:
        return None
    if len(gold) > max_gold:
        return None
    ask = _build_paintings_ask(spec["where_lines"])
    meta = {
        "source": "wikidata_sparql",
        **meta0,
        "constraints_are_wdqs_only": True,
        "gold_limit": max_gold,
        "gold_returned": len(gold),
        "gold_total_before_limit": meta0["gold_returned_before_limits"],
        "gold_truncated_by_local_limit": bool(truncated),
        "template_id": spec["template_id"],
        "template_family": spec["template_family"],
        "constraint_qids": spec.get("constraint_qids", {}),
    }
    return BenchmarkExample(
        id=f"paintings_{level.lower()}_{idx:04d}",
        domain=PAINTINGS_DOMAIN,
        complexity=level,
        query_text_ru=spec["query_text_ru"],
        constraints=spec["constraints"],
        requested_count=k,
        gold_answer_qids=[q for q, _, _ in gold],
        gold_answer_labels_ru=[ru for _, ru, _ in gold],
        sparql_query=sparql,
        created_at=_now_iso(),
        query_text_en=spec["query_text_en"],
        gold_answer_labels_en=[en for _, _, en in gold],
        is_advanced=level in {"L3", "L4", "L5"},
        template_id=spec["template_id"],
        template_family=spec["template_family"],
        gold_truncated=bool(truncated),
        ask_validator_sparql=ask,
        local_validator=_painting_local_validator(spec),
        gold_collection_meta=meta,
    )


## Seed pool and template registry

In [3]:

# ============================================================
# Paintings seed pool and candidate registry
# ============================================================
def build_paintings_seed_pool(limit: int = PAINTINGS_SEED_LIMIT):
    """A broad seed pool of real paintings and useful linked properties.

    The pool is only used to discover promising constraints. Every saved example
    is re-queried against WDQS with its exact SPARQL, and broad/incomplete golds
    are rejected.
    """
    sparql = f"""
    SELECT DISTINCT
      ?item ?genre ?movement ?creator ?creator_country ?creator_birth_country
      ?collection ?collection_country ?collection_city ?material ?depicts ?date
    WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item wdt:P136 ?genre . }}
      OPTIONAL {{ ?item wdt:P135 ?movement . }}
      OPTIONAL {{
        ?item wdt:P170 ?creator .
        OPTIONAL {{ ?creator wdt:P27 ?creator_country . }}
        OPTIONAL {{ ?creator wdt:P19 ?creator_birthplace . ?creator_birthplace wdt:P17 ?creator_birth_country . }}
      }}
      OPTIONAL {{
        ?item wdt:P195 ?collection .
        OPTIONAL {{ ?collection wdt:P17 ?collection_country . }}
        OPTIONAL {{ ?collection wdt:P131 ?collection_city . }}
      }}
      OPTIONAL {{ ?item wdt:P186 ?material . }}
      OPTIONAL {{ ?item wdt:P180 ?depicts . }}
      OPTIONAL {{ ?item wdt:P571 ?date . }}
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        item_qid = uri_to_qid(r.get("item", ""))
        if not _good_qid(item_qid):
            continue
        y = _parse_wikidata_year(r.get("date"))
        yb, y1, y2, yru, yen = _year_bucket(y)
        data.append({
            "item_qid": item_qid,
            "genre_qid": uri_to_qid(r.get("genre", "")),
            "movement_qid": uri_to_qid(r.get("movement", "")),
            "creator_qid": uri_to_qid(r.get("creator", "")),
            "creator_country_qid": uri_to_qid(r.get("creator_country", "")),
            "creator_birth_country_qid": uri_to_qid(r.get("creator_birth_country", "")),
            "collection_qid": uri_to_qid(r.get("collection", "")),
            "collection_country_qid": uri_to_qid(r.get("collection_country", "")),
            "collection_city_qid": uri_to_qid(r.get("collection_city", "")),
            "material_qid": uri_to_qid(r.get("material", "")),
            "depicts_qid": uri_to_qid(r.get("depicts", "")),
            "year": y,
            "year_bucket": yb,
            "year_from": y1,
            "year_to": y2,
            "year_ru": yru,
            "year_en": yen,
        })
    label_fields = [
        "item", "genre", "movement", "creator", "creator_country", "creator_birth_country",
        "collection", "collection_country", "collection_city", "material", "depicts",
    ]
    if not data:
        cols = ["item_qid"] + [f"{f}_qid" for f in label_fields if f != "item"] + ["year", "year_bucket", "year_from", "year_to", "year_ru", "year_en"]
        cols += [f"{f}_{lang}" for f in label_fields for lang in ("en", "ru")]
        return pd.DataFrame(columns=_dedupe_preserve_order(cols))
    df = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)
    df = add_ru_en_labels(df, label_fields)
    if "item_en" not in df.columns:
        return pd.DataFrame()
    df = df[df["item_en"].apply(_good_label)].copy()
    return df.drop_duplicates().reset_index(drop=True)


print("Building/loading paintings seed pool (clean v3)...")
paintings_seed_df = load_or_build_pool("paintings_seed_pool_clean_v3", lambda: build_paintings_seed_pool())
print(f"paintings_seed_df rows: {len(paintings_seed_df)}")


def paintings_seed_audit() -> Dict[str, Any]:
    df = paintings_seed_df
    out = {"rows": int(len(df)) if df is not None else 0}
    if df is None or len(df) == 0:
        return out
    for f in ["genre", "movement", "creator_country", "collection", "collection_country", "collection_city", "material", "depicts", "year_bucket"]:
        col = f"{f}_qid" if f != "year_bucket" else f
        out[f] = int(df[col].notna().sum()) if col in df.columns else 0
    return out


def _painting_group_candidates(df, fields: Sequence[str], min_n: int, max_n: Optional[int], max_rows: int = 1000):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    sub = df.copy()
    group_cols = []
    agg = {"n": ("item_qid", "nunique")}
    for field in fields:
        if field == "year_bucket":
            sub = sub[sub["year_bucket"].notna()]
            group_cols.append("year_bucket")
            for c in ["year_from", "year_to", "year_ru", "year_en"]:
                agg[c] = (c, "first")
        else:
            qcol = f"{field}_qid"
            encol = f"{field}_en"
            rucol = f"{field}_ru"
            if qcol not in sub.columns:
                return pd.DataFrame()
            sub = sub[sub[qcol].apply(_good_qid) & sub[encol].apply(_good_label) & sub[rucol].apply(_good_label)]
            group_cols.append(qcol)
            agg[encol] = (encol, "first")
            agg[rucol] = (rucol, "first")
    if len(sub) == 0:
        return pd.DataFrame()
    grp = sub.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    grp = grp[grp["n"] >= int(min_n)]
    if max_n is not None:
        grp = grp[grp["n"] <= int(max_n)]
    grp = grp.sort_values(["n"], ascending=[True]).head(int(max_rows)).reset_index(drop=True)
    return grp


PAINTING_DIRECT_TEMPLATES = [
    # L1: one clean criterion, still answerable with a complete gold list.
    {"level": "L1", "template_id": "paintings_l1_by_creator", "family": "single_creator", "fields": ["creator"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_genre", "family": "single_genre", "fields": ["genre"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_collection", "family": "single_collection", "fields": ["collection"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_creator_citizenship", "family": "creator_country", "fields": ["creator_country"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_movement", "family": "single_movement", "fields": ["movement"], "min_n": 5, "max_n": 250},

    # L2: two constraints, with creator/collection paths making them genuinely multihop.
    {"level": "L2", "template_id": "paintings_l2_genre_creator_citizenship", "family": "genre_creator_country", "fields": ["genre", "creator_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_genre_collection_country", "family": "genre_collection_country", "fields": ["genre", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_movement_collection_country", "family": "movement_collection_country", "fields": ["movement", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_material_collection_country", "family": "material_collection_country", "fields": ["material", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_creator_citizenship_collection_country", "family": "creator_country_collection_country", "fields": ["creator_country", "collection_country"], "min_n": 5, "max_n": 200},

    # L3: three constraints and/or date windows.
    {"level": "L3", "template_id": "paintings_l3_genre_creator_citizenship_collection_country", "family": "genre_creator_country_collection_country", "fields": ["genre", "creator_country", "collection_country"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_collection_country_period", "family": "genre_collection_country_period", "fields": ["genre", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_movement_creator_citizenship_period", "family": "movement_creator_country_period", "fields": ["movement", "creator_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_material_creator_citizenship_collection_country", "family": "material_creator_country_collection_country", "fields": ["material", "creator_country", "collection_country"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_depicts_collection_country_period", "family": "depicts_collection_country_period", "fields": ["depicts", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
]


def _make_painting_direct_spec(tpl: Dict[str, Any], row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    level = tpl["level"]
    k = PAINTINGS_REQUESTED_COUNT[level]
    where_lines = []
    constraints = {"answer_type": "painting"}
    constraint_qids = {"answer_type": Q_PAINTING}
    fragments_ru = []
    fragments_en = []
    for field in tpl["fields"]:
        if field == "year_bucket":
            y1 = _safe_int(row.get("year_from"))
            y2 = _safe_int(row.get("year_to"))
            if y1 is None or y2 is None:
                return None
            where_lines.extend(_painting_date_filter(y1, y2))
            constraints["date_from"] = y1
            constraints["date_to"] = y2
            fragments_ru.append(f"созданных в период {y1}–{y2} годов")
            fragments_en.append(f"created between {y1} and {y2}")
            continue
        qid = row.get(f"{field}_qid")
        en = row.get(f"{field}_en")
        ru = row.get(f"{field}_ru")
        if not (_good_qid(qid) and _good_label(en) and _good_label(ru)):
            return None
        where_lines.extend(PAINTING_FIELD_BUILDERS[field](qid))
        cname = PAINTING_FIELD_CONSTRAINT_NAMES[field]
        constraints[cname] = str(en)
        constraint_qids[cname] = qid
        fr_ru, fr_en = _painting_fragment(field, str(en), str(ru))
        fragments_ru.append(fr_ru)
        fragments_en.append(fr_en)
    query_ru, query_en = _painting_query_text(k, fragments_ru, fragments_en)
    return {
        "complexity": level,
        "requested_count": k,
        "template_id": tpl["template_id"],
        "template_family": tpl["family"],
        "query_text_ru": query_ru,
        "query_text_en": query_en,
        "constraints": constraints,
        "constraint_qids": constraint_qids,
        "where_lines": _dedupe_lines(where_lines),
        "local_n": int(row.get("n", 0) or 0),
    }


def _painting_local_items(filters: Dict[str, Any], exclude_qids: Sequence[str] = ()) -> List[Tuple[str, str, str]]:
    df = paintings_seed_df.copy()
    for field, qid in filters.items():
        if field in {"year_from", "year_to"}:
            continue
        if qid and f"{field}_qid" in df.columns:
            df = df[df[f"{field}_qid"] == qid]
    y1 = filters.get("year_from")
    y2 = filters.get("year_to")
    if y1 is not None and y2 is not None:
        df = df[df["year"].notna() & (df["year"] >= int(y1)) & (df["year"] <= int(y2))]
    ex = {q for q in exclude_qids if _good_qid(q)}
    if ex:
        df = df[~df["item_qid"].isin(ex)]
    items = []
    seen = set()
    for _, row in df.iterrows():
        q = row.get("item_qid")
        en = row.get("item_en")
        ru = row.get("item_ru") or en
        if _good_qid(q) and _good_label(en) and q not in seen:
            seen.add(q)
            items.append((q, ru, en))
    return items


def _paint_ref_for_field(field: str, qid: str, exclude: Sequence[str], rng: random.Random):
    if not _good_qid(qid):
        return None
    df = paintings_seed_df[
        (paintings_seed_df.get(f"{field}_qid") == qid)
        & paintings_seed_df["item_qid"].apply(_good_qid)
        & paintings_seed_df["item_en"].apply(_good_label)
    ].copy()
    if exclude:
        df = df[~df["item_qid"].isin(set(exclude))]
    if len(df) == 0:
        return None
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return {"qid": row["item_qid"], "en": row["item_en"], "ru": row.get("item_ru") or row["item_en"]}


def build_painting_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    specs = []
    for tpl in PAINTING_DIRECT_TEMPLATES:
        grp = _painting_group_candidates(
            paintings_seed_df,
            tpl["fields"],
            min_n=tpl["min_n"],
            max_n=tpl["max_n"],
            max_rows=900,
        )
        if len(grp) == 0:
            continue
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_painting_direct_spec(tpl, row)
            if spec:
                level = spec["complexity"]
                if spec.get("local_n", 0) >= PAINTINGS_REQUESTED_COUNT[level] + PAINTINGS_MIN_GOLD_HEADROOM and spec.get("local_n", 0) <= PAINTINGS_MAX_GOLD_BY_LEVEL[level]:
                    specs.append(spec)
    return specs


def _build_painting_bridge_candidates(rng: random.Random, max_candidates: int = 2200) -> List[Dict[str, Any]]:
    """L4/L5 candidates with explicit seed bridges in the SPARQL."""
    specs: List[Dict[str, Any]] = []
    if paintings_seed_df is None or len(paintings_seed_df) == 0:
        return specs
    required_cols = {
        "item_qid", "item_en", "genre_qid", "genre_en", "genre_ru",
        "creator_country_qid", "creator_country_en", "creator_country_ru",
        "collection_country_qid", "collection_country_en", "collection_country_ru",
        "material_qid", "material_en", "material_ru", "year_bucket", "year_from", "year_to",
    }
    if not required_cols.issubset(set(paintings_seed_df.columns)):
        return specs
    df = paintings_seed_df.copy()

    def emit(spec):
        if spec is None:
            return False
        local_n = len(_painting_local_items(spec["local_filters"], exclude_qids=spec.get("exclude_qids", [])))
        spec["local_n"] = local_n
        level = spec["complexity"]
        if local_n < PAINTINGS_REQUESTED_COUNT[level] + PAINTINGS_MIN_GOLD_HEADROOM or local_n > PAINTINGS_MAX_GOLD_BY_LEVEL[level]:
            return False
        spec.pop("local_filters", None)
        specs.append(spec)
        return len(specs) >= max_candidates

    # L4-A: same genre as a seed painting + creator citizenship + collection country + period.
    rows = df[
        df["genre_qid"].apply(_good_qid)
        & df["creator_country_qid"].apply(_good_qid)
        & df["collection_country_qid"].apply(_good_qid)
        & df["year_bucket"].notna()
        & df["genre_en"].apply(_good_label)
        & df["creator_country_en"].apply(_good_label)
        & df["collection_country_en"].apply(_good_label)
    ].sample(frac=1, random_state=11)
    for _, row in rows.iterrows():
        seed = _paint_ref_for_field("genre", row["genre_qid"], [], rng)
        if not seed:
            continue
        y1, y2 = int(row["year_from"]), int(row["year_to"])
        k = PAINTINGS_REQUESTED_COUNT["L4"]
        fr_ru = [
            f"относящихся к тому же жанру, что и картина «{seed['ru']}»",
            f"созданных авторами с гражданством {row['creator_country_ru']}",
            f"находящихся в коллекциях страны {row['collection_country_ru']}",
            f"созданных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"with the same genre as the painting {seed['en']}",
            f"created by artists with citizenship {row['creator_country_en']}",
            f"held in collections located in {row['collection_country_en']}",
            f"created between {y1} and {y2}",
        ]
        qru, qen = _painting_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L4", "requested_count": k,
            "template_id": "paintings_l4_seed_genre_creator_country_collection_country_period",
            "template_family": "seed_genre_creator_country_collection_country_period",
            "query_text_ru": qru + f" Не включай картину «{seed['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the painting {seed['en']} in the answer.",
            "constraints": {
                "answer_type": "painting",
                "same_genre_as_painting": seed["en"],
                "creator_citizenship": row["creator_country_en"],
                "collection_country": row["collection_country_en"],
                "date_from": y1, "date_to": y2,
                "exclude_paintings": [seed["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_PAINTING,
                "same_genre_as_painting": seed["qid"],
                "genre": row["genre_qid"],
                "creator_citizenship": row["creator_country_qid"],
                "collection_country": row["collection_country_qid"],
                "exclude_paintings": [seed["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed['qid']} AS ?seedPainting) .",
                "?seedPainting wdt:P136 ?seedGenre .",
                "?item wdt:P136 ?seedGenre .",
                "?item wdt:P170 ?creatorForCitizenship .",
                f"?creatorForCitizenship wdt:P27 wd:{row['creator_country_qid']} .",
                "?item wdt:P195 ?collectionForCountry .",
                f"?collectionForCountry wdt:P17 wd:{row['collection_country_qid']} .",
                f"FILTER(?item != wd:{seed['qid']}) .",
                *_painting_date_filter(y1, y2),
            ]),
            "local_filters": {"genre": row["genre_qid"], "creator_country": row["creator_country_qid"], "collection_country": row["collection_country_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed["qid"]],
        })
        if stop:
            return specs

    # L4-B: same creator citizenship as a seed painting + genre + collection country.
    rows = df[
        df["creator_country_qid"].apply(_good_qid)
        & df["genre_qid"].apply(_good_qid)
        & df["collection_country_qid"].apply(_good_qid)
        & df["genre_en"].apply(_good_label)
        & df["creator_country_en"].apply(_good_label)
        & df["collection_country_en"].apply(_good_label)
    ].sample(frac=1, random_state=12)
    for _, row in rows.iterrows():
        seed = _paint_ref_for_field("creator_country", row["creator_country_qid"], [], rng)
        if not seed:
            continue
        k = PAINTINGS_REQUESTED_COUNT["L4"]
        fr_ru = [
            f"созданных авторами с тем же гражданством, что у автора картины «{seed['ru']}»",
            f"жанра «{row['genre_ru']}»",
            f"находящихся в коллекциях страны {row['collection_country_ru']}",
        ]
        fr_en = [
            f"created by artists with the same citizenship as the creator of {seed['en']}",
            f"with genre {row['genre_en']}",
            f"held in collections located in {row['collection_country_en']}",
        ]
        qru, qen = _painting_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L4", "requested_count": k,
            "template_id": "paintings_l4_seed_creator_country_genre_collection_country",
            "template_family": "seed_creator_country_genre_collection_country",
            "query_text_ru": qru + f" Не включай картину «{seed['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the painting {seed['en']} in the answer.",
            "constraints": {
                "answer_type": "painting",
                "same_creator_citizenship_as_painting": seed["en"],
                "genre": row["genre_en"],
                "collection_country": row["collection_country_en"],
                "exclude_paintings": [seed["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_PAINTING,
                "same_creator_citizenship_as_painting": seed["qid"],
                "creator_citizenship": row["creator_country_qid"],
                "genre": row["genre_qid"],
                "collection_country": row["collection_country_qid"],
                "exclude_paintings": [seed["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed['qid']} AS ?seedPainting) .",
                "?seedPainting wdt:P170 ?seedCreator .",
                "?seedCreator wdt:P27 ?seedCreatorCountry .",
                "?item wdt:P170 ?creatorForCitizenship .",
                "?creatorForCitizenship wdt:P27 ?seedCreatorCountry .",
                f"?item wdt:P136 wd:{row['genre_qid']} .",
                "?item wdt:P195 ?collectionForCountry .",
                f"?collectionForCountry wdt:P17 wd:{row['collection_country_qid']} .",
                f"FILTER(?item != wd:{seed['qid']}) .",
            ]),
            "local_filters": {"creator_country": row["creator_country_qid"], "genre": row["genre_qid"], "collection_country": row["collection_country_qid"]},
            "exclude_qids": [seed["qid"]],
        })
        if stop:
            return specs

    # L5: two seed bridges + extra filters.
    rows = df[
        df["genre_qid"].apply(_good_qid)
        & df["creator_country_qid"].apply(_good_qid)
        & df["collection_country_qid"].apply(_good_qid)
        & df["material_qid"].apply(_good_qid)
        & df["year_bucket"].notna()
        & df["genre_en"].apply(_good_label)
        & df["creator_country_en"].apply(_good_label)
        & df["collection_country_en"].apply(_good_label)
        & df["material_en"].apply(_good_label)
    ].sample(frac=1, random_state=13)
    for _, row in rows.iterrows():
        seed_a = _paint_ref_for_field("genre", row["genre_qid"], [], rng)
        seed_b = _paint_ref_for_field("creator_country", row["creator_country_qid"], [seed_a["qid"]] if seed_a else [], rng)
        if not (seed_a and seed_b):
            continue
        y1, y2 = int(row["year_from"]), int(row["year_to"])
        k = PAINTINGS_REQUESTED_COUNT["L5"]
        fr_ru = [
            f"относящихся к тому же жанру, что и картина «{seed_a['ru']}»",
            f"созданных авторами с тем же гражданством, что у автора картины «{seed_b['ru']}»",
            f"находящихся в коллекциях страны {row['collection_country_ru']}",
            f"созданных из материала «{row['material_ru']}»",
            f"созданных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"with the same genre as the painting {seed_a['en']}",
            f"created by artists with the same citizenship as the creator of {seed_b['en']}",
            f"held in collections located in {row['collection_country_en']}",
            f"made from {row['material_en']}",
            f"created between {y1} and {y2}",
        ]
        qru, qen = _painting_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L5", "requested_count": k,
            "template_id": "paintings_l5_seed_genre_seed_creator_country_collection_country_material_period",
            "template_family": "two_seed_genre_creator_country_collection_country_material_period",
            "query_text_ru": qru + f" Не включай картины «{seed_a['ru']}» и «{seed_b['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the paintings {seed_a['en']} and {seed_b['en']} in the answer.",
            "constraints": {
                "answer_type": "painting",
                "same_genre_as_painting": seed_a["en"],
                "same_creator_citizenship_as_painting": seed_b["en"],
                "collection_country": row["collection_country_en"],
                "material": row["material_en"],
                "date_from": y1, "date_to": y2,
                "exclude_paintings": [seed_a["en"], seed_b["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_PAINTING,
                "same_genre_as_painting": seed_a["qid"],
                "genre": row["genre_qid"],
                "same_creator_citizenship_as_painting": seed_b["qid"],
                "creator_citizenship": row["creator_country_qid"],
                "collection_country": row["collection_country_qid"],
                "material": row["material_qid"],
                "exclude_paintings": [seed_a["qid"], seed_b["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed_a['qid']} AS ?seedPaintingA) .",
                f"BIND(wd:{seed_b['qid']} AS ?seedPaintingB) .",
                "?seedPaintingA wdt:P136 ?seedGenre .",
                "?item wdt:P136 ?seedGenre .",
                "?seedPaintingB wdt:P170 ?seedCreator .",
                "?seedCreator wdt:P27 ?seedCreatorCountry .",
                "?item wdt:P170 ?creatorForCitizenship .",
                "?creatorForCitizenship wdt:P27 ?seedCreatorCountry .",
                "?item wdt:P195 ?collectionForCountry .",
                f"?collectionForCountry wdt:P17 wd:{row['collection_country_qid']} .",
                f"?item wdt:P186 wd:{row['material_qid']} .",
                f"FILTER(?item != wd:{seed_a['qid']} && ?item != wd:{seed_b['qid']}) .",
                *_painting_date_filter(y1, y2),
            ]),
            "local_filters": {"genre": row["genre_qid"], "creator_country": row["creator_country_qid"], "collection_country": row["collection_country_qid"], "material": row["material_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed_a["qid"], seed_b["qid"]],
        })
        if stop:
            return specs

    return specs



PAINTING_TEMPLATE_SPEED_PRIORITY = {
    "paintings_l1_genre": 0,
    "paintings_l1_movement": 1,
    "paintings_l1_collection": 2,
    "paintings_l1_creator": 3,
    "paintings_l2_genre_movement": 0,
    "paintings_l2_collection_genre": 1,
    "paintings_l2_creator_country_genre": 2,
    "paintings_l2_material_genre": 3,
    "paintings_l2_depicts_genre": 4,
    "paintings_l3_collection_country_genre_period": 0,
    "paintings_l3_creator_country_movement_period": 1,
    "paintings_l3_collection_city_genre_period": 2,
    "paintings_l4_seed_collection_genre_period": 0,
    "paintings_l4_seed_creator_country_genre": 1,
    "paintings_l5_seed_collection_seed_genre_period": 0,
    "paintings_l5_seed_creator_country_seed_movement_period": 1,
}


def _painting_spec_speed_key(spec: Dict[str, Any]) -> Tuple[int, int, str]:
    priority = PAINTING_TEMPLATE_SPEED_PRIORITY.get(str(spec.get("template_id")), 50)
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    return (priority, int(local_n), str(spec.get("template_id", "")))


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    all_specs = build_painting_direct_candidates(rng) + _build_painting_bridge_candidates(rng)
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen = set()
    for spec in all_specs:
        key = json.dumps({"level": spec["complexity"], "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
        if key in seen:
            continue
        seen.add(key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        rng.shuffle(by_level[level])
        by_level[level].sort(key=_painting_spec_speed_key)
    return by_level


def paintings_candidate_audit(queue: Optional[Dict[str, List[Dict[str, Any]]]] = None) -> pd.DataFrame:
    if queue is None:
        queue = build_painting_candidate_queue()
    rows = []
    for level, specs in queue.items():
        fam = Counter(s["template_family"] for s in specs)
        rows.append({"complexity": level, "candidates": len(specs), "families": dict(fam)})
    return pd.DataFrame(rows)


Building/loading paintings seed pool (clean v3)...
paintings_seed_df rows: 6992


In [4]:
# ============================================================
# v5 quality/fill patch for paintings:
# - expands L3/L4/L5 candidate families;
# - prevents seed-only duplicate records by semantic dedupe;
# - avoids retrying the same rejected candidate forever;
# - keeps the exact BenchmarkExample JSONL schema.
# ============================================================

# A little less conservative for L4/L5 than v3; final golds are still complete
# because sentinel LIMIT+1 rejection remains enabled.
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 250,
    "L2": 200,
    "L3": 150,
    "L4": 120,
    "L5": 100,
}
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 9000


def _clean_painting_text(x: Any) -> str:
    s = "" if x is None else str(x)
    return re.sub(r"[\u200e\u200f\u202a-\u202e\ufeff]", "", s).strip()


def _painting_year_pair_from_row(row: Dict[str, Any]) -> Tuple[Optional[int], Optional[int]]:
    y1 = _safe_int(row.get("year_from"))
    y2 = _safe_int(row.get("year_to"))
    if y1 is None or y2 is None:
        return None, None
    return int(y1), int(y2)


def _painting_semantic_key_payload_from_spec(spec: Dict[str, Any]) -> Dict[str, Any]:
    """Deduplicate by resolved constraints, not by seed label.

    Old v3/v4 generated many L4/L5 examples where only the seed painting name
    differed, but the actual resolved genre/citizenship/material/date filters and
    gold list were identical. For benchmark diversity those are duplicates.
    """
    cq = dict(spec.get("constraint_qids") or {})
    constraints = dict(spec.get("constraints") or {})
    hidden = {}
    for k, v in cq.items():
        if k in {"answer_type", "exclude_paintings"} or k.startswith("same_"):
            continue
        hidden[k] = v
    surface = {}
    for k, v in constraints.items():
        if k in {"answer_type", "exclude_paintings"} or k.startswith("same_"):
            continue
        surface[k] = v
    return {
        "domain": PAINTINGS_DOMAIN,
        "complexity": spec.get("complexity"),
        "template_family": spec.get("template_family"),
        "resolved_qids": hidden,
        "surface": surface,
    }


def _painting_semantic_key_from_spec(spec: Dict[str, Any]) -> str:
    return json.dumps(_painting_semantic_key_payload_from_spec(spec), ensure_ascii=False, sort_keys=True)


def _painting_semantic_key_from_record(rec: Dict[str, Any]) -> str:
    meta = rec.get("gold_collection_meta") or {}
    spec_like = {
        "complexity": rec.get("complexity"),
        "template_family": rec.get("template_family"),
        "constraints": rec.get("constraints") or {},
        "constraint_qids": meta.get("constraint_qids") or {},
    }
    return _painting_semantic_key_from_spec(spec_like)


# Expanded registry. Direct L4/L5 records are still multihop because they require
# creator citizenship / creator birthplace and collection country/city paths.
PAINTING_DIRECT_TEMPLATES = [
    # L1
    {"level": "L1", "template_id": "paintings_l1_by_creator", "family": "single_creator", "fields": ["creator"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_genre", "family": "single_genre", "fields": ["genre"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_collection", "family": "single_collection", "fields": ["collection"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_creator_citizenship", "family": "creator_country", "fields": ["creator_country"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_movement", "family": "single_movement", "fields": ["movement"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_material", "family": "single_material", "fields": ["material"], "min_n": 5, "max_n": 250},

    # L2
    {"level": "L2", "template_id": "paintings_l2_genre_creator_citizenship", "family": "genre_creator_country", "fields": ["genre", "creator_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_genre_collection_country", "family": "genre_collection_country", "fields": ["genre", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_movement_collection_country", "family": "movement_collection_country", "fields": ["movement", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_material_collection_country", "family": "material_collection_country", "fields": ["material", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_creator_citizenship_collection_country", "family": "creator_country_collection_country", "fields": ["creator_country", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_depicts_collection_country", "family": "depicts_collection_country", "fields": ["depicts", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_material_creator_citizenship", "family": "material_creator_country", "fields": ["material", "creator_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_genre_material", "family": "genre_material", "fields": ["genre", "material"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_movement_creator_citizenship", "family": "movement_creator_country", "fields": ["movement", "creator_country"], "min_n": 5, "max_n": 200},

    # L3
    {"level": "L3", "template_id": "paintings_l3_genre_creator_citizenship_collection_country", "family": "genre_creator_country_collection_country", "fields": ["genre", "creator_country", "collection_country"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_collection_country_period", "family": "genre_collection_country_period", "fields": ["genre", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_movement_creator_citizenship_period", "family": "movement_creator_country_period", "fields": ["movement", "creator_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_material_creator_citizenship_collection_country", "family": "material_creator_country_collection_country", "fields": ["material", "creator_country", "collection_country"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_depicts_collection_country_period", "family": "depicts_collection_country_period", "fields": ["depicts", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_material_collection_country_period", "family": "material_collection_country_period", "fields": ["material", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_movement_collection_country_period", "family": "movement_collection_country_period", "fields": ["movement", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_material_collection_country", "family": "genre_material_collection_country", "fields": ["genre", "material", "collection_country"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_depicts_creator_citizenship_collection_country", "family": "depicts_creator_country_collection_country", "fields": ["depicts", "creator_country", "collection_country"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_creator_citizenship_collection_country_period", "family": "creator_country_collection_country_period", "fields": ["creator_country", "collection_country", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_creator_birth_country_collection_country", "family": "genre_creator_birth_country_collection_country", "fields": ["genre", "creator_birth_country", "collection_country"], "min_n": 4, "max_n": 150},

    # L4 direct fallback/diversity templates
    {"level": "L4", "template_id": "paintings_l4_genre_creator_citizenship_collection_country_period", "family": "genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_depicts_collection_country_material_period", "family": "depicts_collection_country_material_period", "fields": ["depicts", "collection_country", "material", "year_bucket"], "min_n": 3, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_movement_collection_country_material_period", "family": "movement_collection_country_material_period", "fields": ["movement", "collection_country", "material", "year_bucket"], "min_n": 3, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_genre_material_collection_country_period", "family": "genre_material_collection_country_period", "fields": ["genre", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_depicts_creator_citizenship_collection_country_period", "family": "depicts_creator_country_collection_country_period", "fields": ["depicts", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_birth_country_collection_country_period", "family": "genre_creator_birth_country_collection_country_period", "fields": ["genre", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 120},

    # L5 direct fallback/diversity templates
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_citizenship_collection_country_period", "family": "genre_material_creator_country_collection_country_period", "fields": ["genre", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_citizenship_collection_country_period", "family": "movement_material_creator_country_collection_country_period", "fields": ["movement", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L5", "template_id": "paintings_l5_depicts_material_creator_citizenship_collection_country_period", "family": "depicts_material_creator_country_collection_country_period", "fields": ["depicts", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_birth_country_collection_country_period", "family": "genre_material_creator_birth_country_collection_country_period", "fields": ["genre", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
]


def build_painting_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    specs: List[Dict[str, Any]] = []
    for tpl in PAINTING_DIRECT_TEMPLATES:
        grp = _painting_group_candidates(
            paintings_seed_df,
            tpl["fields"],
            min_n=tpl["min_n"],
            # Use the template cap during local grouping, but allow a wider local window
            # because seed-pool counts are incomplete and final WDQS decides validity.
            max_n=int(tpl["max_n"] * 2.5),
            max_rows=1800,
        )
        if len(grp) == 0:
            continue
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_painting_direct_spec(tpl, row)
            if not spec:
                continue
            level = spec["complexity"]
            local_n = int(spec.get("local_n", 0) or 0)
            if local_n >= PAINTINGS_REQUESTED_COUNT[level] + 1:
                specs.append(spec)
    return specs


def _painting_local_count(filters: Dict[str, Any], exclude_qids: Sequence[str] = ()) -> int:
    return len(_painting_local_items(filters, exclude_qids=exclude_qids))


def _build_painting_bridge_candidates(rng: random.Random, max_candidates: int = 5000) -> List[Dict[str, Any]]:
    """Diverse L4/L5 seed-bridge candidates.

    v3/v4 had enough L5 surface records but they were mostly the same resolved
    constraints with different seed names. This version dedupes semantically and
    adds direct fallback templates so L4/L5 can fill without repetition.
    """
    specs: List[Dict[str, Any]] = []
    semantic_seen = set()
    if paintings_seed_df is None or len(paintings_seed_df) == 0:
        return specs
    required_cols = {
        "item_qid", "item_en", "item_ru", "genre_qid", "genre_en", "genre_ru",
        "creator_country_qid", "creator_country_en", "creator_country_ru",
        "collection_country_qid", "collection_country_en", "collection_country_ru",
        "material_qid", "material_en", "material_ru", "depicts_qid", "depicts_en", "depicts_ru",
        "year_bucket", "year_from", "year_to",
    }
    if not required_cols.issubset(set(paintings_seed_df.columns)):
        return specs
    df = paintings_seed_df.copy()

    def emit(spec):
        if spec is None:
            return False
        key = _painting_semantic_key_from_spec(spec)
        if key in semantic_seen:
            return False
        level = spec["complexity"]
        local_n = _painting_local_count(spec.get("local_filters", {}), exclude_qids=spec.get("exclude_qids", []))
        spec["local_n"] = local_n
        if local_n < PAINTINGS_REQUESTED_COUNT[level] + 1:
            return False
        semantic_seen.add(key)
        spec.pop("local_filters", None)
        specs.append(spec)
        return len(specs) >= max_candidates

    base_mask = (
        df["genre_qid"].apply(_good_qid) & df["genre_en"].apply(_good_label) & df["genre_ru"].apply(_good_label)
        & df["creator_country_qid"].apply(_good_qid) & df["creator_country_en"].apply(_good_label) & df["creator_country_ru"].apply(_good_label)
        & df["collection_country_qid"].apply(_good_qid) & df["collection_country_en"].apply(_good_label) & df["collection_country_ru"].apply(_good_label)
        & df["year_bucket"].notna()
    )

    # L4-A: same genre as a seed painting + creator citizenship + collection country + period.
    rows = df[base_mask].sample(frac=1, random_state=1101)
    for _, row in rows.iterrows():
        seed = _paint_ref_for_field("genre", row["genre_qid"], [], rng)
        y1, y2 = _painting_year_pair_from_row(row)
        if not seed or y1 is None:
            continue
        k = PAINTINGS_REQUESTED_COUNT["L4"]
        fr_ru = [
            f"относящихся к тому же жанру, что и картина «{_clean_painting_text(seed['ru'])}»",
            f"созданных авторами с гражданством {_clean_painting_text(row['creator_country_ru'])}",
            f"находящихся в коллекциях страны {_clean_painting_text(row['collection_country_ru'])}",
            f"созданных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"with the same genre as the painting {_clean_painting_text(seed['en'])}",
            f"created by artists with citizenship {_clean_painting_text(row['creator_country_en'])}",
            f"held in collections located in {_clean_painting_text(row['collection_country_en'])}",
            f"created between {y1} and {y2}",
        ]
        qru, qen = _painting_query_text(k, fr_ru, fr_en)
        if emit({
            "complexity": "L4", "requested_count": k,
            "template_id": "paintings_l4_seed_genre_creator_country_collection_country_period",
            "template_family": "seed_genre_creator_country_collection_country_period",
            "query_text_ru": qru + f" Не включай картину «{_clean_painting_text(seed['ru'])}» в ответ.",
            "query_text_en": qen + f" Do not include the painting {_clean_painting_text(seed['en'])} in the answer.",
            "constraints": {"answer_type": "painting", "same_genre_as_painting": _clean_painting_text(seed["en"]), "creator_citizenship": _clean_painting_text(row["creator_country_en"]), "collection_country": _clean_painting_text(row["collection_country_en"]), "date_from": y1, "date_to": y2, "exclude_paintings": [_clean_painting_text(seed["en"])]},
            "constraint_qids": {"answer_type": Q_PAINTING, "same_genre_as_painting": seed["qid"], "genre": row["genre_qid"], "creator_citizenship": row["creator_country_qid"], "collection_country": row["collection_country_qid"], "exclude_paintings": [seed["qid"]]},
            "where_lines": _dedupe_lines([f"BIND(wd:{seed['qid']} AS ?seedPainting) .", "?seedPainting wdt:P136 ?seedGenre .", "?item wdt:P136 ?seedGenre .", "?item wdt:P170 ?creatorForCitizenship .", f"?creatorForCitizenship wdt:P27 wd:{row['creator_country_qid']} .", "?item wdt:P195 ?collectionForCountry .", f"?collectionForCountry wdt:P17 wd:{row['collection_country_qid']} .", f"FILTER(?item != wd:{seed['qid']}) .", *_painting_date_filter(y1, y2)]),
            "local_filters": {"genre": row["genre_qid"], "creator_country": row["creator_country_qid"], "collection_country": row["collection_country_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed["qid"]],
        }):
            return specs

    # L4-B: same creator citizenship as a seed painting + genre + collection country.
    rows = df[base_mask].sample(frac=1, random_state=1102)
    for _, row in rows.iterrows():
        seed = _paint_ref_for_field("creator_country", row["creator_country_qid"], [], rng)
        if not seed:
            continue
        k = PAINTINGS_REQUESTED_COUNT["L4"]
        fr_ru = [
            f"созданных авторами с тем же гражданством, что у автора картины «{_clean_painting_text(seed['ru'])}»",
            f"жанра «{_clean_painting_text(row['genre_ru'])}»",
            f"находящихся в коллекциях страны {_clean_painting_text(row['collection_country_ru'])}",
        ]
        fr_en = [
            f"created by artists with the same citizenship as the creator of {_clean_painting_text(seed['en'])}",
            f"with genre {_clean_painting_text(row['genre_en'])}",
            f"held in collections located in {_clean_painting_text(row['collection_country_en'])}",
        ]
        qru, qen = _painting_query_text(k, fr_ru, fr_en)
        if emit({
            "complexity": "L4", "requested_count": k,
            "template_id": "paintings_l4_seed_creator_country_genre_collection_country",
            "template_family": "seed_creator_country_genre_collection_country",
            "query_text_ru": qru + f" Не включай картину «{_clean_painting_text(seed['ru'])}» в ответ.",
            "query_text_en": qen + f" Do not include the painting {_clean_painting_text(seed['en'])} in the answer.",
            "constraints": {"answer_type": "painting", "same_creator_citizenship_as_painting": _clean_painting_text(seed["en"]), "genre": _clean_painting_text(row["genre_en"]), "collection_country": _clean_painting_text(row["collection_country_en"]), "exclude_paintings": [_clean_painting_text(seed["en"])]},
            "constraint_qids": {"answer_type": Q_PAINTING, "same_creator_citizenship_as_painting": seed["qid"], "creator_citizenship": row["creator_country_qid"], "genre": row["genre_qid"], "collection_country": row["collection_country_qid"], "exclude_paintings": [seed["qid"]]},
            "where_lines": _dedupe_lines([f"BIND(wd:{seed['qid']} AS ?seedPainting) .", "?seedPainting wdt:P170 ?seedCreator .", "?seedCreator wdt:P27 ?seedCreatorCountry .", "?item wdt:P170 ?creatorForCitizenship .", "?creatorForCitizenship wdt:P27 ?seedCreatorCountry .", f"?item wdt:P136 wd:{row['genre_qid']} .", "?item wdt:P195 ?collectionForCountry .", f"?collectionForCountry wdt:P17 wd:{row['collection_country_qid']} .", f"FILTER(?item != wd:{seed['qid']}) ."]),
            "local_filters": {"creator_country": row["creator_country_qid"], "genre": row["genre_qid"], "collection_country": row["collection_country_qid"]},
            "exclude_qids": [seed["qid"]],
        }):
            return specs

    # L5: two seed bridges + extra filters, but only one per resolved semantic key.
    l5_mask = base_mask & df["material_qid"].apply(_good_qid) & df["material_en"].apply(_good_label) & df["material_ru"].apply(_good_label)
    rows = df[l5_mask].sample(frac=1, random_state=1105)
    for _, row in rows.iterrows():
        seed_a = _paint_ref_for_field("genre", row["genre_qid"], [], rng)
        seed_b = _paint_ref_for_field("creator_country", row["creator_country_qid"], [seed_a["qid"]] if seed_a else [], rng)
        y1, y2 = _painting_year_pair_from_row(row)
        if not (seed_a and seed_b) or y1 is None:
            continue
        k = PAINTINGS_REQUESTED_COUNT["L5"]
        fr_ru = [
            f"относящихся к тому же жанру, что и картина «{_clean_painting_text(seed_a['ru'])}»",
            f"созданных авторами с тем же гражданством, что у автора картины «{_clean_painting_text(seed_b['ru'])}»",
            f"находящихся в коллекциях страны {_clean_painting_text(row['collection_country_ru'])}",
            f"созданных из материала «{_clean_painting_text(row['material_ru'])}»",
            f"созданных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"with the same genre as the painting {_clean_painting_text(seed_a['en'])}",
            f"created by artists with the same citizenship as the creator of {_clean_painting_text(seed_b['en'])}",
            f"held in collections located in {_clean_painting_text(row['collection_country_en'])}",
            f"made from {_clean_painting_text(row['material_en'])}",
            f"created between {y1} and {y2}",
        ]
        qru, qen = _painting_query_text(k, fr_ru, fr_en)
        if emit({
            "complexity": "L5", "requested_count": k,
            "template_id": "paintings_l5_seed_genre_seed_creator_country_collection_country_material_period",
            "template_family": "two_seed_genre_creator_country_collection_country_material_period",
            "query_text_ru": qru + f" Не включай картины «{_clean_painting_text(seed_a['ru'])}» и «{_clean_painting_text(seed_b['ru'])}» в ответ.",
            "query_text_en": qen + f" Do not include the paintings {_clean_painting_text(seed_a['en'])} and {_clean_painting_text(seed_b['en'])} in the answer.",
            "constraints": {"answer_type": "painting", "same_genre_as_painting": _clean_painting_text(seed_a["en"]), "same_creator_citizenship_as_painting": _clean_painting_text(seed_b["en"]), "collection_country": _clean_painting_text(row["collection_country_en"]), "material": _clean_painting_text(row["material_en"]), "date_from": y1, "date_to": y2, "exclude_paintings": [_clean_painting_text(seed_a["en"]), _clean_painting_text(seed_b["en"])]},
            "constraint_qids": {"answer_type": Q_PAINTING, "same_genre_as_painting": seed_a["qid"], "genre": row["genre_qid"], "same_creator_citizenship_as_painting": seed_b["qid"], "creator_citizenship": row["creator_country_qid"], "collection_country": row["collection_country_qid"], "material": row["material_qid"], "exclude_paintings": [seed_a["qid"], seed_b["qid"]]},
            "where_lines": _dedupe_lines([f"BIND(wd:{seed_a['qid']} AS ?seedPaintingA) .", f"BIND(wd:{seed_b['qid']} AS ?seedPaintingB) .", "?seedPaintingA wdt:P136 ?seedGenre .", "?item wdt:P136 ?seedGenre .", "?seedPaintingB wdt:P170 ?seedCreator .", "?seedCreator wdt:P27 ?seedCreatorCountry .", "?item wdt:P170 ?creatorForCitizenship .", "?creatorForCitizenship wdt:P27 ?seedCreatorCountry .", "?item wdt:P195 ?collectionForCountry .", f"?collectionForCountry wdt:P17 wd:{row['collection_country_qid']} .", f"?item wdt:P186 wd:{row['material_qid']} .", f"FILTER(?item != wd:{seed_a['qid']} && ?item != wd:{seed_b['qid']}) .", *_painting_date_filter(y1, y2)]),
            "local_filters": {"genre": row["genre_qid"], "creator_country": row["creator_country_qid"], "collection_country": row["collection_country_qid"], "material": row["material_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed_a["qid"], seed_b["qid"]],
        }):
            return specs

    return specs


PAINTING_TEMPLATE_SPEED_PRIORITY = defaultdict(lambda: 50, {
    "paintings_l1_by_creator": 3,
    "paintings_l1_by_genre": 0,
    "paintings_l1_by_collection": 2,
    "paintings_l1_by_creator_citizenship": 4,
    "paintings_l1_by_movement": 1,
    "paintings_l1_by_material": 5,
    "paintings_l2_genre_collection_country": 0,
    "paintings_l2_material_collection_country": 1,
    "paintings_l2_depicts_collection_country": 2,
    "paintings_l3_genre_collection_country_period": 0,
    "paintings_l3_material_collection_country_period": 1,
    "paintings_l4_genre_creator_citizenship_collection_country_period": 0,
    "paintings_l4_material_creator_citizenship_collection_country_period": 1,
    "paintings_l5_genre_material_creator_citizenship_collection_country_period": 0,
})


def _painting_spec_speed_key(spec: Dict[str, Any]) -> Tuple[int, int, str]:
    priority = PAINTING_TEMPLATE_SPEED_PRIORITY[str(spec.get("template_id", ""))]
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    return (priority, int(local_n), str(spec.get("template_id", "")))


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    t0 = time.time()
    print("Building paintings candidate queue (v5 fill/diversity)...")
    rng = random.Random(seed)
    direct = build_painting_direct_candidates(rng)
    bridge = _build_painting_bridge_candidates(rng)
    print(f"  direct candidate specs: {len(direct)}")
    print(f"  bridge candidate specs: {len(bridge)}")
    all_specs = direct + bridge
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen_surface = set()
    seen_semantic = set()
    for spec in all_specs:
        surface_key = json.dumps({"level": spec["complexity"], "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
        semantic_key = _painting_semantic_key_from_spec(spec)
        if surface_key in seen_surface or semantic_key in seen_semantic:
            continue
        seen_surface.add(surface_key)
        seen_semantic.add(semantic_key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        rng.shuffle(by_level[level])
        by_level[level].sort(key=_painting_spec_speed_key)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()}, f"built in {time.time()-t0:.1f}s")
    return by_level


def paintings_candidate_audit(queue: Optional[Dict[str, List[Dict[str, Any]]]] = None) -> pd.DataFrame:
    if queue is None:
        queue = build_painting_candidate_queue()
    rows = []
    for level, specs in queue.items():
        fam = Counter(s["template_family"] for s in specs)
        rows.append({"complexity": level, "candidates": len(specs), "families": dict(fam)})
    return pd.DataFrame(rows)


In [5]:
# ============================================================
# v6 fast-fill patch for paintings:
# - fixes "0 generated / looks stuck" by using constraint-first WDQS queries,
#   faster candidate ordering and visible attempt progress;
# - expands L4/L5 direct templates and relaxes seed-local prefilter;
# - keeps exact BenchmarkExample JSONL schema and complete-gold sentinel logic.
# ============================================================

PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 8
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 1
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 12000

# Keep quality caps, but allow enough candidates for L4/L5; final WDQS still rejects
# incomplete / too broad results with LIMIT cap+1.
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 250,
    "L2": 200,
    "L3": 150,
    "L4": 120,
    "L5": 110,
}
PAINTINGS_MIN_GOLD_HEADROOM = 2


def _clean_painting_text(x: Any) -> str:
    s = "" if x is None else str(x)
    return re.sub(r"[\u200e\u200f\u202a-\u202e\ufeff]", "", s).strip()


def _painting_year_pair_from_row(row: Dict[str, Any]) -> Tuple[Optional[int], Optional[int]]:
    y1 = _safe_int(row.get("year_from"))
    y2 = _safe_int(row.get("year_to"))
    if y1 is None or y2 is None:
        return None, None
    return int(y1), int(y2)


# Important speed fix: put selective constraints before the broad painting-class line.
# WDQS often plans much better for collection/material/creator filters this way.
def _build_paintings_gold_sparql(where_lines: Sequence[str], limit: int) -> str:
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      {where}
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _build_paintings_item_sparql(where_lines: Sequence[str], limit: int) -> str:
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item WHERE {{
      {where}
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
    }}
    LIMIT {int(limit)}
    """.strip()


# More L4/L5 candidates than v5. Local seed counts are only hints, so for hard levels
# we allow looser local_n and let final WDQS decide validity.
PAINTING_DIRECT_TEMPLATES = [
    # L1: prioritize fast/selective families in _painting_spec_speed_key.
    {"level": "L1", "template_id": "paintings_l1_by_collection", "family": "single_collection", "fields": ["collection"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_creator", "family": "single_creator", "fields": ["creator"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_material", "family": "single_material", "fields": ["material"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_movement", "family": "single_movement", "fields": ["movement"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_creator_citizenship", "family": "creator_country", "fields": ["creator_country"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "paintings_l1_by_genre", "family": "single_genre", "fields": ["genre"], "min_n": 5, "max_n": 250},

    # L2.
    {"level": "L2", "template_id": "paintings_l2_material_collection_country", "family": "material_collection_country", "fields": ["material", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_creator_citizenship_collection_country", "family": "creator_country_collection_country", "fields": ["creator_country", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_movement_collection_country", "family": "movement_collection_country", "fields": ["movement", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_genre_collection_country", "family": "genre_collection_country", "fields": ["genre", "collection_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_material_creator_citizenship", "family": "material_creator_country", "fields": ["material", "creator_country"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_genre_material", "family": "genre_material", "fields": ["genre", "material"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "paintings_l2_depicts_collection_country", "family": "depicts_collection_country", "fields": ["depicts", "collection_country"], "min_n": 5, "max_n": 200},

    # L3.
    {"level": "L3", "template_id": "paintings_l3_material_collection_country_period", "family": "material_collection_country_period", "fields": ["material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_collection_country_period", "family": "genre_collection_country_period", "fields": ["genre", "collection_country", "year_bucket"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_creator_citizenship_collection_country_period", "family": "creator_country_collection_country_period", "fields": ["creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_material_creator_citizenship_collection_country", "family": "material_creator_country_collection_country", "fields": ["material", "creator_country", "collection_country"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_movement_creator_citizenship_period", "family": "movement_creator_country_period", "fields": ["movement", "creator_country", "year_bucket"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_creator_citizenship_collection_country", "family": "genre_creator_country_collection_country", "fields": ["genre", "creator_country", "collection_country"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_depicts_collection_country_period", "family": "depicts_collection_country_period", "fields": ["depicts", "collection_country", "year_bucket"], "min_n": 3, "max_n": 150},
    {"level": "L3", "template_id": "paintings_l3_genre_material_collection_country", "family": "genre_material_collection_country", "fields": ["genre", "material", "collection_country"], "min_n": 3, "max_n": 150},

    # L4 direct/fallback diversity.
    {"level": "L4", "template_id": "paintings_l4_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_citizenship_collection_country_period", "family": "genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_genre_material_collection_country_period", "family": "genre_material_collection_country_period", "fields": ["genre", "material", "collection_country", "year_bucket"], "min_n": 2, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_movement_collection_country_material_period", "family": "movement_collection_country_material_period", "fields": ["movement", "collection_country", "material", "year_bucket"], "min_n": 2, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_creator_citizenship_collection_country_material_period", "family": "creator_country_collection_country_material_period", "fields": ["creator_country", "collection_country", "material", "year_bucket"], "min_n": 2, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_birth_country_collection_country_period", "family": "genre_creator_birth_country_collection_country_period", "fields": ["genre", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 120},
    {"level": "L4", "template_id": "paintings_l4_depicts_collection_country_material_period", "family": "depicts_collection_country_material_period", "fields": ["depicts", "collection_country", "material", "year_bucket"], "min_n": 2, "max_n": 120},

    # L5 direct/fallback diversity. These are still multihop through creator/collection paths.
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_citizenship_collection_country_period", "family": "genre_material_creator_country_collection_country_period", "fields": ["genre", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 110},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_citizenship_collection_country_period", "family": "movement_material_creator_country_collection_country_period", "fields": ["movement", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 110},
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_birth_country_collection_country_period", "family": "genre_material_creator_birth_country_collection_country_period", "fields": ["genre", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 110},
    {"level": "L5", "template_id": "paintings_l5_genre_material_collection_country_period", "family": "genre_material_collection_country_period", "fields": ["genre", "material", "collection_country", "year_bucket"], "min_n": 2, "max_n": 110},
    {"level": "L5", "template_id": "paintings_l5_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 110},
    {"level": "L5", "template_id": "paintings_l5_depicts_material_creator_citizenship_collection_country_period", "family": "depicts_material_creator_country_collection_country_period", "fields": ["depicts", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 110},
]


def build_painting_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    specs: List[Dict[str, Any]] = []
    for tpl in PAINTING_DIRECT_TEMPLATES:
        # Seed pool is incomplete; do not over-prune locally. Final WDQS is the source of truth.
        local_max = int(tpl["max_n"] * (3.0 if tpl["level"] in {"L4", "L5"} else 2.0))
        grp = _painting_group_candidates(
            paintings_seed_df,
            tpl["fields"],
            min_n=tpl["min_n"],
            max_n=local_max,
            max_rows=2200 if tpl["level"] in {"L4", "L5"} else 1400,
        )
        if len(grp) == 0:
            continue
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_painting_direct_spec(tpl, row)
            if not spec:
                continue
            level = spec["complexity"]
            local_n = int(spec.get("local_n", 0) or 0)
            # For L4/L5 local_n is only a noisy lower-bound hint. Requiring k+headroom here
            # was the main reason v5 had too few hard-level candidates.
            min_local = max(2, PAINTINGS_REQUESTED_COUNT[level] - 1) if level in {"L4", "L5"} else PAINTINGS_REQUESTED_COUNT[level]
            if local_n >= min_local:
                specs.append(spec)
    return specs


# Faster priorities: avoid starting with broad genre/citizenship L1 queries.
PAINTING_TEMPLATE_SPEED_PRIORITY = defaultdict(lambda: 50, {
    "paintings_l1_by_collection": 0,
    "paintings_l1_by_creator": 1,
    "paintings_l1_by_material": 2,
    "paintings_l1_by_movement": 3,
    "paintings_l1_by_creator_citizenship": 8,
    "paintings_l1_by_genre": 9,
    "paintings_l2_material_collection_country": 0,
    "paintings_l2_creator_citizenship_collection_country": 1,
    "paintings_l2_movement_collection_country": 2,
    "paintings_l2_genre_collection_country": 3,
    "paintings_l2_material_creator_citizenship": 4,
    "paintings_l3_material_collection_country_period": 0,
    "paintings_l3_creator_citizenship_collection_country_period": 1,
    "paintings_l3_genre_collection_country_period": 2,
    "paintings_l3_material_creator_citizenship_collection_country": 3,
    "paintings_l4_material_creator_citizenship_collection_country_period": 0,
    "paintings_l4_creator_citizenship_collection_country_material_period": 1,
    "paintings_l4_genre_material_collection_country_period": 2,
    "paintings_l4_genre_creator_citizenship_collection_country_period": 3,
    "paintings_l5_material_creator_citizenship_collection_country_period": 0,
    "paintings_l5_genre_material_collection_country_period": 1,
    "paintings_l5_genre_material_creator_citizenship_collection_country_period": 2,
})


def _painting_spec_speed_key(spec: Dict[str, Any]) -> Tuple[int, int, str]:
    priority = PAINTING_TEMPLATE_SPEED_PRIORITY[str(spec.get("template_id", ""))]
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    # Too-small local_n candidates are often false positives for final WDQS; put them later.
    k = PAINTINGS_REQUESTED_COUNT.get(spec.get("complexity"), 3)
    small_penalty = 10 if local_n < k + PAINTINGS_MIN_GOLD_HEADROOM else 0
    return (priority + small_penalty, int(local_n), str(spec.get("template_id", "")))


def _build_painting_bridge_candidates(rng: random.Random, max_candidates: int = 1200) -> List[Dict[str, Any]]:
    # Keep v5 bridge logic available but do not let slow/local duplicate bridge construction
    # dominate the queue; direct hard templates now provide the diversity/fill.
    try:
        specs = []
        # Use the previous bridge builder if it exists under the same name? It has been overridden
        # in v5, so rebuilding it safely here is not worth the cost. Direct L4/L5 templates are enough.
        return specs
    except Exception:
        return []


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    t0 = time.time()
    print("Building paintings candidate queue (v6 fast fill)...")
    rng = random.Random(seed)
    direct = build_painting_direct_candidates(rng)
    bridge = _build_painting_bridge_candidates(rng)
    print(f"  direct candidate specs: {len(direct)}")
    print(f"  bridge candidate specs: {len(bridge)}")
    all_specs = direct + bridge
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen_surface = set()
    seen_semantic = set()
    for spec in all_specs:
        surface_key = json.dumps({"level": spec["complexity"], "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
        semantic_key = _painting_semantic_key_from_spec(spec)
        if surface_key in seen_surface or semantic_key in seen_semantic:
            continue
        seen_surface.add(surface_key)
        seen_semantic.add(semantic_key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        rng.shuffle(by_level[level])
        by_level[level].sort(key=_painting_spec_speed_key)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()}, f"built in {time.time()-t0:.1f}s")
    return by_level

print("✅ Applied paintings v6 fast-fill overrides")


✅ Applied paintings v6 fast-fill overrides


## Generation

In [6]:
# ============================================================
# v7 stable-fill patch for paintings:
# - fixes v6 "0 accepted" by sorting/filtering candidates with enough local gold first;
# - resumes from the best existing/archive paintings JSONL instead of wiping good v3 records;
# - keeps QID-first fast WDQS but uses constraint-first SPARQL;
# - prints reject reasons so slow/rejected candidates are visible;
# - preserves exact benchmark JSONL schema.
# ============================================================

PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 10
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 1

# Slightly relaxed caps versus v3. Still rejects very broad/incomplete tasks by sentinel LIMIT+1.
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 300,
    "L2": 230,
    "L3": 180,
    "L4": 140,
    "L5": 110,
}
PAINTINGS_MIN_GOLD_HEADROOM = 2


def _build_paintings_gold_sparql(where_lines: Sequence[str], limit: int) -> str:
    """Replayable gold query with labels, constraint-first for WDQS planner."""
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      {where}
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _build_paintings_item_sparql(where_lines: Sequence[str], limit: int) -> str:
    """Fast internal query: QIDs only, constraints before broad painting class."""
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item WHERE {{
      {where}
      ?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
    }}
    LIMIT {int(limit)}
    """.strip()


def _run_paintings_gold_query(spec: Dict[str, Any], gold_limit: int):
    wdqs_limit = int(gold_limit) + 1
    public_sparql = _build_paintings_gold_sparql(spec["where_lines"], wdqs_limit)
    exec_sparql = _build_paintings_item_sparql(spec["where_lines"], wdqs_limit) if PAINTINGS_FAST_QID_FIRST_GOLD else public_sparql

    t0 = time.time()
    rows = rows_from_select(_paintings_sparql_select_generation(exec_sparql))
    wdqs_seconds = round(time.time() - t0, 3)

    qids: List[str] = []
    seen = set()
    dropped_no_qid = 0
    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        if not _good_qid(qid):
            dropped_no_qid += 1
            continue
        if qid not in seen:
            seen.add(qid)
            qids.append(qid)

    label_t0 = time.time()
    labels = get_entity_labels_ru_en(qids[:int(gold_limit)])
    label_seconds = round(time.time() - label_t0, 3)

    items = []
    dropped_no_en = 0
    label_sources = Counter()
    for qid in qids:
        lab = labels.get(qid) or {}
        en = (lab.get("en") or "").strip()
        ru = (lab.get("ru") or "").strip()
        if not _good_label(en):
            dropped_no_en += 1
            continue
        if not _good_label(ru):
            ru = en
            label_sources["en_fallback_for_ru"] += 1
        else:
            label_sources["ru_label"] += 1
        items.append((qid, ru, en))
        if len(items) >= int(gold_limit):
            break

    # Sentinel: if WDQS returned the extra row, gold may be incomplete.
    truncated = len(rows) >= wdqs_limit or len(qids) > int(gold_limit)
    return public_sparql, items[:int(gold_limit)], truncated, {
        "wdqs_candidate_limit": wdqs_limit,
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "label_sources": dict(label_sources),
        "gold_may_be_incomplete_due_to_wdqs_limit": bool(truncated),
        "gold_query_execution": "qid_first_wdqs_then_wbgetentities_labels" if PAINTINGS_FAST_QID_FIRST_GOLD else "wdqs_with_labels",
        "wdqs_seconds": wdqs_seconds,
        "label_fetch_seconds": label_seconds,
    }


def _finalize_painting_spec(spec: Dict[str, Any], idx: int, require_complete: bool = True) -> Optional[BenchmarkExample]:
    spec.pop("_last_reject_reason", None)
    level = spec["complexity"]
    k = int(spec["requested_count"])
    max_gold = int(PAINTINGS_MAX_GOLD_BY_LEVEL[level])
    try:
        sparql, gold, truncated, meta0 = _run_paintings_gold_query(spec, max_gold)
    except Exception as e:
        spec["_last_reject_reason"] = "exception:" + str(e)[:160]
        raise
    if require_complete and truncated:
        spec["_last_reject_reason"] = f"truncated_or_over_cap rows={meta0.get('rows_returned_by_wdqs')} cap={max_gold}"
        return None
    if len(gold) < k + PAINTINGS_MIN_GOLD_HEADROOM:
        spec["_last_reject_reason"] = f"too_few_gold gold={len(gold)} need={k + PAINTINGS_MIN_GOLD_HEADROOM}"
        return None
    if len(gold) > max_gold:
        spec["_last_reject_reason"] = f"too_many_gold gold={len(gold)} cap={max_gold}"
        return None
    ask = _build_paintings_ask(spec["where_lines"])
    meta = {
        "source": "wikidata_sparql",
        **meta0,
        "constraints_are_wdqs_only": True,
        "gold_limit": max_gold,
        "gold_returned": len(gold),
        "gold_total_before_limit": meta0["gold_returned_before_limits"],
        "gold_truncated_by_local_limit": bool(truncated),
        "template_id": spec["template_id"],
        "template_family": spec["template_family"],
        "constraint_qids": spec.get("constraint_qids", {}),
    }
    return BenchmarkExample(
        id=f"paintings_{level.lower()}_{idx:04d}",
        domain=PAINTINGS_DOMAIN,
        complexity=level,
        query_text_ru=spec["query_text_ru"],
        constraints=spec["constraints"],
        requested_count=k,
        gold_answer_qids=[q for q, _, _ in gold],
        gold_answer_labels_ru=[ru for _, ru, _ in gold],
        sparql_query=sparql,
        created_at=_now_iso(),
        query_text_en=spec["query_text_en"],
        gold_answer_labels_en=[en for _, _, en in gold],
        is_advanced=level in {"L3", "L4", "L5"},
        template_id=spec["template_id"],
        template_family=spec["template_family"],
        gold_truncated=bool(truncated),
        ask_validator_sparql=ask,
        local_validator=_painting_local_validator(spec),
        gold_collection_meta=meta,
    )


def build_painting_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    specs: List[Dict[str, Any]] = []
    for tpl in PAINTING_DIRECT_TEMPLATES:
        level = tpl["level"]
        k = PAINTINGS_REQUESTED_COUNT[level]
        # Let the seed pool propose candidates, but prefer specs with enough local support.
        local_max = int(tpl["max_n"] * (2.5 if level in {"L4", "L5"} else 2.0))
        grp = _painting_group_candidates(
            paintings_seed_df,
            tpl["fields"],
            min_n=max(2, min(int(tpl["min_n"]), k)),
            max_n=local_max,
            max_rows=2600 if level in {"L4", "L5"} else 1800,
        )
        if len(grp) == 0:
            continue
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_painting_direct_spec(tpl, row)
            if not spec:
                continue
            local_n = _safe_int(spec.get("local_n"), 0) or 0
            # For L1/L2/L3, seed pool local_n is reliable enough: don't waste WDQS on exact-count candidates.
            if level in {"L1", "L2", "L3"} and local_n < k + PAINTINGS_MIN_GOLD_HEADROOM:
                continue
            # For hard levels keep slightly smaller candidates, but sort them later.
            if level in {"L4", "L5"} and local_n < max(2, k):
                continue
            specs.append(spec)
    return specs


PAINTING_TEMPLATE_SPEED_PRIORITY = defaultdict(lambda: 50, {
    "paintings_l1_by_material": 0,
    "paintings_l1_by_collection": 1,
    "paintings_l1_by_creator": 2,
    "paintings_l1_by_movement": 3,
    "paintings_l1_by_creator_citizenship": 6,
    "paintings_l1_by_genre": 8,
    "paintings_l2_material_collection_country": 0,
    "paintings_l2_creator_citizenship_collection_country": 1,
    "paintings_l2_movement_collection_country": 2,
    "paintings_l2_genre_collection_country": 3,
    "paintings_l2_material_creator_citizenship": 4,
    "paintings_l3_material_collection_country_period": 0,
    "paintings_l3_creator_citizenship_collection_country_period": 1,
    "paintings_l3_genre_collection_country_period": 2,
    "paintings_l3_material_creator_citizenship_collection_country": 3,
    "paintings_l4_material_creator_citizenship_collection_country_period": 0,
    "paintings_l4_creator_citizenship_collection_country_material_period": 1,
    "paintings_l4_genre_material_collection_country_period": 2,
    "paintings_l4_genre_creator_citizenship_collection_country_period": 3,
    "paintings_l5_material_creator_citizenship_collection_country_period": 0,
    "paintings_l5_genre_material_collection_country_period": 1,
    "paintings_l5_genre_material_creator_citizenship_collection_country_period": 2,
})


def _painting_spec_speed_key(spec: Dict[str, Any]) -> Tuple[int, int, int, str]:
    level = str(spec.get("complexity"))
    priority = int(PAINTING_TEMPLATE_SPEED_PRIORITY[str(spec.get("template_id", ""))])
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    k = PAINTINGS_REQUESTED_COUNT.get(level, 3)
    cap = PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 200)
    if local_n < k + PAINTINGS_MIN_GOLD_HEADROOM:
        bucket = 3
    elif local_n > cap:
        bucket = 2
    elif local_n <= max(35, cap // 3):
        bucket = 0
    else:
        bucket = 1
    # Prefer compact-but-not-tiny gold sets first.
    ideal = {"L1": 35, "L2": 28, "L3": 24, "L4": 18, "L5": 16}.get(level, 25)
    return (bucket, priority, abs(int(local_n) - ideal), str(spec.get("template_id", "")))


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    t0 = time.time()
    print("Building paintings candidate queue (v7 stable fill)...")
    rng = random.Random(seed)
    direct = build_painting_direct_candidates(rng)
    # Keep bridge off: previous v3/v5 bridge caused many semantic duplicates.
    bridge: List[Dict[str, Any]] = []
    print(f"  direct candidate specs: {len(direct)}")
    print(f"  bridge candidate specs: {len(bridge)}")
    all_specs = direct + bridge
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen_surface = set()
    seen_semantic = set()
    for spec in all_specs:
        surface_key = json.dumps({"level": spec["complexity"], "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
        semantic_key = _painting_semantic_key_from_spec(spec) if "_painting_semantic_key_from_spec" in globals() else surface_key
        if surface_key in seen_surface or semantic_key in seen_semantic:
            continue
        seen_surface.add(surface_key)
        seen_semantic.add(semantic_key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        rng.shuffle(by_level[level])
        by_level[level].sort(key=_painting_spec_speed_key)
        preview = [(s.get("template_id"), s.get("local_n")) for s in by_level[level][:5]]
        print(f"  {level} top candidates:", preview)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()}, f"built in {time.time()-t0:.1f}s")
    return by_level


def _count_painting_records(path: Path) -> int:
    try:
        rows = _read_jsonl(path)
        return sum(1 for r in rows if r.get("domain") == PAINTINGS_DOMAIN)
    except Exception:
        return 0


def _recover_best_existing_paintings_jsonl(output_path: Path) -> None:
    """If v5/v6 archived a good run, restore the best file so we resume instead of starting from zero."""
    if output_path.exists() and _count_painting_records(output_path) > 0:
        return
    candidates = []
    for p in output_path.parent.glob("paintings*.jsonl*"):
        if p.name == output_path.name:
            continue
        if p.is_file():
            n = _count_painting_records(p)
            if n > 0:
                candidates.append((n, p.stat().st_mtime, p))
    if not candidates:
        return
    candidates.sort(reverse=True)
    n, _, best = candidates[0]
    shutil.copy2(best, output_path)
    print(f"Recovered {n} existing paintings records from archive/cache: {best.name}")


def generate_paintings_dataset(
    output_path: Path = PAINTINGS_OUTPUT_PATH,
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
    seed: int = 20260528,
    require_complete_gold: bool = True,
    reset_output: bool = False,
) -> List[Dict[str, Any]]:
    rng = random.Random(seed)
    if reset_output:
        _archive_existing_jsonl(output_path, suffix="pre_v7")
    else:
        _recover_best_existing_paintings_jsonl(output_path)

    existing = _read_jsonl(output_path)
    if existing:
        existing_problems = validate_jsonl_exact_format(output_path)
        if len(existing_problems) > 0:
            raise AssertionError(f"Existing JSONL has schema/quality problems; rerun with reset_output=True. Preview: {existing_problems.head(10).to_dict('records')}")
    seen_keys = {_record_key(r) for r in existing}
    # Also dedupe by semantic constraints, to avoid seed/surface duplicates.
    seen_semantic = set()
    for r in existing:
        try:
            seen_semantic.add(json.dumps({"level": r.get("complexity"), "constraints": r.get("constraints")}, ensure_ascii=False, sort_keys=True))
        except Exception:
            pass

    counts = Counter(r.get("complexity") for r in existing if r.get("domain") == PAINTINGS_DOMAIN)
    next_idx = defaultdict(lambda: 1)
    for r in existing:
        if r.get("domain") != PAINTINGS_DOMAIN:
            continue
        m = re.search(r"_(l\d)_(\d{4})$", str(r.get("id", "")))
        if m:
            level = m.group(1).upper()
            next_idx[level] = max(next_idx[level], int(m.group(2)) + 1)

    print("Existing counts:", dict(counts))
    print("Building candidate queue before WDQS generation...")
    queue = build_painting_candidate_queue(seed=seed)
    audit_skips = []
    total_target = sum(target_per_level.values())
    current_total = sum(min(counts.get(level, 0), target) for level, target in target_per_level.items())
    pbar = tqdm(total=total_target, initial=current_total, desc="paintings total")

    for level, target in target_per_level.items():
        accepted_for_level = int(counts.get(level, 0))
        attempts = 0
        rejected = 0
        reject_reasons = Counter()
        specs = list(queue.get(level, []))
        spec_i = 0
        while accepted_for_level < int(target) and attempts < PAINTINGS_MAX_ATTEMPTS_PER_LEVEL:
            attempts += 1
            if not specs:
                audit_skips.append({"level": level, "reason": "no_candidate_specs"})
                break
            spec = specs[spec_i % len(specs)]
            spec_i += 1
            proposed_key = json.dumps({"domain": PAINTINGS_DOMAIN, "complexity": level, "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
            semantic_key = json.dumps({"level": level, "constraints": spec.get("constraints")}, ensure_ascii=False, sort_keys=True)
            if proposed_key in seen_keys or semantic_key in seen_semantic:
                continue
            try:
                ex = _finalize_painting_spec(spec, next_idx[level], require_complete=require_complete_gold)
            except Exception as e:
                reason = "exception"
                reject_reasons[reason] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "error": str(e)[:500], "constraints": spec.get("constraints")})
                rejected += 1
                pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} last={reason}")
                continue
            if ex is None:
                reason = spec.get("_last_reject_reason") or "gold_rejected_or_incomplete"
                reject_reasons[str(reason).split()[0]] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "local_n": spec.get("local_n"), "constraints": spec.get("constraints")})
                rejected += 1
                if attempts % 10 == 0:
                    pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} reasons={dict(reject_reasons.most_common(2))}")
                continue
            rec = _example_to_record(ex)
            if not _schema_is_exact(rec):
                raise AssertionError("Generated record field order does not match BenchmarkExample")
            key = _record_key(rec)
            if key in seen_keys:
                continue
            _append_jsonl(output_path, rec)
            existing.append(rec)
            seen_keys.add(key)
            seen_keys.add(proposed_key)
            seen_semantic.add(semantic_key)
            next_idx[level] += 1
            counts[level] += 1
            accepted_for_level += 1
            pbar.update(1)
            pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} gold={len(ex.gold_answer_qids)}")
        if accepted_for_level < int(target):
            print(f"[WARN] paintings {level}: accepted {accepted_for_level}/{target}; reject reasons: {dict(reject_reasons.most_common(8))}")
    pbar.close()

    audit = {
        "domain": PAINTINGS_DOMAIN,
        "output_path": str(output_path),
        "target_per_level": target_per_level,
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in existing if r.get("domain") == PAINTINGS_DOMAIN)),
        "seed_pool": paintings_seed_audit(),
        "candidate_counts": {lvl: len(v) for lvl, v in queue.items()},
        "skipped_count": len(audit_skips),
        "skipped_preview": audit_skips[-400:],
        "updated_at": _now_iso(),
        "format_fields": BENCHMARK_FIELD_ORDER,
    }
    format_problems = validate_jsonl_exact_format(output_path)
    if len(format_problems) > 0:
        raise AssertionError(f"Generated JSONL format mismatch: {format_problems.head(10).to_dict('records')}")
    _write_json(PAINTINGS_AUDIT_PATH, audit)
    print(f"saved incrementally: {output_path}")
    print(f"audit: {PAINTINGS_AUDIT_PATH}")
    return existing

print("✅ Applied paintings v7 stable-fill overrides")


✅ Applied paintings v7 stable-fill overrides


In [7]:

# ============================================================
# v8 pragmatic stable-fill patch for paintings:
# - v7 still spent minutes on WDQS/API exceptions while trying to fill L3/L4.
# - This patch uses a faster exact query for NEW fill records: direct P31=painting
#   + WDQS labels in the same query. That avoids the slow wbgetentities label pass
#   and avoids the expensive subclass-closure planning that was causing timeouts.
# - Existing recovered records are kept unchanged. New records remain exact-WDQS:
#   SELECT and ASK use the same direct painting class condition.
# ============================================================

PAINTINGS_TARGET_PER_LEVEL = {
    "L1": 20,
    "L2": 22,
    "L3": 24,
    "L4": 22,
    "L5": 22,
}

PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 45
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 2
PAINTINGS_FAST_QID_FIRST_GOLD = False
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 2500

# Keep enough room for complete golds but do not make L4/L5 too strict while filling.
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 300,
    "L2": 230,
    "L3": 180,
    "L4": 160,
    "L5": 130,
}
PAINTINGS_MIN_GOLD_HEADROOM = 2


def _build_paintings_gold_sparql(where_lines: Sequence[str], limit: int) -> str:
    """Exact public query for v8 fill records: direct painting class + labels in WDQS.

    Direct P31 is intentionally used only for new fill records because it is much
    faster and still yields a self-contained exact validator. Existing records keep
    their original subclass-closure SPARQL.
    """
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      {where}
      ?item wdt:P31 wd:{Q_PAINTING} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _build_paintings_item_sparql(where_lines: Sequence[str], limit: int) -> str:
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item WHERE {{
      {where}
      ?item wdt:P31 wd:{Q_PAINTING} .
    }}
    LIMIT {int(limit)}
    """.strip()


def _build_paintings_ask(where_lines: Sequence[str]) -> str:
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?item)
      {where}
      ?item wdt:P31 wd:{Q_PAINTING} .
    }}
    """.strip()


def _paintings_sparql_select_generation(query: str) -> dict:
    wd_obj = globals().get("wd")
    old_timeout = getattr(wd_obj, "timeout", None) if wd_obj is not None else None
    old_retries = getattr(wd_obj, "max_retries", None) if wd_obj is not None else None
    try:
        if wd_obj is not None and old_timeout is not None:
            wd_obj.timeout = int(PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS)
        if wd_obj is not None and old_retries is not None:
            wd_obj.max_retries = int(PAINTINGS_GENERATION_WDQS_MAX_RETRIES)
        return (wd_obj or wd).sparql_select(query, use_cache=True)
    finally:
        if wd_obj is not None and old_timeout is not None:
            wd_obj.timeout = old_timeout
        if wd_obj is not None and old_retries is not None:
            wd_obj.max_retries = old_retries


def _run_paintings_gold_query(spec: Dict[str, Any], gold_limit: int):
    """Run one exact WDQS query and use labels returned by WDQS.

    This intentionally avoids wbgetentities label fetching, which was the source of
    many long exceptions in v6/v7. If WDQS returns LIMIT+1 rows, the candidate is
    still rejected as potentially incomplete.
    """
    wdqs_limit = int(gold_limit) + 1
    public_sparql = _build_paintings_gold_sparql(spec["where_lines"], wdqs_limit)
    t0 = time.time()
    rows = rows_from_select(_paintings_sparql_select_generation(public_sparql))
    wdqs_seconds = round(time.time() - t0, 3)

    items: List[Tuple[str, str, str]] = []
    seen = set()
    dropped_no_qid = 0
    dropped_no_en = 0
    label_sources = Counter()
    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        if not _good_qid(qid):
            dropped_no_qid += 1
            continue
        if qid in seen:
            continue
        seen.add(qid)
        en = str(r.get("itemLabelEn") or "").strip()
        ru = str(r.get("itemLabelRu") or "").strip()
        if not _good_label(en):
            dropped_no_en += 1
            continue
        if not _good_label(ru):
            ru = en
            label_sources["en_fallback_for_ru"] += 1
        else:
            label_sources["ru_label"] += 1
        items.append((qid, ru, en))
        if len(items) >= int(gold_limit):
            break

    truncated = len(rows) >= wdqs_limit
    return public_sparql, items[:int(gold_limit)], truncated, {
        "wdqs_candidate_limit": wdqs_limit,
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "label_sources": dict(label_sources),
        "gold_may_be_incomplete_due_to_wdqs_limit": bool(truncated),
        "gold_query_execution": "wdqs_direct_p31_with_labels_v8",
        "wdqs_seconds": wdqs_seconds,
        "label_fetch_seconds": 0,
    }


def _painting_spec_speed_key(spec: Dict[str, Any]) -> Tuple[int, int, int, str]:
    level = str(spec.get("complexity"))
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    k = PAINTINGS_REQUESTED_COUNT.get(level, 3)
    cap = PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 200)
    priority = int(PAINTING_TEMPLATE_SPEED_PRIORITY[str(spec.get("template_id", ""))])
    # Prefer candidates with real headroom, but do not start with near-cap broad sets.
    if local_n < k + PAINTINGS_MIN_GOLD_HEADROOM:
        bucket = 3
    elif local_n > cap:
        bucket = 2
    elif local_n <= max(45, cap // 2):
        bucket = 0
    else:
        bucket = 1
    ideal = {"L1": 45, "L2": 35, "L3": 28, "L4": 22, "L5": 18}.get(level, 25)
    return (bucket, priority, abs(int(local_n) - ideal), str(spec.get("template_id", "")))

print("✅ Applied paintings v8 pragmatic stable-fill overrides: direct P31 + WDQS labels")


✅ Applied paintings v8 pragmatic stable-fill overrides: direct P31 + WDQS labels


In [8]:
# ============================================================
# v9 quality-target patch for paintings:
# - uses the requested target distribution 15/22/25/25/25;
# - curates existing/resumed JSONL before continuing generation;
# - removes exact/semantic/gold-set duplicates and the old duplicated L5 seed family;
# - enforces target caps per level before filling missing records;
# - rejects new records whose gold set is a duplicate or near-duplicate of kept records;
# - uses direct-only L4/L5 candidate filling to avoid seed-surface duplicates.
# ============================================================

PAINTINGS_TARGET_PER_LEVEL = {
    "L1": 15,
    "L2": 22,
    "L3": 25,
    "L4": 25,
    "L5": 25,
}

# Keep v8's pragmatic WDQS mode, but make the generator more deterministic and duplicate-aware.
PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 45
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 2
PAINTINGS_FAST_QID_FIRST_GOLD = False
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 3500
PAINTINGS_MIN_GOLD_HEADROOM = 2
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 300,
    "L2": 230,
    "L3": 180,
    "L4": 160,
    "L5": 130,
}

# Old v3/v4 L5 seed records were mostly identical resolved tasks with different seed painting names.
PAINTINGS_DROP_EXISTING_TEMPLATE_IDS = {
    "paintings_l5_seed_genre_seed_creator_country_collection_country_material_period",
}
PAINTINGS_LIMIT_ONE_PER_GOLD_TEMPLATE_IDS = {
    "paintings_l4_seed_creator_country_genre_collection_country",
    "paintings_l4_seed_genre_creator_country_collection_country_period",
}

CONTROL_CHARS_RE = re.compile(r"[\u200e\u200f\u202a-\u202e\ufeff]")


def _clean_text_deep(obj: Any) -> Any:
    if isinstance(obj, str):
        return CONTROL_CHARS_RE.sub("", obj).strip()
    if isinstance(obj, list):
        return [_clean_text_deep(x) for x in obj]
    if isinstance(obj, dict):
        return {k: _clean_text_deep(v) for k, v in obj.items()}
    return obj


def _normalize_record_text(rec: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(rec)
    for k in ("query_text_ru", "query_text_en"):
        if k in out:
            out[k] = _clean_text_deep(out[k])
    for k in ("constraints", "gold_answer_labels_ru", "gold_answer_labels_en", "local_validator", "gold_collection_meta"):
        if k in out:
            out[k] = _clean_text_deep(out[k])
    return {field: out.get(field) for field in BENCHMARK_FIELD_ORDER}


def _gold_sig_from_record(rec: Dict[str, Any]) -> Tuple[str, ...]:
    return tuple(sorted(str(q) for q in (rec.get("gold_answer_qids") or []) if str(q).strip()))


def _gold_sig_from_example(ex: BenchmarkExample) -> Tuple[str, ...]:
    return tuple(sorted(str(q) for q in (ex.gold_answer_qids or []) if str(q).strip()))


def _gold_jaccard(a: Sequence[str], b: Sequence[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def _semantic_payload_from_constraints_and_meta(rec: Dict[str, Any]) -> Dict[str, Any]:
    """Global semantic key: no complexity component, and no seed/exclude surface noise.

    This prevents L4/L5 duplicates with identical resolved constraints and prevents
    seed-only variants where the visible seed painting changes but the resolved filter
    and/or gold set is the same.
    """
    constraints = dict(rec.get("constraints") or {})
    meta = rec.get("gold_collection_meta") or {}
    cq = dict((meta.get("constraint_qids") or {}))

    # Prefer resolved QIDs for semantic dedupe; keep meaningful surface constraints as fallback.
    resolved = {}
    for k, v in cq.items():
        if k in {"answer_type", "exclude_paintings"} or str(k).startswith("same_"):
            continue
        resolved[k] = v
    surface = {}
    for k, v in constraints.items():
        if k in {"answer_type", "exclude_paintings"} or str(k).startswith("same_"):
            continue
        surface[k] = v
    return {
        "domain": PAINTINGS_DOMAIN,
        "resolved_qids": resolved,
        "surface": surface,
    }


def _record_global_semantic_key(rec: Dict[str, Any]) -> str:
    return json.dumps(_semantic_payload_from_constraints_and_meta(rec), ensure_ascii=False, sort_keys=True)


def _spec_global_semantic_key(spec: Dict[str, Any]) -> str:
    fake_rec = {
        "constraints": spec.get("constraints") or {},
        "gold_collection_meta": {"constraint_qids": spec.get("constraint_qids") or {}},
    }
    return _record_global_semantic_key(fake_rec)


def _record_quality_sort_key(rec: Dict[str, Any]) -> Tuple[int, int, int, int, int, str]:
    """Lower is better. Used only for curating existing records before fill."""
    level = str(rec.get("complexity"))
    template_id = str(rec.get("template_id") or "")
    g = len(rec.get("gold_answer_qids") or [])
    requested = int(rec.get("requested_count") or PAINTINGS_REQUESTED_COUNT.get(level, 3))
    ideal = {"L1": 45, "L2": 35, "L3": 28, "L4": 24, "L5": 24}.get(level, 30)
    drop_template_penalty = 1000 if template_id in PAINTINGS_DROP_EXISTING_TEMPLATE_IDS else 0
    seed_penalty = 8 if "_seed_" in template_id else 0
    too_few_penalty = max(0, requested + PAINTINGS_MIN_GOLD_HEADROOM - g) * 50
    broad_penalty = max(0, g - PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 9999)) * 20
    # L1 broad tasks are less useful; this makes the 15 kept L1 examples more compact.
    if level == "L1":
        broad_penalty += max(0, g - 150) * 2
    return (
        drop_template_penalty,
        too_few_penalty + broad_penalty,
        seed_penalty,
        abs(g - ideal),
        g,
        str(rec.get("id") or ""),
    )


def _records_near_duplicate(rec: Dict[str, Any], kept_gold_sigs: List[Tuple[str, ...]]) -> bool:
    sig = _gold_sig_from_record(rec)
    if not sig:
        return False
    ss = set(sig)
    for old in kept_gold_sigs:
        if not old:
            continue
        so = set(old)
        # Exact duplicate or very high overlap. For small gold sets, exact overlap is enough;
        # for larger gold sets, 90%+ is effectively the same benchmark task.
        j = len(ss & so) / len(ss | so)
        if sig == old or j >= 0.90:
            return True
    return False


def curate_existing_paintings_records(
    records: List[Dict[str, Any]],
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Remove bad/duplicate/overflow records before resume-fill.

    This is intentionally conservative: keep good existing records, but remove the
    known duplicated L5 seed family, exact cross-level duplicates, identical/near-identical
    gold sets, and extra records above the requested target counts.
    """
    normalized = [_normalize_record_text(r) for r in records if r.get("domain") == PAINTINGS_DOMAIN]
    # Validate shape lightly; full exact validation runs after writing.
    candidates = [r for r in normalized if _schema_is_exact(r) and r.get("complexity") in target_per_level]
    candidates.sort(key=_record_quality_sort_key)

    kept: List[Dict[str, Any]] = []
    dropped: List[Dict[str, Any]] = []
    counts = Counter()
    seen_record_keys = set()
    seen_semantic = set()
    seen_gold_sigs: List[Tuple[str, ...]] = []
    template_gold_kept = set()

    for rec in candidates:
        level = str(rec.get("complexity"))
        rid = rec.get("id")
        template_id = str(rec.get("template_id") or "")
        reason = None

        if counts[level] >= int(target_per_level[level]):
            reason = "over_target_for_level"
        elif template_id in PAINTINGS_DROP_EXISTING_TEMPLATE_IDS:
            reason = "drop_old_seed_duplicate_template"
        else:
            rkey = _record_key(rec)
            skey = _record_global_semantic_key(rec)
            gsig = _gold_sig_from_record(rec)
            tg = (template_id, gsig)
            if rkey in seen_record_keys:
                reason = "duplicate_record_key"
            elif skey in seen_semantic:
                reason = "duplicate_resolved_constraints"
            elif template_id in PAINTINGS_LIMIT_ONE_PER_GOLD_TEMPLATE_IDS and tg in template_gold_kept:
                reason = "template_gold_duplicate"
            elif _records_near_duplicate(rec, seen_gold_sigs):
                reason = "duplicate_or_near_duplicate_gold_set"

        if reason:
            dropped.append({
                "id": rid,
                "complexity": level,
                "template_id": template_id,
                "reason": reason,
                "gold": len(rec.get("gold_answer_qids") or []),
                "constraints": rec.get("constraints"),
            })
            continue

        kept.append(rec)
        counts[level] += 1
        seen_record_keys.add(_record_key(rec))
        seen_semantic.add(_record_global_semantic_key(rec))
        seen_gold_sigs.append(_gold_sig_from_record(rec))
        template_gold_kept.add((template_id, _gold_sig_from_record(rec)))

    # Restore stable output order by level then old numeric id, not by quality score.
    def order_key(rec):
        level_rank = {"L1": 1, "L2": 2, "L3": 3, "L4": 4, "L5": 5}.get(rec.get("complexity"), 9)
        m = re.search(r"_(l\d)_(\d{4})$", str(rec.get("id", "")))
        idx = int(m.group(2)) if m else 9999
        return (level_rank, idx, str(rec.get("id") or ""))

    kept.sort(key=order_key)
    return kept, dropped


def _write_jsonl_exact(path: Path, records: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps({k: rec.get(k) for k in BENCHMARK_FIELD_ORDER}, ensure_ascii=False) + "\n")
    tmp.replace(path)


def _new_example_is_duplicate(ex: BenchmarkExample, rec: Dict[str, Any], seen_semantic: set, seen_gold_sigs: List[Tuple[str, ...]]) -> Tuple[bool, str]:
    skey = _record_global_semantic_key(rec)
    if skey in seen_semantic:
        return True, "duplicate_resolved_constraints"
    gsig = _gold_sig_from_example(ex)
    ss = set(gsig)
    for old in seen_gold_sigs:
        if not old:
            continue
        so = set(old)
        j = len(ss & so) / len(ss | so)
        if gsig == old or j >= 0.90:
            return True, "duplicate_or_near_duplicate_gold_set"
    return False, ""


# Use direct templates only during fill. This avoids the old seed-surface L4/L5 duplicates.
def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    print("Building paintings candidate queue (v9 quality target direct-fill)...")
    all_specs = build_painting_direct_candidates(rng)
    print(f"  direct candidate specs: {len(all_specs)}")
    print("  bridge candidate specs: 0 (disabled in v9 to avoid seed-only duplicates)")
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen = set()
    for spec in all_specs:
        key = _spec_global_semantic_key(spec)
        # Do not mix exact same resolved constraints across levels.
        if key in seen:
            continue
        seen.add(key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        rng.shuffle(by_level[level])
        by_level[level].sort(key=_painting_spec_speed_key)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()})
    return by_level


def generate_paintings_dataset(
    output_path: Path = PAINTINGS_OUTPUT_PATH,
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
    seed: int = 20260528,
    require_complete_gold: bool = True,
    reset_output: bool = False,
) -> List[Dict[str, Any]]:
    if reset_output:
        _archive_existing_jsonl(output_path, suffix="pre_v9")
    else:
        _recover_best_existing_paintings_jsonl(output_path)

    raw_existing = _read_jsonl(output_path)
    if raw_existing:
        # Exact schema validation before curation; if old file is malformed, stop.
        existing_problems = validate_jsonl_exact_format(output_path)
        if len(existing_problems) > 0:
            raise AssertionError(f"Existing JSONL has schema/quality problems; rerun with reset_output=True. Preview: {existing_problems.head(10).to_dict('records')}")

    curated, dropped_existing = curate_existing_paintings_records(raw_existing, target_per_level=target_per_level)
    if raw_existing:
        _write_jsonl_exact(output_path, curated)
        print(f"Curated existing paintings before fill: {len(raw_existing)} -> {len(curated)} kept; dropped {len(dropped_existing)}")
        print("Curated counts:", dict(Counter(r.get("complexity") for r in curated)))

    existing = list(curated)
    seen_keys = {_record_key(r) for r in existing}
    seen_semantic = {_record_global_semantic_key(r) for r in existing}
    seen_gold_sigs = [_gold_sig_from_record(r) for r in existing]

    counts = Counter(r.get("complexity") for r in existing if r.get("domain") == PAINTINGS_DOMAIN)
    next_idx = defaultdict(lambda: 1)
    for r in existing:
        if r.get("domain") != PAINTINGS_DOMAIN:
            continue
        m = re.search(r"_(l\d)_(\d{4})$", str(r.get("id", "")))
        if m:
            level = m.group(1).upper()
            next_idx[level] = max(next_idx[level], int(m.group(2)) + 1)

    print("Existing counts after curation:", dict(counts))
    print("Building candidate queue before WDQS generation...")
    queue = build_painting_candidate_queue(seed=seed)
    audit_skips = list(dropped_existing)
    total_target = sum(target_per_level.values())
    current_total = sum(min(counts.get(level, 0), target) for level, target in target_per_level.items())
    pbar = tqdm(total=total_target, initial=current_total, desc="paintings total")

    for level, target in target_per_level.items():
        accepted_for_level = int(counts.get(level, 0))
        attempts = 0
        rejected = 0
        reject_reasons = Counter()
        specs = list(queue.get(level, []))
        spec_i = 0
        while accepted_for_level < int(target) and attempts < PAINTINGS_MAX_ATTEMPTS_PER_LEVEL:
            attempts += 1
            if not specs:
                audit_skips.append({"level": level, "reason": "no_candidate_specs"})
                break
            spec = specs[spec_i % len(specs)]
            spec_i += 1
            proposed_key = json.dumps({"domain": PAINTINGS_DOMAIN, "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
            semantic_key = _spec_global_semantic_key(spec)
            if proposed_key in seen_keys or semantic_key in seen_semantic:
                continue
            try:
                ex = _finalize_painting_spec(spec, next_idx[level], require_complete=require_complete_gold)
            except Exception as e:
                reason = "exception"
                reject_reasons[reason] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "error": str(e)[:500], "constraints": spec.get("constraints")})
                rejected += 1
                pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} last={reason}")
                continue
            if ex is None:
                reason = spec.get("_last_reject_reason") or "gold_rejected_or_incomplete"
                reject_reasons[str(reason).split()[0]] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "local_n": spec.get("local_n"), "constraints": spec.get("constraints")})
                rejected += 1
                if attempts % 10 == 0:
                    pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} reasons={dict(reject_reasons.most_common(2))}")
                continue
            rec = _example_to_record(ex)
            rec = _normalize_record_text(rec)
            if not _schema_is_exact(rec):
                raise AssertionError("Generated record field order does not match BenchmarkExample")
            dup, dup_reason = _new_example_is_duplicate(ex, rec, seen_semantic, seen_gold_sigs)
            if dup:
                reject_reasons[dup_reason] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": dup_reason, "gold": len(ex.gold_answer_qids), "constraints": rec.get("constraints")})
                rejected += 1
                continue
            key = _record_key(rec)
            if key in seen_keys:
                continue
            _append_jsonl(output_path, rec)
            existing.append(rec)
            seen_keys.add(key)
            seen_keys.add(proposed_key)
            seen_semantic.add(_record_global_semantic_key(rec))
            seen_gold_sigs.append(_gold_sig_from_record(rec))
            next_idx[level] += 1
            counts[level] += 1
            accepted_for_level += 1
            pbar.update(1)
            pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} gold={len(ex.gold_answer_qids)}")
        if accepted_for_level < int(target):
            print(f"[WARN] paintings {level}: accepted {accepted_for_level}/{target}; reject reasons: {dict(reject_reasons.most_common(8))}")
    pbar.close()

    final_rows = _read_jsonl(output_path)
    final_curated, final_dropped = curate_existing_paintings_records(final_rows, target_per_level=target_per_level)
    if len(final_curated) != len(final_rows):
        _write_jsonl_exact(output_path, final_curated)
        audit_skips.extend(final_dropped)
        final_rows = final_curated

    audit = {
        "domain": PAINTINGS_DOMAIN,
        "output_path": str(output_path),
        "target_per_level": target_per_level,
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in final_rows if r.get("domain") == PAINTINGS_DOMAIN)),
        "seed_pool": paintings_seed_audit(),
        "candidate_counts": {lvl: len(v) for lvl, v in queue.items()},
        "skipped_count": len(audit_skips),
        "skipped_preview": audit_skips[-800:],
        "updated_at": _now_iso(),
        "format_fields": BENCHMARK_FIELD_ORDER,
        "quality_patch": "v9_target_15_22_25_25_25_semantic_gold_dedupe",
    }
    format_problems = validate_jsonl_exact_format(output_path)
    if len(format_problems) > 0:
        raise AssertionError(f"Generated JSONL format mismatch: {format_problems.head(10).to_dict('records')}")
    _write_json(PAINTINGS_AUDIT_PATH, audit)
    print(f"saved incrementally: {output_path}")
    print(f"audit: {PAINTINGS_AUDIT_PATH}")
    print("Final counts:", dict(Counter(r.get("complexity") for r in final_rows)))
    return final_rows

print("✅ Applied paintings v9 quality-target patch: target 15/22/25/25/25 + curation + semantic/gold dedupe")


✅ Applied paintings v9 quality-target patch: target 15/22/25/25/25 + curation + semantic/gold dedupe


In [9]:

# ============================================================
# v10 date-diversity patch for paintings:
# - replaces broad century buckets like 1400-1599 with diverse 25/50-year windows;
# - recalculates cached seed-pool year buckets in-memory, so old caches are safe;
# - caps repeated date ranges during curation and during fill generation;
# - keeps the requested target distribution 15/22/25/25/25 and exact JSONL schema;
# - keeps v9 semantic/gold dedupe, but adds date-range diversity to avoid 1400-1599 spam.
# ============================================================

PAINTINGS_DATE_DIVERSITY_PATCH = "v10_diverse_25_50_year_windows"

# Exact target requested by the user.
PAINTINGS_TARGET_PER_LEVEL = {
    "L1": 15,
    "L2": 22,
    "L3": 25,
    "L4": 25,
    "L5": 25,
}

# Keep v9 quality settings, with enough caps for narrow windows to pass.
PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 45
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 2
PAINTINGS_FAST_QID_FIRST_GOLD = False
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 5000
PAINTINGS_MIN_GOLD_HEADROOM = 2
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 300,
    "L2": 230,
    "L3": 180,
    "L4": 160,
    "L5": 130,
}

# Final dataset should not have one date range dominating the whole domain.
# Narrow windows can repeat a little; broad legacy windows are heavily capped.
PAINTINGS_MAX_RECORDS_PER_NARROW_DATE_RANGE = 7
PAINTINGS_MAX_RECORDS_PER_MEDIUM_DATE_RANGE = 5
PAINTINGS_MAX_RECORDS_PER_BROAD_DATE_RANGE = 3
PAINTINGS_MAX_RECORDS_PER_LEGACY_1400_1599 = 2

# Windows intentionally include narrow 25/50-year periods. 1400-1599 is split
# into four separate 50-year windows, which fixes the old over-representation.
PAINTING_DATE_WINDOWS = [
    (1200, 1299),
    (1300, 1349), (1350, 1399),
    (1400, 1449), (1450, 1499), (1500, 1549), (1550, 1599),
    (1600, 1649), (1650, 1699),
    (1700, 1749), (1750, 1799),
    (1800, 1849),
    (1850, 1874), (1875, 1899),
    (1900, 1924), (1925, 1949),
    (1950, 1974), (1975, 1999), (2000, 2026),
]


def _painting_diverse_year_bucket(y: Optional[int]) -> Tuple[Optional[str], Optional[int], Optional[int], Optional[str], Optional[str]]:
    """Diverse date windows for paintings, replacing the old broad 1400-1599 bucket."""
    if y is None:
        return None, None, None, None, None
    y = _safe_int(y)
    if y is None or y < 1000 or y > 2026:
        return None, None, None, None, None
    for y1, y2 in PAINTING_DATE_WINDOWS:
        if y1 <= int(y) <= y2:
            key = f"{y1}_{y2}"
            label = f"{y1}–{y2}"
            return key, y1, y2, label, label
    # Very early dated works are rare in this benchmark; keep them in a broad but explicit bucket.
    if 1000 <= int(y) < 1200:
        return "1000_1199", 1000, 1199, "1000–1199", "1000–1199"
    return None, None, None, None, None

# Override the old helper globally so any future pool rebuild also uses the new ranges.
_year_bucket = _painting_diverse_year_bucket


def _refresh_paintings_seed_year_windows_v10(df: pd.DataFrame) -> pd.DataFrame:
    """Repair/replace year_bucket columns in a cached seed dataframe without rebuilding WDQS pools."""
    if df is None or len(df) == 0:
        return df
    out = df.copy()
    if "year" not in out.columns:
        return out
    buckets = out["year"].apply(lambda y: _painting_diverse_year_bucket(_safe_int(y)))
    out["year_bucket"] = buckets.apply(lambda t: t[0])
    out["year_from"] = buckets.apply(lambda t: t[1])
    out["year_to"] = buckets.apply(lambda t: t[2])
    out["year_ru"] = buckets.apply(lambda t: t[3])
    out["year_en"] = buckets.apply(lambda t: t[4])
    # Make sure numeric columns are usable by groupby/final filters.
    for c in ["year", "year_from", "year_to"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

try:
    paintings_seed_df = _refresh_paintings_seed_year_windows_v10(paintings_seed_df)
    _date_counts_preview = Counter(
        (int(r.year_from), int(r.year_to))
        for r in paintings_seed_df[["year_from", "year_to"]].dropna().itertuples(index=False)
    )
    print("✅ Recomputed paintings seed date windows (v10). Top windows:", _date_counts_preview.most_common(8))
except NameError:
    print("[WARN] paintings_seed_df is not loaded yet; v10 date windows will apply after seed load if this cell is rerun.")


def _date_range_key_from_constraints(constraints: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    y1 = _safe_int((constraints or {}).get("date_from"))
    y2 = _safe_int((constraints or {}).get("date_to"))
    if y1 is None or y2 is None:
        return None
    return (int(y1), int(y2))


def _record_date_range_key(rec: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    return _date_range_key_from_constraints(rec.get("constraints") or {})


def _spec_date_range_key(spec: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    return _date_range_key_from_constraints(spec.get("constraints") or {})


def _date_range_cap(date_key: Optional[Tuple[int, int]]) -> int:
    if date_key is None:
        return 10**9
    y1, y2 = date_key
    span = int(y2) - int(y1) + 1
    if (int(y1), int(y2)) == (1400, 1599):
        return PAINTINGS_MAX_RECORDS_PER_LEGACY_1400_1599
    if span >= 100:
        return PAINTINGS_MAX_RECORDS_PER_BROAD_DATE_RANGE
    if span >= 60:
        return PAINTINGS_MAX_RECORDS_PER_MEDIUM_DATE_RANGE
    return PAINTINGS_MAX_RECORDS_PER_NARROW_DATE_RANGE


def _date_span(date_key: Optional[Tuple[int, int]]) -> int:
    if not date_key:
        return 0
    return int(date_key[1]) - int(date_key[0]) + 1


def _record_quality_sort_key(rec: Dict[str, Any]) -> Tuple[int, int, int, int, int, int, str]:
    """Lower is better. Adds a strong penalty for broad/legacy date windows."""
    level = str(rec.get("complexity"))
    template_id = str(rec.get("template_id") or "")
    g = len(rec.get("gold_answer_qids") or [])
    requested = int(rec.get("requested_count") or PAINTINGS_REQUESTED_COUNT.get(level, 3))
    ideal = {"L1": 45, "L2": 35, "L3": 28, "L4": 24, "L5": 24}.get(level, 30)
    drop_template_penalty = 1000 if template_id in PAINTINGS_DROP_EXISTING_TEMPLATE_IDS else 0
    seed_penalty = 8 if "_seed_" in template_id else 0
    too_few_penalty = max(0, requested + PAINTINGS_MIN_GOLD_HEADROOM - g) * 50
    broad_gold_penalty = max(0, g - PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 9999)) * 20
    if level == "L1":
        broad_gold_penalty += max(0, g - 150) * 2
    dkey = _record_date_range_key(rec)
    span = _date_span(dkey)
    legacy_penalty = 250 if dkey == (1400, 1599) else 0
    broad_date_penalty = 80 if span >= 100 else (35 if span >= 60 else 0)
    no_date_penalty = 0 if level in {"L1", "L2"} else (20 if dkey is None else 0)
    return (
        drop_template_penalty,
        legacy_penalty + broad_date_penalty + no_date_penalty,
        too_few_penalty + broad_gold_penalty,
        seed_penalty,
        abs(g - ideal),
        g,
        str(rec.get("id") or ""),
    )


def curate_existing_paintings_records(
    records: List[Dict[str, Any]],
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """v10 curation: v9 duplicate cleanup + date-range balance."""
    normalized = [_normalize_record_text(r) for r in records if r.get("domain") == PAINTINGS_DOMAIN]
    candidates = [r for r in normalized if _schema_is_exact(r) and r.get("complexity") in target_per_level]
    candidates.sort(key=_record_quality_sort_key)

    kept: List[Dict[str, Any]] = []
    dropped: List[Dict[str, Any]] = []
    counts = Counter()
    date_counts = Counter()
    seen_record_keys = set()
    seen_semantic = set()
    seen_gold_sigs: List[Tuple[str, ...]] = []
    template_gold_kept = set()

    for rec in candidates:
        level = str(rec.get("complexity"))
        rid = rec.get("id")
        template_id = str(rec.get("template_id") or "")
        reason = None
        dkey = _record_date_range_key(rec)

        if counts[level] >= int(target_per_level[level]):
            reason = "over_target_for_level"
        elif template_id in PAINTINGS_DROP_EXISTING_TEMPLATE_IDS:
            reason = "drop_old_seed_duplicate_template"
        elif dkey is not None and date_counts[dkey] >= _date_range_cap(dkey):
            reason = f"date_range_overrepresented_{dkey[0]}_{dkey[1]}"
        else:
            rkey = _record_key(rec)
            skey = _record_global_semantic_key(rec)
            gsig = _gold_sig_from_record(rec)
            tg = (template_id, gsig)
            if rkey in seen_record_keys:
                reason = "duplicate_record_key"
            elif skey in seen_semantic:
                reason = "duplicate_resolved_constraints"
            elif template_id in PAINTINGS_LIMIT_ONE_PER_GOLD_TEMPLATE_IDS and tg in template_gold_kept:
                reason = "template_gold_duplicate"
            elif _records_near_duplicate(rec, seen_gold_sigs):
                reason = "duplicate_or_near_duplicate_gold_set"

        if reason:
            dropped.append({
                "id": rid,
                "complexity": level,
                "template_id": template_id,
                "reason": reason,
                "date_range": list(dkey) if dkey else None,
                "gold": len(rec.get("gold_answer_qids") or []),
                "constraints": rec.get("constraints"),
            })
            continue

        kept.append(rec)
        counts[level] += 1
        if dkey is not None:
            date_counts[dkey] += 1
        seen_record_keys.add(_record_key(rec))
        seen_semantic.add(_record_global_semantic_key(rec))
        seen_gold_sigs.append(_gold_sig_from_record(rec))
        template_gold_kept.add((template_id, _gold_sig_from_record(rec)))

    def order_key(rec):
        level_rank = {"L1": 1, "L2": 2, "L3": 3, "L4": 4, "L5": 5}.get(rec.get("complexity"), 9)
        m = re.search(r"_(l\d)_(\d{4})$", str(rec.get("id", "")))
        idx = int(m.group(2)) if m else 9999
        return (level_rank, idx, str(rec.get("id") or ""))

    kept.sort(key=order_key)
    print("v10 curation date counts:", dict(sorted(date_counts.items())[:12]), "... total date ranges", len(date_counts))
    return kept, dropped


# Direct template registry.  It keeps the successful v8/v9 direct-fill approach but
# will now use the recalculated fine-grained year_bucket values from paintings_seed_df.
PAINTING_DIRECT_TEMPLATES = [
    # L1: no period filters; target only 15, curation keeps the best compact tasks.
    {"level": "L1", "template_id": "paintings_l1_by_collection", "family": "single_collection", "fields": ["collection"], "min_n": 5, "max_n": 260},
    {"level": "L1", "template_id": "paintings_l1_by_creator", "family": "single_creator", "fields": ["creator"], "min_n": 5, "max_n": 260},
    {"level": "L1", "template_id": "paintings_l1_by_material", "family": "single_material", "fields": ["material"], "min_n": 5, "max_n": 260},
    {"level": "L1", "template_id": "paintings_l1_by_movement", "family": "single_movement", "fields": ["movement"], "min_n": 5, "max_n": 260},
    {"level": "L1", "template_id": "paintings_l1_by_creator_citizenship", "family": "creator_country", "fields": ["creator_country"], "min_n": 5, "max_n": 260},
    {"level": "L1", "template_id": "paintings_l1_by_genre", "family": "single_genre", "fields": ["genre"], "min_n": 5, "max_n": 260},

    # L2: two constraints, no period yet.
    {"level": "L2", "template_id": "paintings_l2_material_collection_country", "family": "material_collection_country", "fields": ["material", "collection_country"], "min_n": 5, "max_n": 220},
    {"level": "L2", "template_id": "paintings_l2_creator_citizenship_collection_country", "family": "creator_country_collection_country", "fields": ["creator_country", "collection_country"], "min_n": 5, "max_n": 220},
    {"level": "L2", "template_id": "paintings_l2_movement_collection_country", "family": "movement_collection_country", "fields": ["movement", "collection_country"], "min_n": 5, "max_n": 220},
    {"level": "L2", "template_id": "paintings_l2_genre_collection_country", "family": "genre_collection_country", "fields": ["genre", "collection_country"], "min_n": 5, "max_n": 220},
    {"level": "L2", "template_id": "paintings_l2_material_creator_citizenship", "family": "material_creator_country", "fields": ["material", "creator_country"], "min_n": 5, "max_n": 220},

    # L3: period queries use 25/50-year buckets, not old 1400-1599.
    {"level": "L3", "template_id": "paintings_l3_material_collection_country_period", "family": "material_collection_country_period", "fields": ["material", "collection_country", "year_bucket"], "min_n": 4, "max_n": 180},
    {"level": "L3", "template_id": "paintings_l3_genre_collection_country_period", "family": "genre_collection_country_period", "fields": ["genre", "collection_country", "year_bucket"], "min_n": 4, "max_n": 180},
    {"level": "L3", "template_id": "paintings_l3_creator_citizenship_collection_country_period", "family": "creator_country_collection_country_period", "fields": ["creator_country", "collection_country", "year_bucket"], "min_n": 4, "max_n": 180},
    {"level": "L3", "template_id": "paintings_l3_movement_creator_citizenship_period", "family": "movement_creator_country_period", "fields": ["movement", "creator_country", "year_bucket"], "min_n": 4, "max_n": 180},
    {"level": "L3", "template_id": "paintings_l3_depicts_collection_country_period", "family": "depicts_collection_country_period", "fields": ["depicts", "collection_country", "year_bucket"], "min_n": 4, "max_n": 180},
    {"level": "L3", "template_id": "paintings_l3_material_creator_citizenship_collection_country", "family": "material_creator_country_collection_country", "fields": ["material", "creator_country", "collection_country"], "min_n": 4, "max_n": 180},

    # L4: four constraints; period windows are narrow but still answerable.
    {"level": "L4", "template_id": "paintings_l4_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 160},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_citizenship_collection_country_period", "family": "genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 160},
    {"level": "L4", "template_id": "paintings_l4_genre_material_collection_country_period", "family": "genre_material_collection_country_period", "fields": ["genre", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 160},
    {"level": "L4", "template_id": "paintings_l4_movement_collection_country_material_period", "family": "movement_collection_country_material_period", "fields": ["movement", "collection_country", "material", "year_bucket"], "min_n": 3, "max_n": 160},
    {"level": "L4", "template_id": "paintings_l4_creator_citizenship_collection_country_material_period", "family": "creator_country_collection_country_material_period", "fields": ["creator_country", "collection_country", "material", "year_bucket"], "min_n": 3, "max_n": 160},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_birth_country_collection_country_period", "family": "genre_creator_birth_country_collection_country_period", "fields": ["genre", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 160},
    {"level": "L4", "template_id": "paintings_l4_depicts_collection_country_material_period", "family": "depicts_collection_country_material_period", "fields": ["depicts", "collection_country", "material", "year_bucket"], "min_n": 3, "max_n": 160},

    # L5: hard direct multihop, no seed-only surface variants.
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_citizenship_collection_country_period", "family": "genre_material_creator_country_collection_country_period", "fields": ["genre", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 130},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_citizenship_collection_country_period", "family": "movement_material_creator_country_collection_country_period", "fields": ["movement", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 130},
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_birth_country_collection_country_period", "family": "genre_material_creator_birth_country_collection_country_period", "fields": ["genre", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 130},
    {"level": "L5", "template_id": "paintings_l5_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 130},
    {"level": "L5", "template_id": "paintings_l5_depicts_material_creator_citizenship_collection_country_period", "family": "depicts_material_creator_country_collection_country_period", "fields": ["depicts", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 130},
]

# Speed priority updated for the v10 template set.
PAINTING_TEMPLATE_SPEED_PRIORITY = defaultdict(lambda: 50, {
    "paintings_l1_by_collection": 0,
    "paintings_l1_by_creator": 1,
    "paintings_l1_by_material": 2,
    "paintings_l1_by_movement": 3,
    "paintings_l1_by_creator_citizenship": 8,
    "paintings_l1_by_genre": 9,
    "paintings_l2_material_collection_country": 0,
    "paintings_l2_creator_citizenship_collection_country": 1,
    "paintings_l2_movement_collection_country": 2,
    "paintings_l2_genre_collection_country": 3,
    "paintings_l2_material_creator_citizenship": 4,
    "paintings_l3_material_collection_country_period": 0,
    "paintings_l3_creator_citizenship_collection_country_period": 1,
    "paintings_l3_genre_collection_country_period": 2,
    "paintings_l3_material_creator_citizenship_collection_country": 3,
    "paintings_l3_movement_creator_citizenship_period": 4,
    "paintings_l4_material_creator_citizenship_collection_country_period": 0,
    "paintings_l4_creator_citizenship_collection_country_material_period": 1,
    "paintings_l4_genre_material_collection_country_period": 2,
    "paintings_l4_genre_creator_citizenship_collection_country_period": 3,
    "paintings_l5_material_creator_citizenship_collection_country_period": 0,
    "paintings_l5_genre_material_creator_citizenship_collection_country_period": 1,
    "paintings_l5_genre_material_creator_birth_country_collection_country_period": 2,
    "paintings_l5_movement_material_creator_citizenship_collection_country_period": 3,
})


def _painting_spec_speed_key(spec: Dict[str, Any]) -> Tuple[int, int, int, int, str]:
    level = str(spec.get("complexity"))
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    k = PAINTINGS_REQUESTED_COUNT.get(level, 3)
    cap = PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 200)
    priority = int(PAINTING_TEMPLATE_SPEED_PRIORITY[str(spec.get("template_id", ""))])
    dkey = _spec_date_range_key(spec)
    span = _date_span(dkey)
    # Prefer narrow/medium windows for L3-L5; reject broad legacy period in sort.
    date_bucket = 0
    if dkey == (1400, 1599):
        date_bucket = 5
    elif span >= 100:
        date_bucket = 4
    elif span == 0:
        date_bucket = 2 if level in {"L3", "L4", "L5"} else 0
    elif span <= 60:
        date_bucket = 0
    else:
        date_bucket = 1
    if local_n < k + PAINTINGS_MIN_GOLD_HEADROOM:
        n_bucket = 3
    elif local_n > cap:
        n_bucket = 2
    elif local_n <= max(50, cap // 2):
        n_bucket = 0
    else:
        n_bucket = 1
    ideal = {"L1": 45, "L2": 35, "L3": 28, "L4": 22, "L5": 18}.get(level, 25)
    return (date_bucket, n_bucket, priority, abs(int(local_n) - ideal), str(spec.get("template_id", "")))


def _interleave_specs_by_date_range(specs: List[Dict[str, Any]], rng: random.Random) -> List[Dict[str, Any]]:
    groups: Dict[Any, List[Dict[str, Any]]] = defaultdict(list)
    for s in specs:
        groups[_spec_date_range_key(s)].append(s)
    for k in groups:
        rng.shuffle(groups[k])
        groups[k].sort(key=_painting_spec_speed_key)
    group_keys = list(groups.keys())
    rng.shuffle(group_keys)
    group_keys.sort(key=lambda k: (0 if k is not None else 1, _date_span(k), str(k)))
    out = []
    while any(groups.values()):
        for k in group_keys:
            if groups[k]:
                out.append(groups[k].pop(0))
    return out


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    print("Building paintings candidate queue (v10 date-diverse quality target)...")
    all_specs = build_painting_direct_candidates(rng)
    print(f"  direct candidate specs: {len(all_specs)}")
    print("  bridge candidate specs: 0 (disabled to avoid seed-only duplicates)")
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen = set()
    skipped_legacy_dates = 0
    for spec in all_specs:
        dkey = _spec_date_range_key(spec)
        # Do not create new broad legacy-period tasks. Existing ones are capped in curation.
        if dkey == (1400, 1599) or (_date_span(dkey) >= 100 and spec.get("complexity") in {"L3", "L4", "L5"}):
            skipped_legacy_dates += 1
            continue
        key = _spec_global_semantic_key(spec)
        if key in seen:
            continue
        seen.add(key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        by_level[level] = _interleave_specs_by_date_range(by_level[level], rng)
        preview = [(s.get("template_id"), s.get("local_n"), _spec_date_range_key(s)) for s in by_level[level][:8]]
        print(f"  {level} top candidates:", preview)
    print("  skipped broad legacy-date specs:", skipped_legacy_dates)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()})
    return by_level


def generate_paintings_dataset(
    output_path: Path = PAINTINGS_OUTPUT_PATH,
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
    seed: int = 20260528,
    require_complete_gold: bool = True,
    reset_output: bool = False,
) -> List[Dict[str, Any]]:
    if reset_output:
        _archive_existing_jsonl(output_path, suffix="pre_v10")
    else:
        _recover_best_existing_paintings_jsonl(output_path)

    raw_existing = _read_jsonl(output_path)
    if raw_existing:
        existing_problems = validate_jsonl_exact_format(output_path)
        if len(existing_problems) > 0:
            raise AssertionError(f"Existing JSONL has schema/quality problems; rerun with reset_output=True. Preview: {existing_problems.head(10).to_dict('records')}")

    curated, dropped_existing = curate_existing_paintings_records(raw_existing, target_per_level=target_per_level)
    if raw_existing:
        _write_jsonl_exact(output_path, curated)
        print(f"Curated existing paintings before fill: {len(raw_existing)} -> {len(curated)} kept; dropped {len(dropped_existing)}")
        print("Curated counts:", dict(Counter(r.get("complexity") for r in curated)))
        print("Curated date counts:", Counter(_record_date_range_key(r) for r in curated if _record_date_range_key(r)).most_common(12))

    existing = list(curated)
    seen_keys = {_record_key(r) for r in existing}
    seen_semantic = {_record_global_semantic_key(r) for r in existing}
    seen_gold_sigs = [_gold_sig_from_record(r) for r in existing]
    date_counts = Counter(_record_date_range_key(r) for r in existing if _record_date_range_key(r) is not None)

    counts = Counter(r.get("complexity") for r in existing if r.get("domain") == PAINTINGS_DOMAIN)
    next_idx = defaultdict(lambda: 1)
    for r in existing:
        if r.get("domain") != PAINTINGS_DOMAIN:
            continue
        m = re.search(r"_(l\d)_(\d{4})$", str(r.get("id", "")))
        if m:
            level = m.group(1).upper()
            next_idx[level] = max(next_idx[level], int(m.group(2)) + 1)

    print("Existing counts after v10 curation:", dict(counts))
    print("Building candidate queue before WDQS generation...")
    queue = build_painting_candidate_queue(seed=seed)
    audit_skips = list(dropped_existing)
    total_target = sum(target_per_level.values())
    current_total = sum(min(counts.get(level, 0), target) for level, target in target_per_level.items())
    pbar = tqdm(total=total_target, initial=current_total, desc="paintings total")

    for level, target in target_per_level.items():
        accepted_for_level = int(counts.get(level, 0))
        attempts = 0
        rejected = 0
        reject_reasons = Counter()
        specs = list(queue.get(level, []))
        spec_i = 0
        while accepted_for_level < int(target) and attempts < PAINTINGS_MAX_ATTEMPTS_PER_LEVEL:
            attempts += 1
            if not specs:
                audit_skips.append({"level": level, "reason": "no_candidate_specs"})
                break
            spec = specs[spec_i % len(specs)]
            spec_i += 1
            proposed_key = json.dumps({"domain": PAINTINGS_DOMAIN, "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
            semantic_key = _spec_global_semantic_key(spec)
            dkey = _spec_date_range_key(spec)
            if dkey is not None and date_counts[dkey] >= _date_range_cap(dkey):
                reject_reasons["date_range_cap"] += 1
                continue
            if proposed_key in seen_keys or semantic_key in seen_semantic:
                reject_reasons["duplicate_spec"] += 1
                continue
            try:
                ex = _finalize_painting_spec(spec, next_idx[level], require_complete=require_complete_gold)
            except Exception as e:
                reason = "exception"
                reject_reasons[reason] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "error": str(e)[:500], "constraints": spec.get("constraints")})
                rejected += 1
                pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} last={reason}")
                continue
            if ex is None:
                reason = spec.get("_last_reject_reason") or "gold_rejected_or_incomplete"
                reject_reasons[str(reason).split()[0]] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "local_n": spec.get("local_n"), "date_range": list(dkey) if dkey else None, "constraints": spec.get("constraints")})
                rejected += 1
                if attempts % 10 == 0:
                    pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} reasons={dict(reject_reasons.most_common(2))}")
                continue
            rec = _example_to_record(ex)
            rec = _normalize_record_text(rec)
            if not _schema_is_exact(rec):
                raise AssertionError("Generated record field order does not match BenchmarkExample")
            rec_dkey = _record_date_range_key(rec)
            if rec_dkey is not None and date_counts[rec_dkey] >= _date_range_cap(rec_dkey):
                reason = "date_range_cap_after_wdqs"
                reject_reasons[reason] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": reason, "date_range": list(rec_dkey), "gold": len(ex.gold_answer_qids), "constraints": rec.get("constraints")})
                rejected += 1
                continue
            dup, dup_reason = _new_example_is_duplicate(ex, rec, seen_semantic, seen_gold_sigs)
            if dup:
                reject_reasons[dup_reason] += 1
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": dup_reason, "gold": len(ex.gold_answer_qids), "constraints": rec.get("constraints")})
                rejected += 1
                continue
            key = _record_key(rec)
            if key in seen_keys:
                reject_reasons["duplicate_key"] += 1
                continue
            _append_jsonl(output_path, rec)
            existing.append(rec)
            seen_keys.add(key)
            seen_keys.add(proposed_key)
            seen_semantic.add(_record_global_semantic_key(rec))
            seen_gold_sigs.append(_gold_sig_from_record(rec))
            if rec_dkey is not None:
                date_counts[rec_dkey] += 1
            next_idx[level] += 1
            counts[level] += 1
            accepted_for_level += 1
            pbar.update(1)
            pbar.set_postfix_str(f"{level} attempts={attempts} accepted={accepted_for_level}/{target} rejected={rejected} gold={len(ex.gold_answer_qids)} date={rec_dkey}")
        if accepted_for_level < int(target):
            print(f"[WARN] paintings {level}: accepted {accepted_for_level}/{target}; reject reasons: {dict(reject_reasons.most_common(8))}")
    pbar.close()

    final_rows = _read_jsonl(output_path)
    final_curated, final_dropped = curate_existing_paintings_records(final_rows, target_per_level=target_per_level)
    if len(final_curated) != len(final_rows):
        _write_jsonl_exact(output_path, final_curated)
        audit_skips.extend(final_dropped)
        final_rows = final_curated

    final_date_counts = Counter(_record_date_range_key(r) for r in final_rows if _record_date_range_key(r) is not None)
    audit = {
        "domain": PAINTINGS_DOMAIN,
        "output_path": str(output_path),
        "target_per_level": target_per_level,
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in final_rows if r.get("domain") == PAINTINGS_DOMAIN)),
        "date_counts": {f"{k[0]}_{k[1]}": v for k, v in sorted(final_date_counts.items())},
        "seed_pool": paintings_seed_audit(),
        "candidate_counts": {lvl: len(v) for lvl, v in queue.items()},
        "skipped_count": len(audit_skips),
        "skipped_preview": audit_skips[-1000:],
        "updated_at": _now_iso(),
        "format_fields": BENCHMARK_FIELD_ORDER,
        "quality_patch": PAINTINGS_DATE_DIVERSITY_PATCH,
        "date_windows": PAINTING_DATE_WINDOWS,
    }
    format_problems = validate_jsonl_exact_format(output_path)
    if len(format_problems) > 0:
        raise AssertionError(f"Generated JSONL format mismatch: {format_problems.head(10).to_dict('records')}")
    _write_json(PAINTINGS_AUDIT_PATH, audit)
    print(f"saved incrementally: {output_path}")
    print(f"audit: {PAINTINGS_AUDIT_PATH}")
    print("Final counts:", dict(Counter(r.get("complexity") for r in final_rows)))
    print("Final date counts:", final_date_counts.most_common(20))
    return final_rows

print("✅ Applied paintings v10 date-diversity patch: fine 25/50-year windows + date-range caps")


✅ Recomputed paintings seed date windows (v10). Top windows: [((1400, 1449), 2309), ((1900, 1924), 540), ((1450, 1499), 456), ((1500, 1549), 345), ((1850, 1874), 305), ((1875, 1899), 269), ((1300, 1349), 223), ((1000, 1199), 217)]
✅ Applied paintings v10 date-diversity patch: fine 25/50-year windows + date-range caps


In [10]:
# ============================================================
# v13 FINAL from-scratch quality generator for paintings
# - DOES NOT resume or curate old paintings.jsonl; it always builds a fresh final JSONL.
# - Exact target: L1=15, L2=22, L3=25, L4=25, L5=25.
# - Natural generation order L1 -> L5.
# - Date diversity with narrow 25/50-year windows; no legacy 1400-1599 bucket.
# - Large direct candidate registry for L4/L5; no seed-bridge templates.
# - Multi-phase diversity: strict first, then controlled relaxation so the final file is filled.
# - Complete WDQS golds only, exact 21-field benchmark schema.
# ============================================================

PAINTINGS_FINAL_PATCH = "v13_from_scratch_quality_target_15_22_25_25_25"

PAINTINGS_TARGET_PER_LEVEL = {
    "L1": 15,
    "L2": 22,
    "L3": 25,
    "L4": 25,
    "L5": 25,
}

PAINTINGS_REQUESTED_COUNT = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 3,
}

# Complete gold caps. The query is run with LIMIT cap+1 and is rejected if the
# sentinel row appears, so saved gold lists are complete w.r.t. the SPARQL.
PAINTINGS_MAX_GOLD_BY_LEVEL = {
    "L1": 250,
    "L2": 190,
    "L3": 140,
    "L4": 115,
    "L5": 95,
}
PAINTINGS_MIN_GOLD_HEADROOM = 2
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 12000
PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 45
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 2
PAINTINGS_FAST_QID_FIRST_GOLD = False

# Narrow/diverse windows. New hard-level records should use these, not 1400-1599.
PAINTING_DATE_WINDOWS = [
    (1200, 1299),
    (1250, 1299),
    (1300, 1349), (1350, 1399),
    (1400, 1449), (1450, 1499), (1500, 1549), (1550, 1599),
    (1600, 1649), (1650, 1699),
    (1700, 1749), (1750, 1799),
    (1800, 1849),
    (1850, 1874), (1875, 1899),
    (1900, 1924), (1925, 1949),
    (1950, 1974), (1975, 1999), (2000, 2026),
]

# Keep these caps sane but not so strict that L4/L5 cannot be filled from scratch.
PAINTINGS_LEVEL_DATE_CAP_BASE = {"L1": 999, "L2": 999, "L3": 7, "L4": 7, "L5": 6}
PAINTINGS_GLOBAL_DATE_CAP_BASE = 10
PAINTINGS_COMBO_CAP_BASE = {"L1": 5, "L2": 4, "L3": 4, "L4": 4, "L5": 4}

# Old broad/duplicate families must not be generated in the final file.
PAINTINGS_DROP_TEMPLATE_ID_SUBSTRINGS = ("_seed_",)
PAINTINGS_FORBIDDEN_DATE_RANGES = {(1400, 1599)}

# Broad candidate registry. L4/L5 are direct resolved multihop constraints, no seed bridges.
PAINTING_DIRECT_TEMPLATES = [
    # L1: simple but compact enough to have full golds.
    {"level": "L1", "template_id": "paintings_l1_by_collection", "family": "single_collection", "fields": ["collection"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_creator", "family": "single_creator", "fields": ["creator"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_material", "family": "single_material", "fields": ["material"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_movement", "family": "single_movement", "fields": ["movement"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_genre", "family": "single_genre", "fields": ["genre"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_depicts", "family": "single_depicts", "fields": ["depicts"], "min_n": 5, "max_n": 240},

    # L2.
    {"level": "L2", "template_id": "paintings_l2_material_collection_country", "family": "material_collection_country", "fields": ["material", "collection_country"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_genre_collection_country", "family": "genre_collection_country", "fields": ["genre", "collection_country"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_movement_collection_country", "family": "movement_collection_country", "fields": ["movement", "collection_country"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_depicts_collection_country", "family": "depicts_collection_country", "fields": ["depicts", "collection_country"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_creator_citizenship_collection_country", "family": "creator_country_collection_country", "fields": ["creator_country", "collection_country"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_material_creator_citizenship", "family": "material_creator_country", "fields": ["material", "creator_country"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_collection_material", "family": "collection_material", "fields": ["collection", "material"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_collection_genre", "family": "collection_genre", "fields": ["collection", "genre"], "min_n": 5, "max_n": 185},
    {"level": "L2", "template_id": "paintings_l2_collection_city_material", "family": "collection_city_material", "fields": ["collection_city", "material"], "min_n": 5, "max_n": 185},

    # L3: 3 constraints, often including a narrow date window.
    {"level": "L3", "template_id": "paintings_l3_material_collection_country_period", "family": "material_collection_country_period", "fields": ["material", "collection_country", "year_bucket"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_genre_collection_country_period", "family": "genre_collection_country_period", "fields": ["genre", "collection_country", "year_bucket"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_movement_collection_country_period", "family": "movement_collection_country_period", "fields": ["movement", "collection_country", "year_bucket"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_depicts_collection_country_period", "family": "depicts_collection_country_period", "fields": ["depicts", "collection_country", "year_bucket"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_creator_citizenship_collection_country_period", "family": "creator_country_collection_country_period", "fields": ["creator_country", "collection_country", "year_bucket"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_material_creator_citizenship_collection_country", "family": "material_creator_country_collection_country", "fields": ["material", "creator_country", "collection_country"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_genre_material_collection_country", "family": "genre_material_collection_country", "fields": ["genre", "material", "collection_country"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_depicts_material_collection_country", "family": "depicts_material_collection_country", "fields": ["depicts", "material", "collection_country"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_collection_material_period", "family": "collection_material_period", "fields": ["collection", "material", "year_bucket"], "min_n": 4, "max_n": 140},
    {"level": "L3", "template_id": "paintings_l3_collection_genre_period", "family": "collection_genre_period", "fields": ["collection", "genre", "year_bucket"], "min_n": 4, "max_n": 140},

    # L4: 4 resolved constraints; most have collection/creator paths + narrow period.
    {"level": "L4", "template_id": "paintings_l4_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_material_creator_birth_country_collection_country_period", "family": "material_creator_birth_country_collection_country_period", "fields": ["material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_citizenship_collection_country_period", "family": "genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_genre_material_collection_country_period", "family": "genre_material_collection_country_period", "fields": ["genre", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_movement_material_collection_country_period", "family": "movement_material_collection_country_period", "fields": ["movement", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_depicts_material_collection_country_period", "family": "depicts_material_collection_country_period", "fields": ["depicts", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_depicts_creator_citizenship_collection_country_period", "family": "depicts_creator_country_collection_country_period", "fields": ["depicts", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_movement_creator_citizenship_collection_country_period", "family": "movement_creator_country_collection_country_period", "fields": ["movement", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_collection_material_creator_citizenship_period", "family": "collection_material_creator_country_period", "fields": ["collection", "material", "creator_country", "year_bucket"], "min_n": 3, "max_n": 115},
    {"level": "L4", "template_id": "paintings_l4_collection_city_material_creator_citizenship_period", "family": "collection_city_material_creator_country_period", "fields": ["collection_city", "material", "creator_country", "year_bucket"], "min_n": 3, "max_n": 115},

    # L5: hard direct templates. First set has 5 explicit constraints; fallback set has 4 very selective constraints
    # with two multihop paths + narrow period and is used only if needed by natural fill phases.
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_citizenship_collection_country_period", "family": "genre_material_creator_country_collection_country_period", "fields": ["genre", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_birth_country_collection_country_period", "family": "genre_material_creator_birth_country_collection_country_period", "fields": ["genre", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_citizenship_collection_country_period", "family": "movement_material_creator_country_collection_country_period", "fields": ["movement", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_birth_country_collection_country_period", "family": "movement_material_creator_birth_country_collection_country_period", "fields": ["movement", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_depicts_genre_material_collection_country_period", "family": "depicts_genre_material_collection_country_period", "fields": ["depicts", "genre", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_depicts_material_creator_citizenship_collection_country_period", "family": "depicts_material_creator_country_collection_country_period", "fields": ["depicts", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_depicts_material_creator_birth_country_collection_country_period", "family": "depicts_material_creator_birth_country_collection_country_period", "fields": ["depicts", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_movement_genre_material_collection_country_period", "family": "movement_genre_material_collection_country_period", "fields": ["movement", "genre", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_collection_genre_material_creator_citizenship_period", "family": "collection_genre_material_creator_country_period", "fields": ["collection", "genre", "material", "creator_country", "year_bucket"], "min_n": 3, "max_n": 95},
    # Fallback-hard but still complex.
    {"level": "L5", "template_id": "paintings_l5_hard_material_creator_citizenship_collection_country_period", "family": "hard_material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_hard_material_creator_birth_country_collection_country_period", "family": "hard_material_creator_birth_country_collection_country_period", "fields": ["material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
    {"level": "L5", "template_id": "paintings_l5_hard_genre_creator_citizenship_collection_country_period", "family": "hard_genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 95},
]


def _v13_clean_text(x: Any) -> str:
    return re.sub(r"[\u200e\u200f\u202a-\u202e\ufeff]", "", "" if x is None else str(x)).strip()


def _v13_year_window(y: Any) -> Optional[Tuple[int, int]]:
    yi = _safe_int(y)
    if yi is None:
        return None
    for a, b in PAINTING_DATE_WINDOWS:
        if a <= yi <= b:
            return (a, b)
    return None


def _v13_refresh_seed_year_windows() -> None:
    """Replace old broad cached year buckets with final narrow windows, in-memory."""
    global paintings_seed_df
    if paintings_seed_df is None or len(paintings_seed_df) == 0 or "year" not in paintings_seed_df.columns:
        return
    df = paintings_seed_df.copy()
    starts, ends, keys = [], [], []
    for y in df["year"].tolist():
        win = _v13_year_window(y)
        if win:
            a, b = win
            starts.append(a); ends.append(b); keys.append(f"{a}_{b}")
        else:
            starts.append(None); ends.append(None); keys.append(None)
    df["year_bucket"] = keys
    df["year_from"] = starts
    df["year_to"] = ends
    df["year_ru"] = [f"{a}–{b}" if a is not None else None for a, b in zip(starts, ends)]
    df["year_en"] = [f"{a}-{b}" if a is not None else None for a, b in zip(starts, ends)]
    # Clean labels too; invisible marks in labels polluted older generations.
    for c in list(df.columns):
        if c.endswith("_en") or c.endswith("_ru") or c == "item_en" or c == "item_ru":
            df[c] = df[c].map(_v13_clean_text)
    paintings_seed_df = df


def _v13_date_key_from_constraints(c: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    y1 = _safe_int(c.get("date_from"))
    y2 = _safe_int(c.get("date_to"))
    if y1 is None or y2 is None:
        return None
    return (int(y1), int(y2))


def _v13_date_key_from_spec(spec: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    return _v13_date_key_from_constraints(spec.get("constraints") or {})


def _v13_date_key_from_record(rec: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    return _v13_date_key_from_constraints(rec.get("constraints") or {})


def _v13_span(dkey: Optional[Tuple[int, int]]) -> int:
    if not dkey:
        return 0
    return int(dkey[1]) - int(dkey[0])


def _v13_global_date_cap(dkey: Optional[Tuple[int, int]], phase: int) -> int:
    if dkey is None:
        return 9999
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        return 0
    span = _v13_span(dkey)
    base = PAINTINGS_GLOBAL_DATE_CAP_BASE
    if span > 80:
        base = 4
    elif span <= 50:
        base = PAINTINGS_GLOBAL_DATE_CAP_BASE
    return base + phase * 3


def _v13_level_date_cap(level: str, dkey: Optional[Tuple[int, int]], phase: int) -> int:
    if dkey is None or str(level) in {"L1", "L2"}:
        return 9999
    return int(PAINTINGS_LEVEL_DATE_CAP_BASE.get(str(level), 6) + phase * 2)


def _v13_norm(x: Any) -> str:
    return _v13_clean_text(x).casefold()


def _v13_combo_keys(c: Dict[str, Any]) -> List[Tuple[str, ...]]:
    d = _v13_date_key_from_constraints(c)
    date_s = f"{d[0]}_{d[1]}" if d else "no_date"
    keys = []
    # These cap the old Italy/fresco/religious-art cluster without forbidding it entirely.
    for fields in [
        ("collection_country", "material", "date"),
        ("collection_country", "genre", "date"),
        ("collection_country", "art_movement", "date"),
        ("creator_citizenship", "collection_country", "date"),
        ("creator_birth_country", "collection_country", "date"),
        ("depicts", "collection_country", "date"),
        ("material", "creator_citizenship", "collection_country", "date"),
    ]:
        vals = []
        ok = True
        for f in fields:
            if f == "date":
                vals.append(date_s)
            else:
                v = c.get(f)
                if not v:
                    ok = False
                    break
                vals.append(_v13_norm(v))
        if ok:
            keys.append(tuple([fields[0]] + vals))
    return keys


def _v13_combo_cap(level: str, phase: int) -> int:
    return int(PAINTINGS_COMBO_CAP_BASE.get(str(level), 4) + phase * 2)


def _v13_gold_sig(rec: Dict[str, Any]) -> Tuple[str, ...]:
    return tuple(sorted(str(q) for q in rec.get("gold_answer_qids") or [] if q))


def _v13_gold_is_duplicate_enough(rec: Dict[str, Any], old_sigs: List[Tuple[str, ...]], phase: int) -> Tuple[bool, str]:
    sig = _v13_gold_sig(rec)
    if not sig:
        return False, ""
    ss = set(sig)
    if phase <= 0:
        j_thr, c_thr = 0.80, 0.90
    elif phase == 1:
        j_thr, c_thr = 0.88, 0.96
    elif phase == 2:
        j_thr, c_thr = 0.94, 0.99
    else:
        j_thr, c_thr = 1.01, 1.01  # final phase rejects exact gold only
    for old in old_sigs:
        if sig == old:
            return True, "duplicate_gold_set"
        so = set(old)
        if not so:
            continue
        inter = len(ss & so)
        union = len(ss | so)
        j = inter / union if union else 0
        containment = inter / min(len(ss), len(so)) if min(len(ss), len(so)) else 0
        if j >= j_thr:
            return True, f"near_duplicate_gold_jaccard_{j:.2f}"
        if min(len(ss), len(so)) >= 8 and containment >= c_thr:
            return True, f"near_duplicate_gold_containment_{containment:.2f}"
    return False, ""


def _v13_bad_cluster_penalty(spec_or_rec: Dict[str, Any]) -> int:
    c = spec_or_rec.get("constraints") or {}
    dkey = _v13_date_key_from_constraints(c)
    country = _v13_norm(c.get("collection_country") or c.get("collection_location"))
    material = _v13_norm(c.get("material"))
    genre = _v13_norm(c.get("genre"))
    movement = _v13_norm(c.get("art_movement"))
    penalty = 0
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        penalty += 9999
    if country in {"italy", "vatican city"} and material in {"fresco", "tempera", "plaster"} and dkey and 1300 <= dkey[0] <= 1550:
        penalty += 180
    if country in {"italy", "vatican city"} and genre == "religious art" and dkey and 1300 <= dkey[0] <= 1550:
        penalty += 150
    if country in {"italy", "vatican city"} and movement in {"early renaissance", "high renaissance", "renaissance"}:
        penalty += 120
    return penalty


def _v13_template_priority(template_id: str) -> int:
    # Prefer templates that have date+material+collection/creator paths but are not overly repetitive.
    t = str(template_id)
    if "l1_by_collection" in t or "l1_by_creator" in t:
        return 0
    if "material_collection_country" in t:
        return 1
    if "creator_citizenship_collection_country" in t:
        return 2
    if "depicts" in t:
        return 3
    if "movement" in t:
        return 4
    if "genre" in t:
        return 5
    return 6


def _v13_spec_sort_key(spec: Dict[str, Any]) -> Tuple[int, int, int, int, str]:
    level = str(spec.get("complexity"))
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    ideal = {"L1": 45, "L2": 34, "L3": 24, "L4": 18, "L5": 14}.get(level, 24)
    dkey = _v13_date_key_from_spec(spec)
    span_penalty = 0
    if dkey and _v13_span(dkey) > 80 and level in {"L3", "L4", "L5"}:
        span_penalty = 50
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        span_penalty += 9999
    return (_v13_bad_cluster_penalty(spec) + span_penalty, _v13_template_priority(spec.get("template_id", "")), abs(local_n - ideal), local_n, str(spec.get("template_id")))


def _v13_interleave_specs(specs: List[Dict[str, Any]], rng: random.Random) -> List[Dict[str, Any]]:
    buckets = defaultdict(list)
    for s in specs:
        key = (_v13_date_key_from_spec(s), str(s.get("template_id")), _v13_norm((s.get("constraints") or {}).get("collection_country")))
        buckets[key].append(s)
    for key in buckets:
        buckets[key].sort(key=_v13_spec_sort_key)
    keys = sorted(buckets.keys(), key=lambda k: (0 if k[0] is not None else 1, str(k[0]), k[1], k[2]))
    out = []
    made_progress = True
    while made_progress:
        made_progress = False
        for key in keys:
            if buckets[key]:
                out.append(buckets[key].pop(0))
                made_progress = True
    return out


def build_painting_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    _v13_refresh_seed_year_windows()
    specs: List[Dict[str, Any]] = []
    for tpl in PAINTING_DIRECT_TEMPLATES:
        level = tpl["level"]
        k = PAINTINGS_REQUESTED_COUNT[level]
        max_n = int(tpl.get("max_n", PAINTINGS_MAX_GOLD_BY_LEVEL[level]))
        # Local seed counts are only approximate; let WDQS decide exact golds.
        local_max = int(max(max_n * (2.4 if level in {"L4", "L5"} else 1.6), max_n + 25))
        min_n = max(2, min(int(tpl.get("min_n", k)), k))
        grp = _painting_group_candidates(
            paintings_seed_df,
            tpl["fields"],
            min_n=min_n,
            max_n=local_max,
            max_rows=8000 if level in {"L4", "L5"} else 4500,
        )
        if len(grp) == 0:
            continue
        # Prefer compact groups but sample within the sorted frame for diversity.
        grp = grp.sort_values("n", ascending=True).head(8000)
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_painting_direct_spec(tpl, row)
            if not spec:
                continue
            if any(part in str(spec.get("template_id", "")) for part in PAINTINGS_DROP_TEMPLATE_ID_SUBSTRINGS):
                continue
            dkey = _v13_date_key_from_spec(spec)
            if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
                continue
            # No broad dates for hard levels.
            if level in {"L3", "L4", "L5"} and dkey and _v13_span(dkey) > 80:
                continue
            local_n = _safe_int(spec.get("local_n"), 0) or 0
            if level in {"L1", "L2", "L3"} and local_n < k + PAINTINGS_MIN_GOLD_HEADROOM:
                continue
            if level in {"L4", "L5"} and local_n < k:  # WDQS may have more than seed pool; don't over-filter hard levels.
                continue
            specs.append(spec)
    return specs


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    print("Building paintings candidate queue (v13 from-scratch quality fill)...")
    all_specs = build_painting_direct_candidates(rng)
    print(f"  direct candidate specs: {len(all_specs)}")
    print("  bridge candidate specs: 0 (disabled: seed variants caused duplicate tasks)")
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen = set()
    for spec in all_specs:
        key = _spec_global_semantic_key(spec)
        if key in seen:
            continue
        seen.add(key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        by_level[level] = _v13_interleave_specs(by_level[level], rng)
        preview = [(s.get("template_id"), s.get("local_n"), _v13_date_key_from_spec(s), (s.get("constraints") or {}).get("collection_country")) for s in by_level[level][:12]]
        print(f"  {level} top candidates:", preview)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()})
    return by_level


def _v13_record_passes(
    rec: Dict[str, Any],
    counts: Counter,
    date_counts: Counter,
    level_date_counts: Counter,
    combo_counts: Counter,
    seen_record_keys: set,
    seen_semantic: set,
    seen_gold_sigs: List[Tuple[str, ...]],
    phase: int,
) -> Tuple[bool, str]:
    level = str(rec.get("complexity") or "")
    if counts[level] >= int(PAINTINGS_TARGET_PER_LEVEL[level]):
        return False, "level_full"
    if not _schema_is_exact(rec):
        return False, "schema_mismatch"
    if any(part in str(rec.get("template_id", "")) for part in PAINTINGS_DROP_TEMPLATE_ID_SUBSTRINGS):
        return False, "seed_template_disabled"
    g = len(rec.get("gold_answer_qids") or [])
    req = int(rec.get("requested_count") or PAINTINGS_REQUESTED_COUNT.get(level, 3))
    if g < req + PAINTINGS_MIN_GOLD_HEADROOM:
        return False, "too_few_gold"
    if g > PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 9999):
        return False, "too_many_gold"
    rkey = _record_key(rec)
    if rkey in seen_record_keys:
        return False, "duplicate_record_key"
    skey = _record_global_semantic_key(rec)
    if skey in seen_semantic:
        return False, "duplicate_resolved_constraints"
    near, why = _v13_gold_is_duplicate_enough(rec, seen_gold_sigs, phase=phase)
    if near:
        return False, why
    dkey = _v13_date_key_from_record(rec)
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        return False, "legacy_broad_date_disabled"
    if dkey is not None:
        if level in {"L3", "L4", "L5"} and _v13_span(dkey) > 80:
            return False, "broad_hard_date_disabled"
        if date_counts[dkey] >= _v13_global_date_cap(dkey, phase):
            return False, "date_range_cap"
        if level_date_counts[(level, dkey)] >= _v13_level_date_cap(level, dkey, phase):
            return False, "level_date_range_cap"
    for ck in _v13_combo_keys(rec.get("constraints") or {}):
        if combo_counts[(level, ck)] >= _v13_combo_cap(level, phase):
            return False, "combo_cap"
    return True, ""


def _v13_spec_prefilter(
    spec: Dict[str, Any],
    counts: Counter,
    date_counts: Counter,
    level_date_counts: Counter,
    combo_counts: Counter,
    seen_semantic: set,
    phase: int,
) -> Tuple[bool, str]:
    level = str(spec.get("complexity") or "")
    if counts[level] >= int(PAINTINGS_TARGET_PER_LEVEL[level]):
        return False, "level_full"
    skey = _spec_global_semantic_key(spec)
    if skey in seen_semantic:
        return False, "duplicate_spec_semantic"
    dkey = _v13_date_key_from_spec(spec)
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        return False, "legacy_broad_date_disabled"
    if dkey is not None:
        if level in {"L3", "L4", "L5"} and _v13_span(dkey) > 80:
            return False, "broad_hard_date_disabled"
        # Phase 3 is final fill: keep exact duplicate/gold checks but relax quotas.
        if phase <= 2:
            if date_counts[dkey] >= _v13_global_date_cap(dkey, phase):
                return False, "date_range_cap"
            if level_date_counts[(level, dkey)] >= _v13_level_date_cap(level, dkey, phase):
                return False, "level_date_range_cap"
    if phase <= 2:
        for ck in _v13_combo_keys(spec.get("constraints") or {}):
            if combo_counts[(level, ck)] >= _v13_combo_cap(level, phase):
                return False, "combo_cap"
    return True, ""


def _v13_update_state(rec: Dict[str, Any], counts: Counter, date_counts: Counter, level_date_counts: Counter, combo_counts: Counter, seen_keys: set, seen_semantic: set, seen_gold_sigs: List[Tuple[str, ...]]) -> None:
    level = str(rec.get("complexity"))
    counts[level] += 1
    dkey = _v13_date_key_from_record(rec)
    if dkey:
        date_counts[dkey] += 1
        level_date_counts[(level, dkey)] += 1
    for ck in _v13_combo_keys(rec.get("constraints") or {}):
        combo_counts[(level, ck)] += 1
    seen_keys.add(_record_key(rec))
    seen_semantic.add(_record_global_semantic_key(rec))
    seen_gold_sigs.append(_v13_gold_sig(rec))


def generate_paintings_dataset(
    output_path: Path = PAINTINGS_OUTPUT_PATH,
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
    seed: int = 20260528,
    require_complete_gold: bool = True,
    reset_output: bool = True,
) -> List[Dict[str, Any]]:
    """Generate the final paintings JSONL from scratch. Existing files are archived and ignored."""
    # From-scratch means from-scratch: never recover or curate prior partial JSONL.
    if output_path.exists():
        _archive_existing_jsonl(output_path, suffix="pre_v13_from_scratch")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

    _v13_refresh_seed_year_windows()
    queue = build_painting_candidate_queue(seed=seed)

    counts = Counter()
    date_counts = Counter()
    level_date_counts = Counter()
    combo_counts = Counter()
    seen_record_keys = set()
    seen_semantic = set()
    seen_gold_sigs: List[Tuple[str, ...]] = []
    next_idx = defaultdict(lambda: 1)
    audit_skips = []
    reject_reasons_by_level = defaultdict(Counter)
    dead_specs = set()

    total_target = sum(int(v) for v in target_per_level.values())
    pbar = tqdm(total=total_target, initial=0, desc="paintings total")

    fill_order = ["L1", "L2", "L3", "L4", "L5"]
    phase_names = {0: "strict_diverse", 1: "relax_overlap", 2: "relax_combo_date", 3: "final_fill_exact_only"}
    phases = [0, 1, 2, 3]

    for phase in phases:
        phase_gain = 0
        print(f"\n=== Paintings v13 from-scratch phase {phase}: {phase_names[phase]} ===")
        for level in fill_order:
            target = int(target_per_level[level])
            if counts[level] >= target:
                continue
            attempts = 0
            accepted_before = counts[level]
            specs = list(queue.get(level, []))
            for spec in specs:
                if counts[level] >= target:
                    break
                attempts += 1
                sdead = _spec_global_semantic_key(spec)
                if sdead in dead_specs:
                    continue
                ok, reason = _v13_spec_prefilter(spec, counts, date_counts, level_date_counts, combo_counts, seen_semantic, phase=phase)
                if not ok:
                    reject_reasons_by_level[level][reason] += 1
                    continue
                try:
                    ex = _finalize_painting_spec(spec, next_idx[level], require_complete=require_complete_gold)
                except Exception as e:
                    reject_reasons_by_level[level]["exception"] += 1
                    audit_skips.append({"level": level, "phase": phase, "template_id": spec.get("template_id"), "reason": "exception", "error": str(e)[:500], "constraints": spec.get("constraints")})
                    if attempts % 20 == 0:
                        pbar.set_postfix_str(f"{level} phase={phase} attempts={attempts} accepted={counts[level]}/{target} last=exception")
                    continue
                if ex is None:
                    reason = str(spec.get("_last_reject_reason") or "gold_rejected_or_incomplete")
                    rhead = reason.split()[0].split(":")[0]
                    reject_reasons_by_level[level][rhead] += 1
                    # Gold failures are deterministic for the exact SPARQL; don't retry in later phases.
                    if rhead in {"too_few_gold", "too_many_gold", "truncated_or_over_cap", "gold_rejected_or_incomplete"}:
                        dead_specs.add(sdead)
                    audit_skips.append({"level": level, "phase": phase, "template_id": spec.get("template_id"), "reason": reason, "local_n": spec.get("local_n"), "date_range": list(_v13_date_key_from_spec(spec)) if _v13_date_key_from_spec(spec) else None, "constraints": spec.get("constraints")})
                    if attempts % 20 == 0:
                        pbar.set_postfix_str(f"{level} phase={phase} attempts={attempts} accepted={counts[level]}/{target} reasons={dict(reject_reasons_by_level[level].most_common(2))}")
                    continue
                rec = _normalize_record_text(_example_to_record(ex))
                ok, reason = _v13_record_passes(
                    rec, counts, date_counts, level_date_counts, combo_counts,
                    seen_record_keys, seen_semantic, seen_gold_sigs, phase=phase,
                )
                if not ok:
                    reject_reasons_by_level[level][reason] += 1
                    audit_skips.append({"level": level, "phase": phase, "template_id": spec.get("template_id"), "reason": reason, "gold": len(ex.gold_answer_qids), "date_range": list(_v13_date_key_from_record(rec)) if _v13_date_key_from_record(rec) else None, "constraints": rec.get("constraints")})
                    continue
                # Rewrite id with the next continuous per-level index, because _finalize used the same idx.
                rec["id"] = f"paintings_{level.lower()}_{next_idx[level]:04d}"
                _append_jsonl(output_path, rec)
                _v13_update_state(rec, counts, date_counts, level_date_counts, combo_counts, seen_record_keys, seen_semantic, seen_gold_sigs)
                next_idx[level] += 1
                phase_gain += 1
                pbar.update(1)
                pbar.set_postfix_str(f"{level} phase={phase} accepted={counts[level]}/{target} gold={len(ex.gold_answer_qids)} date={_v13_date_key_from_record(rec)}")
            if counts[level] < target:
                gained = counts[level] - accepted_before
                print(f"[WARN] phase {phase} paintings {level}: accepted {counts[level]}/{target}; gained {gained}; top reject reasons: {dict(reject_reasons_by_level[level].most_common(10))}")
        if all(counts.get(level, 0) >= int(target_per_level[level]) for level in target_per_level):
            break
        if phase_gain == 0:
            print(f"[WARN] phase {phase} made no progress; moving to next relaxation phase.")
    pbar.close()

    final_rows = _read_jsonl(output_path)
    final_counts = Counter(r.get("complexity") for r in final_rows if r.get("domain") == PAINTINGS_DOMAIN)
    final_date_counts = Counter(_v13_date_key_from_record(r) for r in final_rows if _v13_date_key_from_record(r) is not None)

    # Format/schema validation. Do not run aggressive curation after generation; it can drop below target.
    format_problems = validate_jsonl_exact_format(output_path)
    if len(format_problems) > 0:
        raise AssertionError(f"Generated JSONL format mismatch: {format_problems.head(10).to_dict('records')}")

    audit = {
        "domain": PAINTINGS_DOMAIN,
        "output_path": str(output_path),
        "target_per_level": dict(target_per_level),
        "counts_by_complexity": dict(final_counts),
        "date_counts": {f"{k[0]}_{k[1]}": v for k, v in sorted(final_date_counts.items())},
        "seed_pool": paintings_seed_audit(),
        "candidate_counts": {lvl: len(v) for lvl, v in queue.items()},
        "reject_reasons_by_level": {lvl: dict(cnt.most_common(30)) for lvl, cnt in reject_reasons_by_level.items()},
        "skipped_count": len(audit_skips),
        "skipped_preview": audit_skips[-2000:],
        "updated_at": _now_iso(),
        "format_fields": BENCHMARK_FIELD_ORDER,
        "quality_patch": PAINTINGS_FINAL_PATCH,
        "date_windows": PAINTING_DATE_WINDOWS,
        "from_scratch": True,
        "final_target_met": {lvl: final_counts.get(lvl, 0) >= int(target_per_level[lvl]) for lvl in target_per_level},
    }
    _write_json(PAINTINGS_AUDIT_PATH, audit)
    print(f"saved incrementally: {output_path}")
    print(f"audit: {PAINTINGS_AUDIT_PATH}")
    print("Final counts:", dict(final_counts))
    print("Final date counts:", final_date_counts.most_common(30))
    if any(final_counts.get(lvl, 0) < int(target_per_level[lvl]) for lvl in target_per_level):
        print("[WARN] Final target not fully met:", {lvl: (final_counts.get(lvl, 0), int(target_per_level[lvl])) for lvl in target_per_level})
    else:
        print("✅ Final paintings target met exactly/from scratch.")
    return final_rows

print("✅ Applied paintings v13 FROM-SCRATCH quality patch: target 15/22/25/25/25, natural L1→L5 order, date diversity, complete golds")


✅ Applied paintings v13 FROM-SCRATCH quality patch: target 15/22/25/25/25, natural L1→L5 order, date diversity, complete golds


In [11]:
# ============================================================
# v14 FINAL from-scratch QUALITY patch for paintings
# Fixes observed v13 issues:
# - generates from scratch, no resume/old JSON curation;
# - exact target L1=15, L2=22, L3=25, L4=25, L5=25;
# - natural generation order L1 -> L2 -> L3 -> L4 -> L5;
# - much stronger date/country/material/genre diversity;
# - expanded direct L4/L5 registry so hard levels fill without seed-duplicate templates;
# - no broad legacy 1400-1599 period for new records;
# - anti-overlap/near-duplicate gold-set checks retained in all phases;
# - RU query labels improved with deterministic overrides for common painting terms.
# ============================================================

from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Sequence, Tuple
import random, re, json, shutil, time

PAINTINGS_FINAL_PATCH = "v14_from_scratch_balanced_quality"

PAINTINGS_TARGET_PER_LEVEL = {
    "L1": 15,
    "L2": 22,
    "L3": 25,
    "L4": 25,
    "L5": 25,
}
PAINTINGS_REQUESTED_COUNT = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}

PAINTINGS_MAX_GOLD_BY_LEVEL = {"L1": 250, "L2": 180, "L3": 125, "L4": 100, "L5": 90}
PAINTINGS_MIN_GOLD_HEADROOM = 2
PAINTINGS_MAX_ATTEMPTS_PER_LEVEL = 20000
PAINTINGS_GENERATION_WDQS_TIMEOUT_SECONDS = 40
PAINTINGS_GENERATION_WDQS_MAX_RETRIES = 2
PAINTINGS_FAST_QID_FIRST_GOLD = False

# Narrow windows only. The old 1400-1599 bucket is explicitly disabled.
PAINTING_DATE_WINDOWS = [
    (1200, 1249), (1250, 1299),
    (1300, 1349), (1350, 1399),
    (1400, 1449), (1450, 1499), (1500, 1549), (1550, 1599),
    (1600, 1649), (1650, 1699),
    (1700, 1749), (1750, 1799),
    (1800, 1849), (1850, 1874), (1875, 1899),
    (1900, 1924), (1925, 1949),
    (1950, 1974), (1975, 1999), (2000, 2026),
]
PAINTINGS_FORBIDDEN_DATE_RANGES = {(1400, 1599)}
PAINTINGS_DROP_TEMPLATE_ID_SUBSTRINGS = ("_seed_",)

# Deterministic RU overrides. Constraints stay EN-only; this only improves query_text_ru.
PAINTING_RU_OVERRIDES = {
    # materials
    "fresco": "фреска", "tempera": "темпера", "plaster": "штукатурка",
    "oil paint": "масляная краска", "watercolor paint": "акварель", "gouache paint": "гуашь",
    "paper": "бумага", "canvas": "холст", "wood": "дерево", "vellum": "пергамент",
    "ink": "тушь", "pigment": "пигмент", "gold": "золото", "silver": "серебро",
    # genres / movements / depicts
    "religious art": "религиозное искусство", "portrait": "портрет", "landscape art": "пейзаж",
    "still life": "натюрморт", "allegory": "аллегория", "mythological painting": "мифологическая живопись",
    "genre art": "жанровая живопись", "history painting": "историческая живопись",
    "Impressionism": "импрессионизм", "Post-Impressionism": "постимпрессионизм",
    "Baroque": "барокко", "Rococo": "рококо", "Romanticism": "романтизм",
    "Realism": "реализм", "Expressionism": "экспрессионизм", "Cubism": "кубизм",
    "Early Renaissance": "Раннее Возрождение", "High Renaissance": "Высокое Возрождение",
    "Italian Renaissance": "итальянское Возрождение", "Renaissance": "Возрождение",
    "Mary": "Дева Мария", "Jesus Christ": "Иисус Христос", "woman": "женщина", "man": "мужчина",
    "angel": "ангел", "horse": "лошадь", "dog": "собака", "child": "ребёнок",
    # countries / historical citizenships that appear often in Wikidata painting data
    "Italy": "Италия", "Vatican City": "Ватикан", "France": "Франция", "United States": "США",
    "United Kingdom": "Великобритания", "Netherlands": "Нидерланды", "Spain": "Испания",
    "Germany": "Германия", "Belgium": "Бельгия", "Austria": "Австрия", "Sweden": "Швеция",
    "Denmark": "Дания", "Norway": "Норвегия", "Russia": "Россия", "Japan": "Япония",
    "Republic of Florence": "Флорентийская республика", "Kingdom of the Netherlands": "Королевство Нидерландов",
    "Mughal Empire": "Империя Великих Моголов", "Safavid Iran": "Сефевидский Иран",
}

# Hard quality caps by phase. They are soft-relaxed across phases, but never disappear completely
# except in exact-target emergency mode; exact/near duplicate checks are always enforced.
V14_PHASES = [
    {"name": "strict_balanced", "j": 0.72, "contain": 0.86, "date_global": 7,  "date_level": 3, "combo": 2, "value_relax": 0},
    {"name": "balanced",        "j": 0.80, "contain": 0.91, "date_global": 8,  "date_level": 4, "combo": 3, "value_relax": 1},
    {"name": "relaxed",         "j": 0.88, "contain": 0.96, "date_global": 10, "date_level": 5, "combo": 4, "value_relax": 2},
    {"name": "fill",            "j": 0.93, "contain": 0.985,"date_global": 12, "date_level": 6, "combo": 5, "value_relax": 3},
    {"name": "target_guarantee", "j": 0.96, "contain": 0.995,"date_global": 16, "date_level": 8, "combo": 7, "value_relax": 5},
]

V14_COUNTRY_TOTAL_CAP = {
    "italy": 22, "vatican city": 6, "united states": 12, "united kingdom": 10,
    "france": 12, "netherlands": 10, "spain": 8, "germany": 8,
}
V14_COUNTRY_LEVEL_CAP = {"L4": {"italy": 8, "vatican city": 3}, "L5": {"italy": 8, "vatican city": 3}}
V14_MATERIAL_TOTAL_CAP = {"fresco": 14, "tempera": 8, "plaster": 5}
V14_MATERIAL_LEVEL_CAP = {"L4": {"fresco": 5}, "L5": {"fresco": 5, "tempera": 4}}
V14_GENRE_TOTAL_CAP = {"religious art": 13}
V14_GENRE_LEVEL_CAP = {"L4": {"religious art": 6}, "L5": {"religious art": 6}}
V14_CLUSTER_TOTAL_CAP = 7
V14_CLUSTER_LEVEL_CAP = {"L4": 3, "L5": 3}


def _v14_clean_text(x: Any) -> str:
    return re.sub(r"[\u200e\u200f\u202a-\u202e\ufeff]", "", "" if x is None else str(x)).strip()


def _v14_norm(x: Any) -> str:
    return _v14_clean_text(x).casefold()


def _v14_ru_label(field: str, en: Any, ru: Any = None) -> str:
    en_s = _v14_clean_text(en)
    ru_s = _v14_clean_text(ru)
    # Prefer human override over English fallback and over Wikidata RU labels that are actually English.
    if en_s in PAINTING_RU_OVERRIDES:
        return PAINTING_RU_OVERRIDES[en_s]
    if _good_label(ru_s) and ru_s.casefold() != en_s.casefold():
        return ru_s
    if field in {"collection_country", "creator_country", "creator_birth_country"} and en_s in PAINTING_RU_OVERRIDES:
        return PAINTING_RU_OVERRIDES[en_s]
    return ru_s if _good_label(ru_s) else en_s


def _v14_year_window(y: Any) -> Optional[Tuple[int, int]]:
    yi = _safe_int(y)
    if yi is None:
        return None
    for a, b in PAINTING_DATE_WINDOWS:
        if a <= yi <= b:
            return (a, b)
    return None


def _v14_refresh_seed_year_windows() -> None:
    global paintings_seed_df
    if paintings_seed_df is None or len(paintings_seed_df) == 0 or "year" not in paintings_seed_df.columns:
        return
    df = paintings_seed_df.copy()
    starts, ends, keys = [], [], []
    for y in df["year"].tolist():
        win = _v14_year_window(y)
        if win:
            a, b = win
            starts.append(a); ends.append(b); keys.append(f"{a}_{b}")
        else:
            starts.append(None); ends.append(None); keys.append(None)
    df["year_bucket"] = keys
    df["year_from"] = starts
    df["year_to"] = ends
    df["year_ru"] = [f"{a}–{b}" if a is not None else None for a, b in zip(starts, ends)]
    df["year_en"] = [f"{a}-{b}" if a is not None else None for a, b in zip(starts, ends)]
    for c in list(df.columns):
        if c.endswith("_en") or c.endswith("_ru") or c in {"item_en", "item_ru"}:
            df[c] = df[c].map(_v14_clean_text)
    paintings_seed_df = df


def _v14_date_key_from_constraints(c: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    y1 = _safe_int((c or {}).get("date_from")); y2 = _safe_int((c or {}).get("date_to"))
    if y1 is None or y2 is None:
        return None
    return (int(y1), int(y2))


def _v14_date_key_from_spec(spec: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    return _v14_date_key_from_constraints(spec.get("constraints") or {})


def _v14_date_key_from_record(rec: Dict[str, Any]) -> Optional[Tuple[int, int]]:
    return _v14_date_key_from_constraints(rec.get("constraints") or {})


def _v14_span(dkey: Optional[Tuple[int, int]]) -> int:
    return 0 if not dkey else int(dkey[1]) - int(dkey[0])


def _v14_group_candidates(df, fields: Sequence[str], min_n: int, max_n: Optional[int], max_rows: int = 3000):
    """Like _painting_group_candidates, but uses RU overrides and does not drop useful EN-labeled rows."""
    if df is None or len(df) == 0:
        return pd.DataFrame()
    sub = df.copy()
    group_cols = []
    agg = {"n": ("item_qid", "nunique")}
    for field in fields:
        if field == "year_bucket":
            sub = sub[sub["year_bucket"].notna()]
            group_cols.append("year_bucket")
            for c in ["year_from", "year_to", "year_ru", "year_en"]:
                agg[c] = (c, "first")
        else:
            qcol, encol, rucol = f"{field}_qid", f"{field}_en", f"{field}_ru"
            if qcol not in sub.columns or encol not in sub.columns:
                return pd.DataFrame()
            if rucol not in sub.columns:
                sub[rucol] = None
            sub = sub[sub[qcol].apply(_good_qid) & sub[encol].apply(_good_label)]
            if len(sub) == 0:
                return pd.DataFrame()
            sub[rucol] = [_v14_ru_label(field, en, ru) for en, ru in zip(sub[encol].tolist(), sub[rucol].tolist())]
            sub = sub[sub[rucol].apply(_good_label)]
            group_cols.append(qcol)
            agg[encol] = (encol, "first")
            agg[rucol] = (rucol, "first")
    if len(sub) == 0:
        return pd.DataFrame()
    grp = sub.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    grp = grp[grp["n"] >= int(min_n)]
    if max_n is not None:
        grp = grp[grp["n"] <= int(max_n)]
    return grp.sort_values(["n"], ascending=[True]).head(int(max_rows)).reset_index(drop=True)


# Expanded direct registry. L4/L5 include many non-seed paths and fallback-hard variants.
PAINTING_DIRECT_TEMPLATES = [
    {"level": "L1", "template_id": "paintings_l1_by_collection", "family": "single_collection", "fields": ["collection"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_creator", "family": "single_creator", "fields": ["creator"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_material", "family": "single_material", "fields": ["material"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_movement", "family": "single_movement", "fields": ["movement"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_genre", "family": "single_genre", "fields": ["genre"], "min_n": 5, "max_n": 240},
    {"level": "L1", "template_id": "paintings_l1_by_depicts", "family": "single_depicts", "fields": ["depicts"], "min_n": 5, "max_n": 240},
    {"level": "L2", "template_id": "paintings_l2_material_collection_country", "family": "material_collection_country", "fields": ["material", "collection_country"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_genre_collection_country", "family": "genre_collection_country", "fields": ["genre", "collection_country"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_movement_collection_country", "family": "movement_collection_country", "fields": ["movement", "collection_country"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_depicts_collection_country", "family": "depicts_collection_country", "fields": ["depicts", "collection_country"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_creator_citizenship_collection_country", "family": "creator_country_collection_country", "fields": ["creator_country", "collection_country"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_collection_material", "family": "collection_material", "fields": ["collection", "material"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_collection_genre", "family": "collection_genre", "fields": ["collection", "genre"], "min_n": 5, "max_n": 175},
    {"level": "L2", "template_id": "paintings_l2_collection_city_material", "family": "collection_city_material", "fields": ["collection_city", "material"], "min_n": 5, "max_n": 175},
    # L3
    {"level": "L3", "template_id": "paintings_l3_material_collection_country_period", "family": "material_collection_country_period", "fields": ["material", "collection_country", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_genre_collection_country_period", "family": "genre_collection_country_period", "fields": ["genre", "collection_country", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_movement_collection_country_period", "family": "movement_collection_country_period", "fields": ["movement", "collection_country", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_depicts_collection_country_period", "family": "depicts_collection_country_period", "fields": ["depicts", "collection_country", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_material_creator_citizenship_period", "family": "material_creator_country_period", "fields": ["material", "creator_country", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_genre_material_collection_country", "family": "genre_material_collection_country", "fields": ["genre", "material", "collection_country"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_collection_material_period", "family": "collection_material_period", "fields": ["collection", "material", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_collection_genre_period", "family": "collection_genre_period", "fields": ["collection", "genre", "year_bucket"], "min_n": 4, "max_n": 125},
    {"level": "L3", "template_id": "paintings_l3_collection_city_material_period", "family": "collection_city_material_period", "fields": ["collection_city", "material", "year_bucket"], "min_n": 4, "max_n": 125},
    # L4
    {"level": "L4", "template_id": "paintings_l4_material_creator_citizenship_collection_country_period", "family": "material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_material_creator_birth_country_collection_country_period", "family": "material_creator_birth_country_collection_country_period", "fields": ["material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_genre_creator_citizenship_collection_country_period", "family": "genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_genre_material_collection_country_period", "family": "genre_material_collection_country_period", "fields": ["genre", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_movement_material_collection_country_period", "family": "movement_material_collection_country_period", "fields": ["movement", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_depicts_material_collection_country_period", "family": "depicts_material_collection_country_period", "fields": ["depicts", "material", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_collection_material_creator_citizenship_period", "family": "collection_material_creator_country_period", "fields": ["collection", "material", "creator_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_collection_genre_material_period", "family": "collection_genre_material_period", "fields": ["collection", "genre", "material", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_collection_city_material_creator_citizenship_period", "family": "collection_city_material_creator_country_period", "fields": ["collection_city", "material", "creator_country", "year_bucket"], "min_n": 3, "max_n": 100},
    {"level": "L4", "template_id": "paintings_l4_depicts_creator_citizenship_collection_country_period", "family": "depicts_creator_country_collection_country_period", "fields": ["depicts", "creator_country", "collection_country", "year_bucket"], "min_n": 3, "max_n": 100},
    # L5, true 5-constraint + hard fallback variants
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_citizenship_collection_country_period", "family": "genre_material_creator_country_collection_country_period", "fields": ["genre", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_genre_material_creator_birth_country_collection_country_period", "family": "genre_material_creator_birth_country_collection_country_period", "fields": ["genre", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_citizenship_collection_country_period", "family": "movement_material_creator_country_collection_country_period", "fields": ["movement", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_movement_material_creator_birth_country_collection_country_period", "family": "movement_material_creator_birth_country_collection_country_period", "fields": ["movement", "material", "creator_birth_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_depicts_genre_material_collection_country_period", "family": "depicts_genre_material_collection_country_period", "fields": ["depicts", "genre", "material", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_depicts_material_creator_citizenship_collection_country_period", "family": "depicts_material_creator_country_collection_country_period", "fields": ["depicts", "material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_depicts_genre_creator_citizenship_collection_country_period", "family": "depicts_genre_creator_country_collection_country_period", "fields": ["depicts", "genre", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_movement_genre_material_collection_country_period", "family": "movement_genre_material_collection_country_period", "fields": ["movement", "genre", "material", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_collection_genre_material_creator_citizenship_period", "family": "collection_genre_material_creator_country_period", "fields": ["collection", "genre", "material", "creator_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_collection_depicts_material_creator_citizenship_period", "family": "collection_depicts_material_creator_country_period", "fields": ["collection", "depicts", "material", "creator_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_collection_city_genre_material_period", "family": "collection_city_genre_material_period", "fields": ["collection_city", "genre", "material", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_hard_material_creator_citizenship_collection_country_period", "family": "hard_material_creator_country_collection_country_period", "fields": ["material", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_hard_genre_creator_citizenship_collection_country_period", "family": "hard_genre_creator_country_collection_country_period", "fields": ["genre", "creator_country", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
    {"level": "L5", "template_id": "paintings_l5_hard_depicts_material_collection_country_period", "family": "hard_depicts_material_collection_country_period", "fields": ["depicts", "material", "collection_country", "year_bucket"], "min_n": 2, "max_n": 90},
]


def _make_painting_direct_spec(tpl: Dict[str, Any], row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    level = tpl["level"]
    k = PAINTINGS_REQUESTED_COUNT[level]
    where_lines, fragments_ru, fragments_en = [], [], []
    constraints = {"answer_type": "painting"}
    constraint_qids = {"answer_type": Q_PAINTING}
    for field in tpl["fields"]:
        if field == "year_bucket":
            y1, y2 = _safe_int(row.get("year_from")), _safe_int(row.get("year_to"))
            if y1 is None or y2 is None:
                return None
            if (int(y1), int(y2)) in PAINTINGS_FORBIDDEN_DATE_RANGES or (int(y2) - int(y1) > 80 and level in {"L3", "L4", "L5"}):
                return None
            where_lines.extend(_painting_date_filter(int(y1), int(y2)))
            constraints["date_from"], constraints["date_to"] = int(y1), int(y2)
            fragments_ru.append(f"созданных в период {int(y1)}–{int(y2)} годов")
            fragments_en.append(f"created between {int(y1)} and {int(y2)}")
            continue
        qid, en, ru0 = row.get(f"{field}_qid"), row.get(f"{field}_en"), row.get(f"{field}_ru")
        if not (_good_qid(qid) and _good_label(en)):
            return None
        ru = _v14_ru_label(field, en, ru0)
        if not _good_label(ru):
            return None
        where_lines.extend(PAINTING_FIELD_BUILDERS[field](qid))
        cname = PAINTING_FIELD_CONSTRAINT_NAMES[field]
        constraints[cname] = _v14_clean_text(en)
        constraint_qids[cname] = qid
        fr_ru, fr_en = _painting_fragment(field, str(en), str(ru))
        fragments_ru.append(fr_ru); fragments_en.append(fr_en)
    query_ru, query_en = _painting_query_text(k, fragments_ru, fragments_en)
    return {
        "complexity": level,
        "requested_count": k,
        "template_id": tpl["template_id"],
        "template_family": tpl["family"],
        "query_text_ru": _v14_clean_text(query_ru),
        "query_text_en": _v14_clean_text(query_en),
        "constraints": constraints,
        "constraint_qids": constraint_qids,
        "where_lines": _dedupe_lines(where_lines),
        "local_n": int(row.get("n", 0) or 0),
    }


def _v14_template_priority(template_id: str) -> int:
    t = str(template_id)
    if "l1_by_collection" in t or "l1_by_creator" in t:
        return 0
    if "collection_city" in t or "collection_" in t:
        return 1
    if "depicts" in t:
        return 2
    if "movement" in t:
        return 3
    if "material" in t:
        return 4
    if "genre" in t:
        return 5
    return 6


def _v14_cluster_signature(c: Dict[str, Any]) -> Optional[str]:
    country = _v14_norm(c.get("collection_country"))
    material = _v14_norm(c.get("material"))
    genre = _v14_norm(c.get("genre"))
    movement = _v14_norm(c.get("art_movement"))
    dkey = _v14_date_key_from_constraints(c)
    if country in {"italy", "vatican city"} and material in {"fresco", "tempera", "plaster"} and dkey and 1200 <= dkey[0] <= 1550:
        if genre == "religious art" or movement in {"early renaissance", "high renaissance", "italian renaissance", "renaissance"}:
            return "italy_vatican_religious_fresco_renaissance"
    return None


def _v14_bad_cluster_penalty(spec_or_rec: Dict[str, Any]) -> int:
    c = spec_or_rec.get("constraints") or {}
    dkey = _v14_date_key_from_constraints(c)
    country, material, genre, movement = _v14_norm(c.get("collection_country")), _v14_norm(c.get("material")), _v14_norm(c.get("genre")), _v14_norm(c.get("art_movement"))
    penalty = 0
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        penalty += 9999
    if country in {"italy", "vatican city"}:
        penalty += 60
    if country in {"italy", "vatican city"} and material in {"fresco", "tempera", "plaster"} and dkey and 1200 <= dkey[0] <= 1550:
        penalty += 220
    if country in {"italy", "vatican city"} and genre == "religious art" and dkey and 1200 <= dkey[0] <= 1550:
        penalty += 220
    if country in {"italy", "vatican city"} and movement in {"early renaissance", "high renaissance", "italian renaissance", "renaissance"}:
        penalty += 160
    # Strongly prefer later periods because prior runs had almost no 1600+ coverage.
    if dkey:
        if dkey[0] >= 1600:
            penalty -= 120
        if dkey[0] >= 1800:
            penalty -= 160
        if dkey[0] >= 1900:
            penalty -= 190
    return penalty


def _v14_spec_sort_key(spec: Dict[str, Any]) -> Tuple[int, int, int, int, str]:
    level = str(spec.get("complexity"))
    local_n = _safe_int(spec.get("local_n"), 10**9) or 10**9
    ideal = {"L1": 45, "L2": 34, "L3": 24, "L4": 18, "L5": 12}.get(level, 24)
    return (_v14_bad_cluster_penalty(spec), _v14_template_priority(spec.get("template_id", "")), abs(local_n - ideal), local_n, str(spec.get("template_id")))


def _v14_interleave_specs(specs: List[Dict[str, Any]], rng: random.Random) -> List[Dict[str, Any]]:
    buckets = defaultdict(list)
    for s in specs:
        c = s.get("constraints") or {}
        key = (_v14_date_key_from_spec(s), _v14_norm(c.get("collection_country")), _v14_norm(c.get("material")), _v14_norm(c.get("genre")), str(s.get("template_id")))
        buckets[key].append(s)
    for k in buckets:
        buckets[k].sort(key=_v14_spec_sort_key)
    keys = sorted(buckets.keys(), key=lambda k: (0 if k[0] and k[0][0] >= 1600 else 1, str(k[0]), k[1], k[2], k[3], k[4]))
    out = []
    while True:
        moved = False
        for k in keys:
            if buckets[k]:
                out.append(buckets[k].pop(0)); moved = True
        if not moved:
            break
    return out


def build_painting_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    _v14_refresh_seed_year_windows()
    specs: List[Dict[str, Any]] = []
    for tpl in PAINTING_DIRECT_TEMPLATES:
        level = tpl["level"]
        k = PAINTINGS_REQUESTED_COUNT[level]
        max_n = int(tpl.get("max_n", PAINTINGS_MAX_GOLD_BY_LEVEL[level]))
        # For hard levels, the seed pool is incomplete; keep more local groups and let WDQS validate.
        local_max = int(max(max_n * (3.5 if level in {"L4", "L5"} else 1.8), max_n + 40))
        min_n = int(tpl.get("min_n", k))
        grp = _v14_group_candidates(
            paintings_seed_df,
            tpl["fields"],
            min_n=max(2, min_n),
            max_n=local_max,
            max_rows=16000 if level in {"L4", "L5"} else 7000,
        )
        if len(grp) == 0:
            continue
        grp = grp.sort_values("n", ascending=True).head(12000 if level in {"L4", "L5"} else 5000)
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_painting_direct_spec(tpl, row)
            if not spec:
                continue
            if any(part in str(spec.get("template_id", "")) for part in PAINTINGS_DROP_TEMPLATE_ID_SUBSTRINGS):
                continue
            dkey = _v14_date_key_from_spec(spec)
            if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
                continue
            if level in {"L3", "L4", "L5"} and dkey and _v14_span(dkey) > 80:
                continue
            local_n = _safe_int(spec.get("local_n"), 0) or 0
            if level in {"L1", "L2", "L3"} and local_n < k + PAINTINGS_MIN_GOLD_HEADROOM:
                continue
            # For L4/L5 allow local_n < final minimum because WDQS may return more than the seed pool.
            specs.append(spec)
    return specs


def build_painting_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    print("Building paintings candidate queue (v14 balanced from-scratch quality)...")
    all_specs = build_painting_direct_candidates(rng)
    print(f"  direct candidate specs: {len(all_specs)}")
    print("  bridge candidate specs: 0 (disabled permanently: seed variants caused duplicate tasks)")
    by_level = {level: [] for level in PAINTINGS_TARGET_PER_LEVEL}
    seen = set()
    for spec in all_specs:
        key = _spec_global_semantic_key(spec)
        if key in seen:
            continue
        seen.add(key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        by_level[level] = _v14_interleave_specs(by_level[level], rng)
        preview = [(s.get("template_id"), s.get("local_n"), _v14_date_key_from_spec(s), (s.get("constraints") or {}).get("collection_country"), (s.get("constraints") or {}).get("material")) for s in by_level[level][:12]]
        print(f"  {level} top candidates:", preview)
    print("  queue counts:", {lvl: len(v) for lvl, v in by_level.items()})
    return by_level


def _v14_gold_sig(rec: Dict[str, Any]) -> Tuple[str, ...]:
    return tuple(sorted(str(q) for q in rec.get("gold_answer_qids") or [] if q))


def _v14_gold_near_duplicate(rec: Dict[str, Any], old_sigs: List[Tuple[str, ...]], phase_cfg: Dict[str, Any]) -> Tuple[bool, str]:
    sig = _v14_gold_sig(rec)
    if not sig:
        return False, ""
    ss = set(sig)
    for old in old_sigs:
        if sig == old:
            return True, "duplicate_gold_set"
        so = set(old)
        if not so:
            continue
        inter = len(ss & so); union = len(ss | so)
        j = inter / union if union else 0
        containment = inter / min(len(ss), len(so)) if min(len(ss), len(so)) else 0
        if j >= float(phase_cfg["j"]):
            return True, f"near_duplicate_gold_jaccard_{j:.2f}"
        if min(len(ss), len(so)) >= 6 and containment >= float(phase_cfg["contain"]):
            return True, f"near_duplicate_gold_containment_{containment:.2f}"
    return False, ""


def _v14_combo_keys(c: Dict[str, Any]) -> List[Tuple[str, ...]]:
    d = _v14_date_key_from_constraints(c)
    date_s = f"{d[0]}_{d[1]}" if d else "no_date"
    keys = []
    combos = [
        ("collection_country", "material", "date"),
        ("collection_country", "genre", "date"),
        ("collection_country", "art_movement", "date"),
        ("creator_citizenship", "collection_country", "date"),
        ("creator_birth_country", "collection_country", "date"),
        ("depicts", "collection_country", "date"),
        ("material", "creator_citizenship", "collection_country", "date"),
        ("genre", "material", "collection_country", "date"),
    ]
    for fields in combos:
        vals, ok = [], True
        for f in fields:
            if f == "date":
                vals.append(date_s)
            else:
                v = c.get(f)
                if not v:
                    ok = False; break
                vals.append(_v14_norm(v))
        if ok:
            keys.append(tuple(["combo"] + list(fields) + vals))
    return keys


def _v14_value_keys(c: Dict[str, Any], level: str) -> List[Tuple[str, ...]]:
    keys = []
    for fld, name in [("collection_country", "country"), ("material", "material"), ("genre", "genre")]:
        v = c.get(fld)
        if v:
            nv = _v14_norm(v)
            keys.append((name, nv))
            keys.append((name, level, nv))
    cluster = _v14_cluster_signature(c)
    if cluster:
        keys.append(("cluster", cluster)); keys.append(("cluster", level, cluster))
    return keys


def _v14_value_cap_for_key(key: Tuple[str, ...], phase_cfg: Dict[str, Any]) -> Optional[int]:
    relax = int(phase_cfg.get("value_relax", 0))
    if key[0] == "country" and len(key) == 2:
        base = V14_COUNTRY_TOTAL_CAP.get(key[1])
        return None if base is None else base + relax * 4
    if key[0] == "country" and len(key) == 3:
        base = V14_COUNTRY_LEVEL_CAP.get(key[1], {}).get(key[2])
        return None if base is None else base + relax * 2
    if key[0] == "material" and len(key) == 2:
        base = V14_MATERIAL_TOTAL_CAP.get(key[1])
        return None if base is None else base + relax * 3
    if key[0] == "material" and len(key) == 3:
        base = V14_MATERIAL_LEVEL_CAP.get(key[1], {}).get(key[2])
        return None if base is None else base + relax * 2
    if key[0] == "genre" and len(key) == 2:
        base = V14_GENRE_TOTAL_CAP.get(key[1])
        return None if base is None else base + relax * 3
    if key[0] == "genre" and len(key) == 3:
        base = V14_GENRE_LEVEL_CAP.get(key[1], {}).get(key[2])
        return None if base is None else base + relax * 2
    if key[0] == "cluster" and len(key) == 2:
        return V14_CLUSTER_TOTAL_CAP + relax * 2
    if key[0] == "cluster" and len(key) == 3:
        return V14_CLUSTER_LEVEL_CAP.get(key[1], 9999) + relax
    return None


def _v14_caps_pass(level: str, c: Dict[str, Any], date_counts: Counter, level_date_counts: Counter, combo_counts: Counter, value_counts: Counter, phase_cfg: Dict[str, Any]) -> Tuple[bool, str]:
    dkey = _v14_date_key_from_constraints(c)
    if dkey in PAINTINGS_FORBIDDEN_DATE_RANGES:
        return False, "legacy_broad_date_disabled"
    if dkey is not None:
        if level in {"L3", "L4", "L5"} and _v14_span(dkey) > 80:
            return False, "broad_hard_date_disabled"
        if date_counts[dkey] >= int(phase_cfg["date_global"]):
            return False, "date_range_cap"
        if level in {"L3", "L4", "L5"} and level_date_counts[(level, dkey)] >= int(phase_cfg["date_level"]):
            return False, "level_date_range_cap"
    for ck in _v14_combo_keys(c):
        if combo_counts[(level, ck)] >= int(phase_cfg["combo"]):
            return False, "combo_cap"
    for vk in _v14_value_keys(c, level):
        cap = _v14_value_cap_for_key(vk, phase_cfg)
        if cap is not None and value_counts[vk] >= cap:
            return False, f"value_cap_{'_'.join(vk)}"
    return True, ""


def _v14_spec_prefilter(spec: Dict[str, Any], counts: Counter, date_counts: Counter, level_date_counts: Counter, combo_counts: Counter, value_counts: Counter, seen_semantic: set, phase_cfg: Dict[str, Any]) -> Tuple[bool, str]:
    level = str(spec.get("complexity") or "")
    if counts[level] >= int(PAINTINGS_TARGET_PER_LEVEL[level]):
        return False, "level_full"
    if _spec_global_semantic_key(spec) in seen_semantic:
        return False, "duplicate_spec_semantic"
    return _v14_caps_pass(level, spec.get("constraints") or {}, date_counts, level_date_counts, combo_counts, value_counts, phase_cfg)


def _v14_record_passes(rec: Dict[str, Any], counts: Counter, date_counts: Counter, level_date_counts: Counter, combo_counts: Counter, value_counts: Counter, seen_record_keys: set, seen_semantic: set, seen_gold_sigs: List[Tuple[str, ...]], phase_cfg: Dict[str, Any]) -> Tuple[bool, str]:
    level = str(rec.get("complexity") or "")
    if counts[level] >= int(PAINTINGS_TARGET_PER_LEVEL[level]):
        return False, "level_full"
    if not _schema_is_exact(rec):
        return False, "schema_mismatch"
    if any(part in str(rec.get("template_id", "")) for part in PAINTINGS_DROP_TEMPLATE_ID_SUBSTRINGS):
        return False, "seed_template_disabled"
    g = len(rec.get("gold_answer_qids") or [])
    req = int(rec.get("requested_count") or PAINTINGS_REQUESTED_COUNT.get(level, 3))
    if g < req + PAINTINGS_MIN_GOLD_HEADROOM:
        return False, "too_few_gold"
    if g > PAINTINGS_MAX_GOLD_BY_LEVEL.get(level, 9999):
        return False, "too_many_gold"
    if _record_key(rec) in seen_record_keys:
        return False, "duplicate_record_key"
    if _record_global_semantic_key(rec) in seen_semantic:
        return False, "duplicate_resolved_constraints"
    near, why = _v14_gold_near_duplicate(rec, seen_gold_sigs, phase_cfg)
    if near:
        return False, why
    ok, why = _v14_caps_pass(level, rec.get("constraints") or {}, date_counts, level_date_counts, combo_counts, value_counts, phase_cfg)
    if not ok:
        return False, why
    return True, ""


def _v14_update_state(rec: Dict[str, Any], counts: Counter, date_counts: Counter, level_date_counts: Counter, combo_counts: Counter, value_counts: Counter, seen_keys: set, seen_semantic: set, seen_gold_sigs: List[Tuple[str, ...]]) -> None:
    level = str(rec.get("complexity"))
    counts[level] += 1
    c = rec.get("constraints") or {}
    dkey = _v14_date_key_from_record(rec)
    if dkey:
        date_counts[dkey] += 1
        level_date_counts[(level, dkey)] += 1
    for ck in _v14_combo_keys(c):
        combo_counts[(level, ck)] += 1
    for vk in _v14_value_keys(c, level):
        value_counts[vk] += 1
    seen_keys.add(_record_key(rec))
    seen_semantic.add(_record_global_semantic_key(rec))
    seen_gold_sigs.append(_v14_gold_sig(rec))


def generate_paintings_dataset(
    output_path: Path = PAINTINGS_OUTPUT_PATH,
    target_per_level: Dict[str, int] = PAINTINGS_TARGET_PER_LEVEL,
    seed: int = 20260528,
    require_complete_gold: bool = True,
    reset_output: bool = True,
) -> List[Dict[str, Any]]:
    """Generate the final paintings JSONL from scratch with balanced quality constraints."""
    if output_path.exists():
        _archive_existing_jsonl(output_path, suffix="pre_v14_from_scratch")
    alt = output_path.with_name("paintings1.jsonl")
    if alt.exists():
        _archive_existing_jsonl(alt, suffix="pre_v14_from_scratch")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

    _v14_refresh_seed_year_windows()
    queue = build_painting_candidate_queue(seed=seed)

    counts, date_counts, level_date_counts = Counter(), Counter(), Counter()
    combo_counts, value_counts = Counter(), Counter()
    seen_record_keys, seen_semantic = set(), set()
    seen_gold_sigs: List[Tuple[str, ...]] = []
    next_idx = defaultdict(lambda: 1)
    audit_skips = []
    reject_reasons_by_level = defaultdict(Counter)
    dead_specs = set()

    total_target = sum(int(v) for v in target_per_level.values())
    pbar = tqdm(total=total_target, initial=0, desc="paintings total")
    fill_order = ["L1", "L2", "L3", "L4", "L5"]

    for phase_i, phase_cfg in enumerate(V14_PHASES):
        phase_gain = 0
        print(f"\n=== Paintings v14 phase {phase_i}: {phase_cfg['name']} ===")
        for level in fill_order:
            target = int(target_per_level[level])
            if counts[level] >= target:
                continue
            attempts = 0
            accepted_before = counts[level]
            specs = list(queue.get(level, []))
            for spec in specs:
                if counts[level] >= target:
                    break
                attempts += 1
                sdead = _spec_global_semantic_key(spec)
                if sdead in dead_specs:
                    continue
                ok, reason = _v14_spec_prefilter(spec, counts, date_counts, level_date_counts, combo_counts, value_counts, seen_semantic, phase_cfg)
                if not ok:
                    reject_reasons_by_level[level][reason] += 1
                    continue
                try:
                    ex = _finalize_painting_spec(spec, next_idx[level], require_complete=require_complete_gold)
                except Exception as e:
                    reject_reasons_by_level[level]["exception"] += 1
                    audit_skips.append({"level": level, "phase": phase_i, "template_id": spec.get("template_id"), "reason": "exception", "error": str(e)[:500], "constraints": spec.get("constraints")})
                    if attempts % 25 == 0:
                        pbar.set_postfix_str(f"{level} phase={phase_i} attempts={attempts} accepted={counts[level]}/{target} last=exception")
                    continue
                if ex is None:
                    reason = str(spec.get("_last_reject_reason") or "gold_rejected_or_incomplete")
                    rhead = reason.split()[0].split(":")[0]
                    reject_reasons_by_level[level][rhead] += 1
                    if rhead in {"too_few_gold", "too_many_gold", "truncated_or_over_cap", "gold_rejected_or_incomplete"}:
                        dead_specs.add(sdead)
                    audit_skips.append({"level": level, "phase": phase_i, "template_id": spec.get("template_id"), "reason": reason, "local_n": spec.get("local_n"), "date_range": list(_v14_date_key_from_spec(spec)) if _v14_date_key_from_spec(spec) else None, "constraints": spec.get("constraints")})
                    if attempts % 25 == 0:
                        pbar.set_postfix_str(f"{level} phase={phase_i} attempts={attempts} accepted={counts[level]}/{target} reasons={dict(reject_reasons_by_level[level].most_common(2))}")
                    continue
                rec = _normalize_record_text(_example_to_record(ex))
                rec["query_text_ru"] = _v14_clean_text(rec.get("query_text_ru"))
                rec["query_text_en"] = _v14_clean_text(rec.get("query_text_en"))
                ok, reason = _v14_record_passes(rec, counts, date_counts, level_date_counts, combo_counts, value_counts, seen_record_keys, seen_semantic, seen_gold_sigs, phase_cfg)
                if not ok:
                    reject_reasons_by_level[level][reason] += 1
                    audit_skips.append({"level": level, "phase": phase_i, "template_id": spec.get("template_id"), "reason": reason, "gold": len(ex.gold_answer_qids), "date_range": list(_v14_date_key_from_record(rec)) if _v14_date_key_from_record(rec) else None, "constraints": rec.get("constraints")})
                    continue
                rec["id"] = f"paintings_{level.lower()}_{next_idx[level]:04d}"
                _append_jsonl(output_path, rec)
                _v14_update_state(rec, counts, date_counts, level_date_counts, combo_counts, value_counts, seen_record_keys, seen_semantic, seen_gold_sigs)
                next_idx[level] += 1
                phase_gain += 1
                pbar.update(1)
                pbar.set_postfix_str(f"{level} phase={phase_i} accepted={counts[level]}/{target} gold={len(ex.gold_answer_qids)} date={_v14_date_key_from_record(rec)}")
            if counts[level] < target:
                gained = counts[level] - accepted_before
                print(f"[WARN] phase {phase_i} paintings {level}: accepted {counts[level]}/{target}; gained {gained}; top reject reasons: {dict(reject_reasons_by_level[level].most_common(12))}")
        if all(counts.get(level, 0) >= int(target_per_level[level]) for level in target_per_level):
            break
        if phase_gain == 0:
            print(f"[WARN] phase {phase_i} made no progress; moving to next relaxation phase.")
    pbar.close()

    final_rows = _read_jsonl(output_path)
    final_counts = Counter(r.get("complexity") for r in final_rows if r.get("domain") == PAINTINGS_DOMAIN)
    final_date_counts = Counter(_v14_date_key_from_record(r) for r in final_rows if _v14_date_key_from_record(r) is not None)
    format_problems = validate_jsonl_exact_format(output_path)
    if len(format_problems) > 0:
        raise AssertionError(f"Generated JSONL format mismatch: {format_problems.head(10).to_dict('records')}")

    audit = {
        "domain": PAINTINGS_DOMAIN,
        "output_path": str(output_path),
        "target_per_level": dict(target_per_level),
        "counts_by_complexity": dict(final_counts),
        "date_counts": {f"{k[0]}_{k[1]}": v for k, v in sorted(final_date_counts.items())},
        "value_counts": {"|".join(map(str, k)): v for k, v in value_counts.most_common(200)},
        "seed_pool": paintings_seed_audit(),
        "candidate_counts": {lvl: len(v) for lvl, v in queue.items()},
        "reject_reasons_by_level": {lvl: dict(cnt.most_common(40)) for lvl, cnt in reject_reasons_by_level.items()},
        "skipped_count": len(audit_skips),
        "skipped_preview": audit_skips[-2000:],
        "updated_at": _now_iso(),
        "format_fields": BENCHMARK_FIELD_ORDER,
        "quality_patch": PAINTINGS_FINAL_PATCH,
        "date_windows": PAINTING_DATE_WINDOWS,
        "from_scratch": True,
        "fill_order": fill_order,
        "final_target_met": {lvl: final_counts.get(lvl, 0) >= int(target_per_level[lvl]) for lvl in target_per_level},
        "notes": "v14 uses soft-relaxed diversity caps but keeps exact schema, complete gold sentinel and anti-duplicate gold-set checks in every phase.",
    }
    _write_json(PAINTINGS_AUDIT_PATH, audit)
    print(f"saved incrementally: {output_path}")
    print(f"audit: {PAINTINGS_AUDIT_PATH}")
    print("Final counts:", dict(final_counts))
    print("Final date counts:", final_date_counts.most_common(30))
    if any(final_counts.get(lvl, 0) < int(target_per_level[lvl]) for lvl in target_per_level):
        print("[WARN] Final target not fully met:", {lvl: (final_counts.get(lvl, 0), int(target_per_level[lvl])) for lvl in target_per_level})
    else:
        print("✅ Final paintings v14 target met from scratch.")
    return final_rows

print("✅ Applied paintings v14 FROM-SCRATCH balanced quality patch: exact target 15/22/25/25/25, L1→L5 order, stronger date/country/material/genre diversity, expanded L4/L5 candidates")


✅ Applied paintings v14 FROM-SCRATCH balanced quality patch: exact target 15/22/25/25/25, L1→L5 order, stronger date/country/material/genre diversity, expanded L4/L5 candidates


In [12]:
# ============================================================
# Generate paintings JSONL from scratch (v14 balanced final quality target)
# ============================================================
# This intentionally ignores old paintings.jsonl/paintings1.jsonl and archives any
# existing output before creating a fresh final file.
RUN_PAINTINGS_GENERATION = True
RESET_PAINTINGS_OUTPUT = True
if RUN_PAINTINGS_GENERATION:
    paintings_records = generate_paintings_dataset(reset_output=RESET_PAINTINGS_OUTPUT)
else:
    print("Generation skipped. Use smoke_test_paintings() or generate_paintings_dataset() manually.")


Building paintings candidate queue (v14 balanced from-scratch quality)...
  direct candidate specs: 1159
  bridge candidate specs: 0 (disabled permanently: seed variants caused duplicate tasks)
  L1 top candidates: [('paintings_l1_by_collection', 47, None, None, None), ('paintings_l1_by_creator', 31, None, None, None), ('paintings_l1_by_depicts', 34, None, None, None), ('paintings_l1_by_movement', 31, None, None, None), ('paintings_l1_by_genre', 8, None, None, None), ('paintings_l1_by_genre', 22, None, None, None), ('paintings_l1_by_genre', 19, None, None, None), ('paintings_l1_by_genre', 18, None, None, None), ('paintings_l1_by_genre', 23, None, None, None), ('paintings_l1_by_genre', 19, None, None, None), ('paintings_l1_by_genre', 16, None, None, None), ('paintings_l1_by_genre', 240, None, None, None)]
  L2 top candidates: [('paintings_l2_collection_genre', 22, None, None, None), ('paintings_l2_collection_genre', 10, None, None, None), ('paintings_l2_collection_genre', 15, None, None

paintings total:   1%|          | 1/112 [00:00<00:00, 260.31it/s, L1 phase=0 accepted=1/15 gold=48 date=None]


=== Paintings v14 phase 0: strict_balanced ===


paintings total:  46%|████▌     | 51/112 [12:26<19:16, 18.96s/it, L3 phase=0 accepted=14/25 gold=73 date=(1450, 1499)]                                                         

[WARN] phase 0 paintings L3: accepted 14/25; gained 14; top reject reasons: {'too_few_gold': 36, 'level_date_range_cap': 27, 'near_duplicate_gold_containment_1.00': 18, 'truncated_or_over_cap': 9, 'near_duplicate_gold_containment_0.92': 2, 'duplicate_gold_set': 1, 'near_duplicate_gold_containment_0.89': 1, 'near_duplicate_gold_containment_0.93': 1, 'near_duplicate_gold_containment_0.95': 1, 'near_duplicate_gold_jaccard_0.98': 1, 'near_duplicate_gold_jaccard_0.74': 1, 'near_duplicate_gold_containment_0.94': 1}


...

[WARN] phase 3 paintings L5: accepted 19/25; gained 3; top reject reasons: {'date_range_cap': 520, 'too_few_gold': 366, 'value_cap_material_fresco': 167, 'near_duplicate_gold_containment_1.00': 87, 'duplicate_spec_semantic': 41, 'duplicate_gold_set': 25, 'value_cap_material_tempera': 25, 'level_date_range_cap': 23, 'combo_cap': 9, 'truncated_or_over_cap': 3, 'near_duplicate_gold_jaccard_0.83': 2, 'near_duplicate_gold_jaccard_0.91': 2}

=== Paintings v14 phase 4: target_guarantee ===
[WARN] phase 4 paintings L3: accepted 20/25; gained 0; top reject reasons: {'near_duplicate_gold_containment_1.00': 115, 'duplicate_spec_semantic': 69, 'too_few_gold': 55, 'level_date_range_cap': 55, 'truncated_or_over_cap': 11, 'duplicate_gold_set': 5, 'near_duplicate_gold_jaccard_0.98': 5, 'near_duplicate_gold_containment_0.92': 4, 'near_duplicate_gold_jaccard_0.88': 3, 'near_duplicate_gold_containment_0.93': 2, 'near_duplicate_gold_containment_0.95': 2, 'near_duplicate_gold_containment_0.94': 2}


paintings total:  87%|████████▋ | 97/112 [40:00<04:06, 16.43s/it, L4 phase=4 accepted=21/25 gold=8 date=(1450, 1499)] 

[WARN] phase 4 paintings L4: accepted 21/25; gained 2; top reject reasons: {'near_duplicate_gold_containment_1.00': 216, 'too_few_gold': 97, 'duplicate_spec_semantic': 61, 'date_range_cap': 60, 'value_cap_material_fresco': 56, 'level_date_range_cap': 41, 'duplicate_gold_set': 22, 'duplicate_record_key': 5, 'near_duplicate_gold_jaccard_0.98': 3, 'near_duplicate_gold_jaccard_0.79': 2, 'exception': 2, 'near_duplicate_gold_containment_0.94': 2}


paintings total:  88%|████████▊ | 99/112 [41:32<05:27, 25.18s/it, L5 phase=4 attempts=450 accepted=21/25 reasons={'date_range_cap': 578, 'too_few_gold': 386}]

[WARN] phase 4 paintings L5: accepted 21/25; gained 2; top reject reasons: {'date_range_cap': 590, 'too_few_gold': 398, 'value_cap_material_fresco': 167, 'near_duplicate_gold_containment_1.00': 129, 'duplicate_spec_semantic': 60, 'duplicate_gold_set': 41, 'value_cap_material_tempera': 25, 'level_date_range_cap': 23, 'combo_cap': 9, 'truncated_or_over_cap': 3, 'near_duplicate_gold_jaccard_0.83': 2, 'near_duplicate_gold_jaccard_0.91': 2}
saved incrementally: out_wikidata_benchmark/domain_outputs/paintings.jsonl
audit: out_wikidata_benchmark/domain_outputs/paintings.audit.json
Final counts: {'L1': 15, 'L2': 22, 'L3': 20, 'L4': 21, 'L5': 21}
Final date counts: [((1450, 1499), 16), ((1300, 1349), 12), ((1400, 1449), 12), ((1850, 1874), 5), ((1600, 1649), 3), ((1500, 1549), 3), ((1875, 1899), 3), ((1250, 1299), 3), ((1750, 1799), 3), ((1550, 1599), 1)]
[WARN] Final target not fully met: {'L1': (15, 15), 'L2': (22, 22), 'L3': (20, 25), 'L4': (21, 25), 'L5': (21, 25)}
